## This file should give an overview over the current dataset and the meta data format
- from the final design file
  - removed variants
- identified problems with the design file and corrected them
- during preparing this notebook I realized that I will create my own region.bed file and variant_region_map.tsv file (change "," to "~" and ">" to "*")
  - second case is only necessary for the group of `GC_Mendelian_variants` which is not included in the region.bed

In [2]:
import ast
from importlib import reload
import numpy as np
import os
import pandas as pd
import re
import yaml


# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf
reload(hf)
# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [3]:
# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class' # SNP
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'

In [3]:
#helpful functions
def set_modified_chromosome(blat_chromosome):
    """i.e. from NC_000001.11 to chr1, ..."""
    if '23' in blat_chromosome:
        return 'chrX'
    if '24' in blat_chromosome:
        return 'chrY'
    chr_number = int(blat_chromosome.split('_')[1].split('.')[0])
    return 'chr%s'%(chr_number)


def set_chr_start_end_strand(row):
    """
    For blat matchable sequences and missing sequences set chr, start, end amd strand
    """
    if row['gene_enhancer_id'] == 'blat_matchable':
        row[col_chr] = row['blat_chr']
        row[col_start] = row['blat_start']
        row[col_end] = row['blat_end']
        row[col_strand] = row['blat_strand']
        return row
    if pd.isna(row['tested_region_chr']):
        # error: unexpected case
        raise ValueError('Unexpected case: tested_region_chr is na')
    row[col_chr] = row['tested_region_chr']
    row[col_start] = row['tested_region_start']
    row[col_end] = row['tested_region_end']
    row[col_strand] = row['tested_region_strand']
    return row


def get_chrom_pos_ref_alt_pattern(header):
    """
    In the 80K MPRA design each variant has a chr-pos-ref-alt pattern in the header
    This function is able to match it and returns the result
    """
    pattern = r'([A-Z]|[0-9]+)-[0-9]+-[A-Z]-[A-Z]'
    matches = re.search(pattern, header)
    if matches:
        return matches.group()
    else: return "NA"


# dict of chr number to refseq chromosome number
chrom_2_refseq = {"1": "NC_000001.11",
    "2": "NC_000002.12",
    "3": "NC_000003.12",
    "4": "NC_000004.12",
    "5": "NC_000005.10",
    "6": "NC_000006.12",
    "7": "NC_000007.14",
    "8": "NC_000008.11",
    "9": "NC_000009.12",
    "10": "NC_000010.11",
    "11": "NC_000011.10",
    "12": "NC_000012.12",
    "13": "NC_000013.11",
    "14": "NC_000014.9",
    "15": "NC_000015.10",
    "16": "NC_000016.10",
    "17": "NC_000017.11",
    "18": "NC_000018.10",
    "19": "NC_000019.10",
    "20": "NC_000020.11",
    "21": "NC_000021.9",
    "22": "NC_000022.11",
    "X": "NC_000023.11",
    "Y": "NC_000024.10"}

# get args with click: input file, seperator, ids of chr, pos, ref, alt in string

def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) - 1 # (input: 1-based => 0-based)
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'

def get_spdi(row, header_col='name'):
    """
    Returns the SPDI identifier for the given variant (tested only)
    Assumption: allele need to be set beforehands
    """
    # TODO: add case for controls
    if not (hf.is_alternative(row[col_allele])):
        row[col_SPDI] = np.nan
        return row
    # identify the variant chrom-pos-ref-alt pattern
    chrom_pos_ref_alt = hf.get_chrom_pos_ref_alt_pattern(row[header_col])
    if chrom_pos_ref_alt == "NA":
        raise ValueError('Variant pattern could not be found')
    # create the SPDI identifier
    row[col_SPDI] = create_speedy_chromosomes(chrom_pos_ref_alt, seperator='-', indices=[0,1,2,3])
    return row



# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

# list columns col_variant_class, col_variant_pos, col_SPDI, col_allele,
list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]


### Dataset renaming: 
##### Rename region.bed.gz
- Rename `region.bed.gz` and `variant_region_map.tsv` like deduplicated
- `region.bed.gz` => `/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/scripts/rename_region_bed_2_design_fa.py` 
  - python /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/scripts/rename_region_bed_2_design_fa.py --input-file /home/kisa/coding/80K_MPRA/design_data/design_info/regions.bed --output-file /home/kisa/coding/80K_MPRA/design_data/design_info/renamed_regions.bed --file-type bed
- nrows: 28390
- nrows (tested): 27556

In [73]:
# python /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/scripts/rename_file_2_design_style.py --input-file /home/kisa/coding/80K_MPRA/design_data/design_info/regions.bed --output-file /home/kisa/coding/80K_MPRA/design_data/design_info/renamed_regions.bed.gz --file-type bed
# python /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/scripts/rename_file_2_design_style.py --input-file /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/unifying_headers_kilian_042024/renamed_regions.bed.gz --output-file /home/kisa/coding/80K_MPRA/design_data/design_info/renamed_regions.bed.gz --file-type bed
# python /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/scripts/rename_file_2_design_style.py --input-file /home/kisa/coding/80K_MPRA/design_data/design_info/regions.bed.gz --output-file /home/kisa/coding/80K_MPRA/design_data/design_info/renamed_regions_v2.bed.gz --file-type bed

In [5]:
# verify renamed_region.bed
region_bed = pd.read_csv('/home/kisa/coding/80K_MPRA/design_data/design_info/renamed_regions.bed.gz', sep='\t', header=None)
# region_bed = pd.read_csv(config['files']['final_design']['region_bed'], sep='\t', header=None)
# region_bed = pd.read_csv(config['files']['final_design']['region_bed_not_renamed'], sep='\t', header=None)
region_bed.columns = ['region_chr', 'region_start', 'region_end', 'region_name', 'region_score', 'region_strand']

metadata_path = config['files']['creating']['metadata_table']
metadata_path = config['files']['creating']['datafreeze_table']
meta_data_file = pd.read_csv(metadata_path, sep="\t")

# Apply the safe_eval function to the specified columns
for col in list_columns:
    meta_data_file[col] = meta_data_file[col].apply(safe_eval)

# only tested sequences
tested_meta_data_file = meta_data_file.loc[meta_data_file[col_name].str.startswith('cardiac_neuro_cava_random')]

# tested_meta_data_file # 73940
tested_meta_data_file['variant_pattern'] = tested_meta_data_file[col_name].apply(get_chrom_pos_ref_alt_pattern)

# remove the alternative sequences
tested_element_meta_data_file = tested_meta_data_file.loc[tested_meta_data_file['variant_pattern'] == 'NA']
# remove the reference sequences
tested_element_meta_data_file = tested_element_meta_data_file.loc[~tested_element_meta_data_file[col_name].str.contains(':REF_')]

# investigate if all element headers can be matched by the region bed (expectation: yes all can be matched)
check_new_region_header = tested_element_meta_data_file.merge(region_bed, left_on=col_name, right_on='region_name', how='left')
check_new_region_header.columns
unmergable_fasta_bed = check_new_region_header.loc[check_new_region_header['region_start'].isna()] # 0
unmergable_fasta_bed.shape[0] # 0 => all could be matched so new header works better

/tmp/ipykernel_258216/290029870.py:8: DtypeWarning: Columns (8,9,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  meta_data_file = pd.read_csv(config['files']['creating']['datafreeze_table'], sep="\t")


0

##### Verify variant region map renaming
- NOTE: Variant ID does not need to be unique: we designed the same variant multiple times
- ref and alt are matchable 
- region matches to the new region.bed file
- nrow: 47044 (variants in the variant map)

In [75]:
# python /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/scripts/rename_file_2_design_style.py --input-file /home/kisa/coding/80K_MPRA/design_data/design_info/variant_region_map.tsv.gz --output-file /home/kisa/coding/80K_MPRA/design_data/design_info/renamed_variant_region_map.tsv.gz --file-type variant_map
# python /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/scripts/rename_file_2_design_style.py --input-file /home/kisa/coding/80K_MPRA/design_data/design_info/variant_region_map.tsv.gz --output-file /home/kisa/coding/80K_MPRA/design_data/design_info/renamed_variant_region_map_unique.tsv.gz --file-type variant_map

In [ ]:
# read new variant region map
# variant_map = pd.read_csv('/home/kisa/coding/80K_MPRA/design_data/design_info/variant_region_map.tsv.gz', sep='\t')
variant_map = pd.read_csv('/home/kisa/coding/80K_MPRA/design_data/design_info/renamed_variant_region_map.tsv.gz', sep='\t', header=None)
variant_map.columns = ['ID', 'Region', 'REF', 'ALT']
print('Variant map shape:', variant_map.shape[0]) # 47043
# tested metadats file:

# show all rows where ID column is not unique
duplicates_in_variant_ID_column = variant_map.loc[variant_map.duplicated(subset='ID', keep=False)]
# write to file and understand the problem:
# duplicates_in_variant_ID_column.to_csv('/home/kisa/coding/80K_MPRA/design_data/design_info/duplicates_in_variant_ID_column.tsv', sep='\t', index=False)

duplicated_variant_map_ID_list = duplicates_in_variant_ID_column['ID'].unique()

def make_variant_map_id_unique(row):
    """
    For all these cases in the variant_map where ID is not unique use the entry in the ALT column instead of the ID column
    """
    if row['ID'] in duplicated_variant_map_ID_list:
        row['ID'] = row['ALT']
    return row
# for all these cases in the variant_map where ID is not unique use the entry in the ALT column instead of the ID column
variant_map = variant_map.apply(make_variant_map_id_unique, axis=1)

# write to file: renamed_variant_region_map_unique.tsv
# variant_map.to_csv('/home/kisa/coding/80K_MPRA/design_data/design_info/renamed_variant_region_map_unique.tsv.gz', sep='\t', index=False, compression='gzip')
variant_map[['ID', 'REF', 'ALT']].to_csv('/home/kisa/coding/80K_MPRA/design_data/design_info/renamed_variant_region_map_unique_mprasnakeflow_input.tsv.gz', sep='\t', index=False, compression='gzip')

# check if all alternative sequences can be matched (variant pattern != "NA")
all_alternative_sequences = tested_meta_data_file.loc[tested_meta_data_file[col_allele] == 'alt']
alternate_sequences_matching = all_alternative_sequences.merge(variant_map, left_on=col_header, right_on='ALT', how='left')
not_matched_alternative_sequences = alternate_sequences_matching.loc[alternate_sequences_matching['REF'].isna()]
print('alt: ', not_matched_alternative_sequences.shape[0])

# Check if all REF can be matched (header contains ":REF_")
all_reference_sequences = tested_meta_data_file.loc[tested_meta_data_file[col_allele] == 'ref']
reference_sequences_matching = all_reference_sequences.merge(variant_map, left_on=col_header, right_on='REF', how='left')
not_matched_reference_sequences = reference_sequences_matching.loc[reference_sequences_matching['ALT'].isna()]
print('ref: ', not_matched_reference_sequences.shape[0])

# Check if all Elements can be matched (header contains ":REF_")
all_element_sequences = tested_meta_data_file.loc[tested_meta_data_file['variant_pattern'] == 'NA']
all_element_sequences = all_element_sequences.loc[~all_element_sequences[col_header].str.contains(':REF_')]
element_sequences_matching = all_element_sequences.merge(region_bed, left_on=col_header, right_on='region_name', how='left')
not_matched_element_sequences = element_sequences_matching.loc[element_sequences_matching['region_start'].isna()]
print('element: ', not_matched_element_sequences.shape[0])

# Check if Region column from the variant_map can be matched with the region.bed
# Regions column only holds the name from the region.bed file

# tested variant map
tested_variant_map = variant_map.loc[variant_map['Region'].str.contains('cardiac_neuro_cava_random')]

variants_regions = tested_variant_map.merge(region_bed, left_on='Region', right_on='region_name', how='left')
matched_variants_regions = variants_regions.loc[~variants_regions['region_name'].isna()]
not_matched_variants_regions = variants_regions.loc[variants_regions['region_name'].isna()]
print('region: ', not_matched_variants_regions.shape[0]) # 0 => no match


Variant map shape: 47044
alt:  0
ref:  0
element:  0
region:  0


##### Verify design file renaming + Collision handling: 


- I found in GC_Selvarajan again alt and ref which need to be changed to match the region bed (see above for the code to remove these)
- ['GC_Selvarajan:REF_rs216222|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Selvarajan:REF_rs754064|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Selvarajan:ALT_rs2297787|STARR-seq-HepG2_fwd_tile1-1_rs2297787']

In [12]:
def rename_selvarajan_names(name, names_of_interest=['GC_Selvarajan:REF_rs216222|STARR-seq-HepG2_fwd_tile1-1',
                                                    'GC_Selvarajan:REF_rs754064|STARR-seq-HepG2_fwd_tile1-1',
                                                    'GC_Selvarajan:ALT_rs2297787|STARR-seq-HepG2_fwd_tile1-1_rs2297787']):
    if name not in names_of_interest:
        return name
    if 'REF_' in name:
        return name.replace('REF_', '')
    else:
        return '_'.join(name.replace('ALT_', '').split('_')[:-1])

In [ ]:
pre_metadata_df = hf.fasta_to_dataframe('/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/removed_brackets_design_no_duplicates_sequence_and_header.fa', columns=[col_name, col_sequence])
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))
design_names = set(pre_metadata_df[col_name].to_list())
pre_metadata_df['test_name'] = pre_metadata_df[col_name].apply(rename_selvarajan_names)
renamed_names = set(pre_metadata_df['test_name'].to_list())
# check if the exchange worked:
print('the new names: ', renamed_names - design_names)
print('the old names: ', design_names - renamed_names)


{'GC_Selvarajan:rs754064|STARR-seq-HepG2_fwd_tile1-1', 'GC_Selvarajan:rs216222|STARR-seq-HepG2_fwd_tile1-1', 'GC_Selvarajan:rs2297787|STARR-seq-HepG2_fwd_tile1-1'}
{'GC_Selvarajan:REF_rs754064|STARR-seq-HepG2_fwd_tile1-1', 'GC_Selvarajan:ALT_rs2297787|STARR-seq-HepG2_fwd_tile1-1_rs2297787', 'GC_Selvarajan:REF_rs216222|STARR-seq-HepG2_fwd_tile1-1'}


In [18]:
# rename the columns
pre_metadata_df[col_name] = pre_metadata_df['test_name']
hf.write_fasta(pre_metadata_df, '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/removed_brackets_design_no_duplicates_sequence_and_header.fa', header=[col_name, 'sequence'])

True

{'GC_Selvarajan:ALT_rs2297787|STARR-seq-HepG2_fwd_tile1-1_rs2297787',
 'GC_Selvarajan:REF_rs216222|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Selvarajan:REF_rs754064|STARR-seq-HepG2_fwd_tile1-1'}

In [ ]:
group_name = 'GC_Selvarajan'
selvarajan_design = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name]


In [2]:
example_header = 'GC_Selvarajan:REF_rs216222|STARR-seq-HepG2_fwd_tile1-1'
example_header.replace('REF_', '')
example_header = 'GC_Selvarajan:ALT_rs2297787|STARR-seq-HepG2_fwd_tile1-1_rs2297787'
'_'.join(example_header.replace('ALT_', '').split('_')[:-1])


'GC_Selvarajan:rs2297787|STARR-seq-HepG2_fwd_tile1-1'

- collision handling: 
    - read the data: `/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/experiment/correct_umi_final_resequencing/design_check_collisions.err`
    - if 'negative_neuron_NP' in row: remove the elements which are not this element; if 'REF' in row remove the others, elif 'reference' in row: take it elif take first elmenent: 

In [134]:
# list of headers to be removed:
# 1,2: collision with reference
# 3: collision with C_positive_neuron_NP
# 4, 5, 6: same group (GC_Mendelian_variants): remove second
# 7: collision with reference (MK)
# 8: same group (GC_Mendelian_variants): remove second
# 9, 10: same group (MK): remove second
# 11, 12: same group (GC_Mendelian_variants), both references: remove second
# 13: same group (MK), both references: remove second
# 14: collision with C_negaive_neuron_NP
# 15: MK variants: removed second
# 16: GC_Mendelian_variants: both reference: removed second
# 17: MK: both references: removed second
# 18: collision with G_postive_neuron_NP: removed MK reference
# 19, 20: same group (GC_Mendelian_variants): remove second
# 21, 22, 23: same group (MK): remove second
# 24: both references (MK): removed second
# 25: GC_Selvarajan collision with cardiac_neuro_cava_random: removed GC_Selvarajan
# 26, 27: same group (GC_Mendelian_variants): remove second
# 28, 29: collision of C_positive_neuron_MK of 1 reference with 2 variants: removed variants
# 30: collision MK reference with variant: removed variant
# 31: collision MK variant and reference: removed variant
# 32: collision GC_Mendelian_variants: removed second
# 33: collision GC_Mendelian_variants: removed second
# 34: collision MK: removed second
# 35: collision GC_Mendelian_variants: removed second
# 35: collision GC_Mendelian_variants: removed second
# 36, 37: collision: MK removed variant
# 38, 39, 40: collision: GC_Mendelian_variants removed second
# 41: collision C_negative_neuron_MK: GC_Vista removed
# 42, 43: collision: GC_Mendelian_variants removed second
# 44: collision: MK removed second
# 45: collision: MK removed variant
# 46: collision: MK removed second
# 47: collision: GC_Mendelian_variants removed second
# 48: collision: MK removed variant
# 49, 50: collision: GC_Mendelian_variants removed second
# 51: collision MK reference and C_positive_neuron_MK: removed MK reference
# 52: collision MK: removed second
# 53: collision GC_salvarajan and cardiac_neuro_cava_random: removed GC_salvarajan
# 54, 55: collision: GC_Mendelian_variants removed second
# 56: collision: MK newcore vs tile: removed tile
# 57: collision MK: removed second
# 58: collision C_positive_neuron_NP vs MK newcore reference: removed MK newcore reference
# 59: collisoin MK: removed second
# 60: collision GC_Mendelian_variants: removed second
# 61: important: collision: Forward collision 59 for sequences:     cardiac_neuro_cava_random:REF_CELF2|ENSG00000048740.19|EH38E1447739_fwd_tile1-1 C_negative_neuron_NP:Fetal_Cerebrum_Cicero_chr10_11131297_11131567_2.37359878574768: removed C_negative_neuron_NP
# 62, 63: collision (MK): reference with two variants
# 64: collision (GC_Mendelian_variants): removed second
# UNTIL: Forward collision 62 for sequences:
# 65, 66: collision (GC_Mendelian_variants): removed second
# 67: collision (vista vs cardiac_neuro_cava_random): removed vista
# 68: collision (GC_Mendelian_variants) two references: removed second
# 69: collision (MK) two references: removed second
# 70: collision (MK) two variants vs reference: removed variants
# 71: collision (GC_Mendelian_variants): removed second
# 72: collision (MK): removed second
# 73: collision (GC_Mendelian_variants): removed second

removable_header_bc_of_collisions = ['MK:tile_19098|chr18-25465452+25465722|G-A-1', 'MK:tile_19098|chr18-25465452+25465722|A-G-268', 'MK:tile_37449|chr6-98703408+98703677|reference', 'GC_Mendelian_variants:ALT_chr7:156791255G*C|SHH_chr7:156791274T*TTAAGGAAGTGATT|SHH', 'GC_Mendelian_variants:ALT_chr7:156791581A*G|SHH_chr7:156791579C*T|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791542A*C|SHH', 'MK:tile_47638|chr16-52435608+52435878|A-T-1', 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791581A*G|SHH', 'MK:tile_47629|chr14-103542886+103543156|C-G-33', 'MK:tile_47742|chr5-87944920+87945190|G-C-165', 'GC_Mendelian_variants:REF_chr7:156791472C*G|SHH',
 'GC_Mendelian_variants:REF_chr7:156791579C*T|SHH', 'MK:tile_47629|chr14-103542886+103543156|reference', 'GC_Vista:fb;fm_mm1912_vistaElementControl|chr7:42140681-42140930', 'MK:tile_47627|chr14-103542806+103543076|T-A-248', 'GC_Mendelian_variants:REF_chr10:23219434A*G|PTF1A', 'MK:tile_47626|chr14-103542805+103543075|reference',
 'MK:tile_37403|chr6-98416848+98417117|reference', 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791459T*C|SHH', 'GC_Mendelian_variants:ALT_chr7:156791255G*C|SHH_chr7:156791255G*C|SHH', 'MK:tile_47744|chr5-87945000+87945270|T-A-62', 'MK:tile_47743|chr5-87944921+87945191|T-A-141', 'MK:tile_47626|chr14-103542805+103543075|C-G-114',
 'MK:tile_47745|chr5-87945001+87945271|reference', 'GC_Selvarajan:ALT_rs499966|STARR-seq-HepG2_fwd_tile1-1_rs499966', 'GC_Mendelian_variants:ALT_chr7:156791474G*A|SHH_chr7:156791571T*A|SHH', 'GC_Mendelian_variants:ALT_chr10:23219436A*G|PTF1A_chr10:23219434A*G|PTF1A',
 'C_positive_neuron_MK:tile_34824_chr5_141041068_141041337_A_T_1_0.562204129450572', 'C_positive_neuron_MK:tile_34824_chr5_141041068_141041337_G_C_269_0.550310056665328', 'MK:tile_47785|chr6-164344756+164345026|G-C-269', 'MK:tile_47725|chr4-125519552+125519822|T-A-269', 'GC_Mendelian_variants:ALT_chr10:23219434A*G|PTF1A_chr10:23219517A*C|PTF1A',
 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791547A*G|SHH', 'MK:tile_47629|chr14-103542886+103543156|T-A-168', 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791472C*G|SHH', 'GC_Mendelian_variants:ALT_chr10:23219434A*G|PTF1A_chr10:23219376A*C|PTF1A', 'MK:tile_47562|chr1-52663056+52663326|C-G-1', 'MK:tile_47791|chr6-170438656+170438926|G-C-1',
 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791480G*A|SHH', 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791472C*T|SHH', 'GC_Mendelian_variants:ALT_chr7:156791474G*A|SHH_chr7:156791542A*C|SHH', 'GC_Vista:fb_hs262_vistaElementControl|chr5:77645014-77645263', 'GC_Mendelian_variants:ALT_chr7:156791474G*A|SHH_chr7:156791581A*G|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791255G*C|SHH_chr7:156791257G*A|SHH', 'MK:tile_47744|chr5-87945000+87945270|G-C-265', 'MK:tile_47747|chr5-87945240+87945510|T-A-1', 'MK:tile_47627|chr14-103542806+103543076|C-G-129', 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791474G*A|SHH', 'MK:tile_47598|chr12-102961789+102962059|A-T-1',
 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791472C*T|SHH', 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791579C*T|SHH', 'MK:rdhs_334645|chr2-6772346+6772615|reference', 'MK:tile_47744|chr5-87945000+87945270|T-A-241', 'GC_Selvarajan:REF_rs499966|STARR-seq-HepG2_fwd_tile1-1', 'GC_Mendelian_variants:ALT_chr7:156791474G*A|SHH_chr7:156791413A*C|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791581A*G|SHH_chr7:156791472C*G|SHH', 'MK:tile_44725|chr8-127646130+127646399|reference', 'MK:tile_47629|chr14-103542886+103543156|C-G-49', 'MK:newcore_405894|chrX-71182006+71182275|reference', 'GC_Mendelian_variants:ALT_chr7:156791474G*A|SHH_chr7:156791547A*G|SHH', 'C_negative_neuron_NP:Fetal_Cerebrum_Cicero_chr10_11131297_11131567_2.37359878574768',
 'MK:tile_985|chr1-33363965+33364235|C-T-268', 'MK:tile_985|chr1-33363965+33364235|T-C-1', 'GC_Mendelian_variants:ALT_chr10:23219436A*G|PTF1A_chr10:23219436A*G|PTF1A', 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791571T*A|SHH', 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791474G*A|SHH', 'GC_Vista:fb;mb__vistaElementControl|chr14:78308172-78308421',
 'GC_Mendelian_variants:REF_chr7:156791255G*C|SHH', 'MK:tile_47743|chr5-87944921+87945191|reference', 'MK:tile_10173|chr12-113932852+113933122|C-T-268', 'MK:tile_10173|chr12-113932852+113933122|G-A-1', 'GC_Mendelian_variants:ALT_chr10:23219434A*G|PTF1A_chr10:23219508A*G|PTF1A', 'MK:tile_47744|chr5-87945000+87945270|G-C-85', 'GC_Mendelian_variants:ALT_chr7:156791581A*G|SHH_chr7:156791480G*A|SHH'
]

In [165]:
print(len(removable_header_bc_of_collisions))
print(len(set(removable_header_bc_of_collisions))) # all unique => sanity check

74
74


In [136]:
# read design file:
# design_file_with_collisions = hf.fasta_to_dataframe('/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/removed_brackets_design_no_duplicates_sequence_and_header.fa')
# # remove these headers:
# design_file_without_collisions = design_file_with_collisions.loc[~design_file_with_collisions['header'].isin(removable_header_bc_of_collisions)]
# design_file_of_collisions = design_file_with_collisions.loc[design_file_with_collisions['header'].isin(removable_header_bc_of_collisions)]

# # write both files:
# hf.write_fasta(design_file_without_collisions, '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header_with_adapter_no_brackets_no_collisions.fa')
# hf.write_fasta(design_file_of_collisions, '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header_with_adapter_no_brackets_collisions.fa')

In [137]:
design_file_without_collisions

NameError: name 'design_file_without_collisions' is not defined

In [ ]:
design_file_of_collisions

In [81]:
# ! python /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/scripts/rename_file_2_design_style.py --input-file /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header_with_adapter_no_brackets_no_collisions.fa --output-file /home/kisa/coding/80K_MPRA/design_data/design_info/renamed_design_no_duplicates_sequence_and_header_with_adapter_no_brackets_no_collisions.fa --file-type design_fasta

In [82]:
# # prepare label file
# ( echo -e "name\tlabel"; \
#     grep "^>" /home/kisa/coding/80K_MPRA/design_data/design_info/renamed_design_no_duplicates_sequence_and_header_with_adapter_no_brackets_no_collisions.fa | sed 's/^>//g' | awk -F ":" '{print $0"\t"$1}'; ) \
# > /home/kisa/coding/80K_MPRA/design_data/design_info/renamed_design_no_duplicates_sequence_and_header_with_adapter_no_brackets_no_collisions_label.tsv

### Investigate the design file
- how many regions are there? 
- how many controls? 
- how many variants? 

### Add the variant information to the variantis in the variant_region_map.tsv file
- go through the metadata file: 
  - add REF, ALT (for each variant) NA otherwise
  - add region_id column 
    - variants: left join on REF / ALT 
    - elements: left join on name
    - region.bed: check how many regions: 28390
    - variant_region_map: check how many regions: 20035
    

In [138]:
# read current metadata from tested sequences

# read current metadata from control sequences

##### Check the other way around: can all variant region map alt be matched? - expectation: yes

In [7]:
# what is the number of unique values in the ALT column?
print(F"Unique variants: {tested_variant_map['ALT'].nunique()}")
# merge renamed variant region map with design file => check if same number of variants still there and if all recognized as ALT
var_map_alternate_sequences_matching = tested_variant_map.merge(tested_meta_data_file[[col_header, col_category]], left_on='ALT', right_on=col_header, how='left')
print(var_map_alternate_sequences_matching.loc[~var_map_alternate_sequences_matching[col_header].isna()][col_category].value_counts()) # => same number => worked well
# what is the number of unique values in the REF column?
print(F"Unique references associated to variant: {tested_variant_map['REF'].nunique()}")
# merge renamed variant region map with design file => check if same number of variants still there and if all recognized as REF
var_map_reference_sequences_matching = tested_variant_map.merge(tested_meta_data_file[[col_header, col_category, col_allele]], left_on='REF', right_on=col_header, how='inner')
print(var_map_reference_sequences_matching.loc[~var_map_reference_sequences_matching[col_header].isna()][col_header].nunique()) # => same number => worked well


Unique variants: 46374
variant    46374
Name: category, dtype: int64
Unique references associated to variant: 18572
18572


#### Found ALT sequences which are not in the variant region map

In [8]:
# find elements with ALT_: check for headers which are not in REF_ID and ALT of variant region map
element_tested_meta = tested_meta_data_file.loc[tested_meta_data_file[col_category] == 'element']
# check for headers with ALT_
element_tested_meta.loc[element_tested_meta[col_header].str.contains('ALT_')].shape[0]
element_tested_meta.loc[element_tested_meta[col_header].str.contains('ALT_')][col_header].to_list()

[]

- I found in GC_Selvarajan again alt and ref which need to be changed to match the region bed (see above for the code to remove these)

### Split metadata file into Tested and Control
- Check if all regions in region.bed can be found in the design fasta: yes they can
- Tested:
  - add regions of variants: 
    - add variant map information (use Region to add region information) 
    - add SPDI 
    - (Verify 3 sample results)
- Control:
  - add regions of variants: 
    - add variant map information (use Region to add region information)
    - How to add SPDI?
    - How to get variant position?  
    - (Verify 3 sample results)
  - Use Region.bed to add regions also to other controls (How well does this work? Alternatives?)

In [169]:
meta_data_file.shape

(73846, 32)

In [170]:
meta_data_file.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_region_id', 'tmp_open_NGN2_WTC11_NP_screen_regions',
       'tmp_chr_start_end', 'tmp_open_NGN2_WTC11_NP',
       'tmp_open_h1_neuro_dnase_screen', 'tmp_MPRA_primary_region_overlap',
       'tmp_MPRA_organoid_region_overlap', 'tmp_gene_name', 'tmp_gene_set',
       'tmp_is_neuro', 'tmp_is_cardiac', 'tmp_is_cava', 'tmp_is_random'],
      dtype='object')

In [171]:
# split metadata into tested and controls and modify them individually
tested_meta_data_file
tested_meta_data_file = meta_data_file.loc[meta_data_file[col_name].str.startswith('cardiac_neuro_cava_random')]



# control_meta_data_file
control_meta_data_file = meta_data_file.loc[~meta_data_file[col_name].str.startswith('cardiac_neuro_cava_random')]
control_meta_data_file['tmp_label'].value_counts()

Series([], Name: tmp_label, dtype: int64)

In [172]:
meta_data_file['tmp_label'].value_counts()

cardiac_neuro_cava_random    73846
Name: tmp_label, dtype: int64

#### Tested
- add chr start end for all tested sequences
- add a region column: if allele: 'alt' => variant region map => ALT_ID else NA
- add if open in WTC11

In [16]:
# read renamed bed file
region_bed = pd.read_csv(config['files']['final_design']['region_bed'], sep='\t', header=None)
region_bed.columns = ['region_chr', 'region_start', 'region_end', 'region_name', 'region_score', 'region_strand']
region_bed




,region_chr,region_start,region_end,region_name,region_score,region_strand
0,chr1,2179507,2179777,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
1,chr1,2181843,2182113,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
2,chr1,2182439,2182709,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
3,chr1,2182830,2183100,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
4,chr1,2185027,2185297,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
...,...,...,...,...,...,...
28385,chrX,154531979,154532249,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28386,chrX,154539055,154539325,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28387,chrX,154545000,154545270,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-
28388,chrX,154549802,154550072,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,.,-


In [17]:
tested_meta_data_file.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref', 'chr',
       'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI',
       'allele', 'info'],
      dtype='object')

##### Focus on variants: 

In [15]:
# load variant region map
variant_region_map = pd.read_csv(config['files']['final_design']['variant_table'], sep='\t', header=None)
variant_region_map.columns = ['ID', 'Region', 'REF_ID', 'ALT_ID']
tested_variant_region_map = variant_region_map.loc[variant_region_map['ID'].str.startswith('cardiac_neuro_cava_random')]
print(tested_variant_region_map.shape[0]) # 46374

46374


###### Split Metadata file in alt and reference variant related sequences
- optional load the metadata file you want to use here

In [18]:
# optional: read file: /home/kisa/coding/80K_MPRA/metadata_info/metadata_tested_region_2010_validate.tsv
metadata_file_path = '/home/kisa/coding/80K_MPRA/metadata_info/metadata_tested_region_2010_validate.tsv'
tested_meta_data_file = pd.read_csv(metadata_file_path, sep='\t', low_memory=False)

# list columns col_variant_class, col_variant_pos, col_SPDI, col_allele,
list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]
# Apply the safe_eval function to the specified columns
for col in list_columns:
    tested_meta_data_file[col] = tested_meta_data_file[col].apply(safe_eval)

tested_meta_data_file.head()

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2181843,2182113,+,None,None,None,None,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2182439,2182709,+,None,None,None,None,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2182830,2183100,+,None,None,None,None,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2185027,2185297,+,None,None,None,None,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2188389,2188659,+,None,None,None,None,cardiac_neuro_cava_random


In [19]:
alt_tested_meta_data_file = tested_meta_data_file.loc[tested_meta_data_file[col_allele].apply(hf.is_alternative)]
ref_tested_meta_data_file = tested_meta_data_file.loc[tested_meta_data_file[col_allele].apply(hf.is_reference)]

Add genomic coordinates to variants
- match with variant region map 
- use ALT_ID for alternative use REF for for references

In [20]:
alt_tested_meta_data_file = alt_tested_meta_data_file.merge(variant_region_map[['ALT_ID', 'Region']], left_on=col_header, right_on='ALT_ID', how='left')
# sanity check if all are matched:
print(f"Number of unmatched alternative sequences with variant region map: {alt_tested_meta_data_file.loc[alt_tested_meta_data_file['Region'].isna()].shape[0]}") # 0

# same for reference
# drop duplicated rows for reference
reference_variant_region_map = variant_region_map[['REF_ID', 'Region']].drop_duplicates()
ref_tested_meta_data_file = ref_tested_meta_data_file.merge(reference_variant_region_map, left_on=col_header, right_on='REF_ID', how='left')
# sanity check if all are matched:
print(f"Number of unmatched reference sequences with variant region map: {ref_tested_meta_data_file.loc[ref_tested_meta_data_file['Region'].isna()].shape[0]}") # 0

# Number of alternative and reference sequences respectively
print(f"Number of alternative sequences: {alt_tested_meta_data_file.shape[0]}")
print(f"Number of alternative sequences: {alt_tested_meta_data_file[col_name].nunique()}")
print(f"Number of reference sequences: {ref_tested_meta_data_file.shape[0]}")
print(f"Number of reference sequences: {ref_tested_meta_data_file[col_name].nunique()}")

# clean up: remove REF and ALT_ID columns
alt_tested_meta_data_file.drop(columns=['ALT_ID'], inplace=True)
ref_tested_meta_data_file.drop(columns=['REF_ID'], inplace=True)

Number of unmatched alternative sequences with variant region map: 0
Number of unmatched reference sequences with variant region map: 0
Number of alternative sequences: 46374
Number of alternative sequences: 46374
Number of reference sequences: 18572
Number of reference sequences: 18572


In [21]:
# test for reference sequences to merge with bed
ref_tested_meta_data_file = ref_tested_meta_data_file.merge(region_bed, left_on='Region', right_on='region_name', how='left')
# sanity check if all are matched:
print(f"Number of unmatched reference sequences with region bed: {ref_tested_meta_data_file.loc[ref_tested_meta_data_file['region_start'].isna()].shape[0]}") # 0

# Check shape:
print(f"Shape of reference sequences: {ref_tested_meta_data_file.shape[0]}")
print(f"Shape of reference sequences (nunique): {ref_tested_meta_data_file[col_name].nunique()}")

# drop score column
# ref_tested_meta_data_file[['name', 'sequence', 'region_chr', 'region_start', 'region_end', 'region_strand']]['sequence'].to_list()[0][15:]
ref_tested_meta_data_file_test = ref_tested_meta_data_file.drop(columns=['region_score'])
# sanity check for one example: (UCSC result: https://genome-euro.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A2179507%2D2179527&hgsid=343387888_iQXjnQZYpq2C3OuXkQ0df7Tbpnem)

ref_tested_meta_data_file_test.columns
# rename the column names: 'region_chr', 'region_start', 'region_end', 'region_strand' chr, start, end, strand
# drop the already existing columns: chr, start, end, strand

ref_tested_meta_data_file_test.drop(columns=[col_chr, col_start, col_end, col_strand], inplace=True)
# ref_tested_meta_data_file_test = ref_tested_meta_data_file_test.drop(columns=['region_chr', 'region_start', 'region_end', 'region_strand'])
ref_tested_meta_data_file_test = ref_tested_meta_data_file_test.rename(columns={'region_chr': col_chr, 'region_start': col_start, 'region_end': col_end, 'region_strand': col_strand})
ref_tested_meta_data_file_test
ref_tested_meta_data_file_test.columns

print(f"Number of na values in start: {ref_tested_meta_data_file_test[col_start].isna().sum()}")

# clean up: remove all columns with "region_" prefix
columns_to_drop = [col for col in ref_tested_meta_data_file.columns if col.startswith('region_') or col == 'REF_ID']
ref_tested_meta_data_file.drop(columns=columns_to_drop, inplace=True)

Number of unmatched reference sequences with region bed: 0
Shape of reference sequences: 18572
Shape of reference sequences: 18572
Number of na values in start: 0


In [180]:
# ref_tested_meta_data_file_test.loc[ref_tested_meta_data_file_test[col_start].isna()]['name'].to_list()

[]

In [22]:
alt_tested_meta_data_file.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref', 'chr',
       'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI',
       'allele', 'info', 'Region'],
      dtype='object')

In [23]:
ref_tested_meta_data_file.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref', 'chr',
       'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI',
       'allele', 'info', 'Region'],
      dtype='object')

In [24]:
# combine ref and alternative sequences metadata table again

# alt_tested_meta_data_file = tested_meta_data_file.loc[tested_meta_data_file['allele'] == 'alt']
# ref_tested_meta_data_file = tested_meta_data_file.loc[tested_meta_data_file['allele'] == 'ref']

variant_tested_meta_data_file = pd.concat([alt_tested_meta_data_file, ref_tested_meta_data_file], ignore_index=True)


# test for reference sequences to merge with bed
variant_tested_meta_data_file = variant_tested_meta_data_file.merge(region_bed, left_on='Region', right_on='region_name', how='left')
# sanity check if all are matched:
print(f"Number of unmatched variant sequences with region bed: {variant_tested_meta_data_file.loc[variant_tested_meta_data_file['region_start'].isna()].shape[0]}") # 0

# Check shape:
print(f"Shape of variant metadata: {variant_tested_meta_data_file.shape[0]} (expected: 64946)")

# drop score column
variant_tested_meta_data_file = variant_tested_meta_data_file.drop(columns=['region_score'])
# sanity check for one example: (UCSC result: https://genome-euro.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A2179507%2D2179527&hgsid=343387888_iQXjnQZYpq2C3OuXkQ0df7Tbpnem)

variant_tested_meta_data_file.columns
# rename the column names: 'region_chr', 'region_start', 'region_end', 'region_strand' chr, start, end, strand
# drop the already existing columns: chr, start, end, strand
variant_tested_meta_data_file.drop(columns=[col_chr, col_start, col_end, col_strand], inplace=True)

variant_tested_meta_data_file = variant_tested_meta_data_file.rename(columns={'region_chr': col_chr, 'region_start': col_start, 'region_end': col_end, 'region_strand': col_strand})
variant_tested_meta_data_file

print(f"Number of na values in start: {variant_tested_meta_data_file[col_start].isna().sum()}") # 0

Number of unmatched variant sequences with region bed: 0
Shape of variant metadata: 64946 (expected: 64946)
Number of na values in start: 0


In [25]:
# sanity check for the numbers of ref and alternative sequences:
# variant_tested_meta_data_file.columns
variant_tested_meta_data_file[col_name].nunique() # 46374 + 18572 = 64946

64946

Add variant position to variants

In [26]:
def get_variant_position(row, header_col='name', with_adapter=False):
    """
    Returns the 0-based variant position if it is a variant (start is 0-based as well)
    Start is 0-based
    Position is 1-based
    """
    adapter_count = 0
    if with_adapter:
        adapter_count = 15
    if row[col_category] != 'variant' or ':REF_' in row[header_col]:
        row[col_variant_pos] = np.nan
    else: # compute variant position based on chr-pos-ref-alt pattern
        chrom_pos_ref_alt = get_chrom_pos_ref_alt_pattern(row[header_col])
        chrom, pos, ref, alt = chrom_pos_ref_alt.split('-')
        row[col_variant_pos] = (int(pos) - 1) - row[col_start] # 0-based
        if row[col_strand] == "-":
            row[col_variant_pos] = 270 - row[col_variant_pos] - 1 # 0-based
        row[col_variant_pos] = row[col_variant_pos] + adapter_count
    return row

In [27]:
variant_tested_meta_data_file = variant_tested_meta_data_file.apply(get_variant_position, axis=1)
# please upload without the adapter => string position of variant is withouth adapter

In [187]:
# # sanity check of variant position:
# variant_tested_meta_data_file_var_position.loc[variant_tested_meta_data_file_var_position[col_variant_pos].isna()].shape[0] # 18572 == #REF
# variant_tested_meta_data_file_var_position.loc[~variant_tested_meta_data_file_var_position[col_variant_pos].isna()][[col_sequence, col_name, col_start, col_end, col_variant_pos, col_strand]].head()
# # example: cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778490_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778490|1-2191444-G-A start: 2191262; end: 2191532
# variant_tested_meta_data_file_var_position.loc[~variant_tested_meta_data_file_var_position[col_variant_pos].isna()][[col_sequence, col_name, col_start, col_end, col_variant_pos, col_strand]][col_sequence].to_list()[1]
# ## ref: AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCTCAGTAGCTGAGCACGCCCTGTGGTGTGGACAGGCCAGCCCGGCTGCAGGACAGTGGGGGCTCAGTCCAGAAGCCTCTGGTGGCTTCTGCGTGTGGGCAGGGGAGCAGGGCTGCAGGGTGGGCTGGCACCTGGCAGCGTGAGGGGCCTCAGGGCTGTTAAAATCCCAACGACCGGCCGGGCGCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCCGAGACGGGCGGATCACCTGCATTGCGTGAACCGA
# ## alt: AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCTCAGTAGCTGAGCACGCCCTGTGGTGTGGACAGGCCAGCCCGGCTGCAGGACAGTGGGGGCTCAGTCCAGAAGCCTCTGGTGGCTTCTGCGTGTGGGCAGGGGAGCAGGGCTGCAGGGTGGGCTGGCACCTGGCAGCGTGAGGGGCCTCAGAGCTGTTAAAATCCCAACGACCGGCCGGGCGCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCCGAGACGGGCGGATCACCTGCATTGCGTGAACCGA
# #      '--------adapter-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------'

# are the forward strand variants zero-based? (goal: 233)
# e.g. cardiac_neuro_cava_random:ALT_PRDM16|ENSG00000142611.17|EH38E2780177_fwd_tile1-1_PRDM16|ENSG00000142611.17|EH38E2780177|1-3467717-C-T
# AGGACCGGATCAACTTCGGCTGTGGGTGGCAGACCTGGGGGTCTCCCTCCTGTGCCCTCAGAGTCACCCTGCCCATGCAGGGCCCTGCCGTCGTGCGTGCGAGGGCCAGCCCTGCTCAGACTACCCTTCAGGGAGCTGCGGGCGCCAGGAGTCCCTTCCCAGGGCCCCCGCCCCAGGAGAGGGCTGAATGATCTGCGTAGGGAGACAGCTGGGATGGGAGGGGCCGGGACTTGGAGCTGGGGCCAGGGTCAAGACACTCCTTCACTTTCCATATCTGAAATTGGGCATTGCGTGAACCGA
# cardiac_neuro_cava_random:REF_PRDM16|ENSG00000142611.17|EH38E2780177_fwd_tile1-1
# AGGACCGGATCAACTTCGGCTGTGGGTGGCAGACCTGGGGGTCTCCCTCCTGTGCCCTCAGAGTCACCCTGCCCATGCAGGGCCCTGCCGTCGTGCGTGCGAGGGCCAGCCCTGCTCAGACTACCCTTCAGGGAGCTGCGGGCGCCAGGAGTCCCTTCCCAGGGCCCCCGCCCCAGGAGAGGGCTGAATGATCTGCGTAGGGAGACAGCTGGGATGGGAGGGGCCGGGACTTGGAGCTGGGGCCAGGGCCAAGACACTCCTTCACTTTCCATATCTGAAATTGGGCATTGCGTGAACCGA

# reverse strand variant position correct? (they are not 0-based)
# e.g. cardiac_neuro_cava_random:ALT_RERE|ENSG00000142599.20|EH38E2783639_rev_tile1-1_RERE|ENSG00000142599.20|EH38E2783639|1-8313001-C-T
# AGGACCGGATCAACTCAAAATTACAGTCCTGTAATGACAACACATTGCAAACTCCAGAGGACAGGGCCATCCAATTCTTTTTTGGCACCTCCTGCAACACCTGGCAGAGGCCGAGCACTGACACCCACATGCTTATTTACTGTACGTGGAGTAGGATCGGTTGAAGTTGCGCGATTTACCCAGGCACACACGGGCACACTGTGGGATAAAACAGATTGTCCTGGGCTGCCCTGGGCATGCCGGTGAAACTCCATTGAACCATTTCTGCAGGAAAATGAATTCTGACATTGCGTGAACCGA
# cardiac_neuro_cava_random:REF_RERE|ENSG00000142599.20|EH38E2783639_rev_tile1-1
# AGGACCGGATCAACTCAAAATTACAGTCCTGTAATGACAACACATTGCAAACTCCAGAGGACAGGGCCATCCAATTCTTTTTTGGCACCTCCTGCAACACCTGGCAGAGGCCGAGCACTGACACCCACATGCTTATTTACTGTACGTGGAGTAGGATCGGTTGAAGTTGCGCGATTTACCCGGGCACACACGGGCACACTGTGGGATAAAACAGATTGTCCTGGGCTGCCCTGGGCATGCCGGTGAAACTCCATTGAACCATTTCTGCAGGAAAATGAATTCTGACATTGCGTGAACCGA

In [188]:
# merging_region_bed_tested_metadata = variant_tested_meta_data_file.merge(region_bed, left_on=col_name, right_on='region_name', how='left')

In [203]:
# if you want to add the SPDI on all rows not only to variants (will not change anything but variant and element df will be concatenated later (cleaner to work with individually))
# tested_meta_data_file = tested_meta_data_file.apply(lambda row: get_spdi(row, header_col=col_name), axis=1)
# tested_meta_data_file.loc[tested_meta_data_file['allele'] == 'alt']['SPDI']
# # validate spdi with
# # write SPDI column to file and test with spdi_batch.py
# tested_meta_data_file.loc[~tested_meta_data_file['SPDI'].isna()][['SPDI']].to_csv('tested_variants_spdi_1607.csv', header=False, index=None)

# run python /home/kisa/coding/80K_MPRA/igvf_spdi_demo_mike/spdi_batch.py -i tested_variants_spdi_1607.csv -t SPDI> verifying_with_spdi_batch.csv (more info see https://github.com/mikelove/igvf_spdi_demo)
# if no "warnings" in resulting file the spdis are validated (may take a while)
# tested_meta_data_file.to_csv(config['files']['creating']['metadata_table_local_tested_sep'], sep="\t", index=False)

Add SPDI list of associated variants
- read variant region map 
- get ref_alt_matching dict: ref_id: [alt_id1, alt_id2, ...]
- from metadata file get alt_id to spdi dict: alt_id: SPDI (get_SPDI needed)
- add function to add SPDI lists for REF seqs which adds list of ['ref', 'ref', ...] to the allele column of reference
- added to `tested_meta_data_file`

In [28]:
variant_tested_meta_data_file = variant_tested_meta_data_file.apply(lambda row: get_spdi(row, header_col=col_name), axis=1)

In [29]:
variant_tested_meta_data_file[col_SPDI].isna().sum() # 18572

18572

In [30]:
variant_region_map = pd.read_csv(config['files']['final_design']['variant_table'], sep='\t',header=None)
variant_region_map.columns = ['ID', 'Region', 'REF_ID', 'ALT_ID']
variant_region_map

,ID,Region,REF_ID,ALT_ID
0,ID,Region,REF,ALT
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
...,...,...,...,...
47040,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618
47041,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757
47042,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373
47043,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656


In [31]:
# make dict for REF: [list of ALT_IDs associated to this REF] from the variant_region_map
ref_alt_dict = variant_region_map.groupby('REF_ID')['ALT_ID'].apply(list).to_dict()
ref_alt_dict

{'C_positive_heart_CAD:REF_rs1037169': ['C_positive_heart_CAD:ALT_rs1037169_rs1037169'],
 'C_positive_heart_CAD:REF_rs10418535': ['C_positive_heart_CAD:ALT_rs10418535_rs10418535'],
 'C_positive_heart_CAD:REF_rs1051338': ['C_positive_heart_CAD:ALT_rs1051338_rs1051338'],
 'C_positive_heart_CAD:REF_rs10750098': ['C_positive_heart_CAD:ALT_rs10750098_rs10750098'],
 'C_positive_heart_CAD:REF_rs10774625': ['C_positive_heart_CAD:ALT_rs10774625_rs10774625'],
 'C_positive_heart_CAD:REF_rs10811656': ['C_positive_heart_CAD:ALT_rs10811656_rs10811656'],
 'C_positive_heart_CAD:REF_rs1122608': ['C_positive_heart_CAD:ALT_rs1122608_rs1122608'],
 'C_positive_heart_CAD:REF_rs11556924': ['C_positive_heart_CAD:ALT_rs11556924_rs11556924'],
 'C_positive_heart_CAD:REF_rs12444113': ['C_positive_heart_CAD:ALT_rs12444113_rs12444113'],
 'C_positive_heart_CAD:REF_rs12721051': ['C_positive_heart_CAD:ALT_rs12721051_rs12721051'],
 'C_positive_heart_CAD:REF_rs12740374': ['C_positive_heart_CAD:ALT_rs12740374_rs12740374'

In [32]:
ref_alt_dict['cardiac_neuro_cava_random:REF_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1']

['cardiac_neuro_cava_random:ALT_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1_ANK3|ENSG00000151150.22|EH38E1470846|10-60353898-C-T',
 'cardiac_neuro_cava_random:ALT_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1_ANK3|ENSG00000151150.22|EH38E1470846|10-60353906-G-A',
 'cardiac_neuro_cava_random:ALT_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1_ANK3|ENSG00000151150.22|EH38E1470846|10-60353928-C-T']

In [33]:
# # make dict for ALT_ID to SPDI from the metadata table
# alt_spdi_dict = variant_tested_meta_data_file.loc[variant_tested_meta_data_file[col_allele].apply(hf.is_alternative)][['name', 'SPDI']].set_index('name').to_dict()['SPDI']
# # alt_spdi_dict = tested_meta_data_file.loc[tested_meta_data_file[col_allele] == 'alt'][['name', 'SPDI']].set_index('name').to_dict()['SPDI']
# alt_spdi_dict
# len(alt_spdi_dict) # using variant_tested_meta_data: 46374

In [34]:
# alt_spdi_dict

In [35]:
# make dict for REF_ID: [list of ALT_IDs associated to this REF] from the variant_region_map
ref_alt_dict = variant_region_map.groupby('REF_ID')['ALT_ID'].apply(list).to_dict()
# ref_alt_dict

# make dict for ALT_ID to SPDI from the metadata table
alt_spdi_dict = variant_tested_meta_data_file.loc[variant_tested_meta_data_file[col_allele].apply(hf.is_alternative)][['name', 'SPDI']].set_index('name').to_dict()['SPDI']
# alt_spdi_dict

# function to add a list of SPDI values from the REF to the metadata table
def add_spdi_values_2_reference(row):
    if row[col_allele] == 'ref':
        # check if row[col_name] is in ref_alt_dict
        if not row[col_name] in ref_alt_dict:
            # raise exception
            raise ValueError('Reference ID not found in ref_alt_dict')
        row[col_SPDI] = [alt_spdi_dict[alt_id] for alt_id in ref_alt_dict[row[col_name]]]
        row[col_allele] = ['ref' for _ in ref_alt_dict[row[col_name]]]
    return row

In [36]:
ref_alt_dict['cardiac_neuro_cava_random:REF_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1']

['cardiac_neuro_cava_random:ALT_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1_ANK3|ENSG00000151150.22|EH38E1470846|10-60353898-C-T',
 'cardiac_neuro_cava_random:ALT_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1_ANK3|ENSG00000151150.22|EH38E1470846|10-60353906-G-A',
 'cardiac_neuro_cava_random:ALT_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1_ANK3|ENSG00000151150.22|EH38E1470846|10-60353928-C-T']

In [37]:
alt_spdi_dict

{'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C': 'NC_000001.11:2179590:T:C',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778490_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778490|1-2191444-G-A': 'NC_000001.11:2191443:G:A',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778492_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778492|1-2192015-G-T': 'NC_000001.11:2192014:G:T',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E1311587_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E1311587|1-2192366-T-G': 'NC_000001.11:2192365:T:G',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778494_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778494|1-2193142-G-A': 'NC_000001.11:2193141:G:A',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778498_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778498|1-2194344-T-G': 'NC_000001.11:2194343:T:G',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG

In [38]:
alt_spdi_dict['cardiac_neuro_cava_random:ALT_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1_ANK3|ENSG00000151150.22|EH38E1470846|10-60353898-C-T']
alt_spdi_dict['cardiac_neuro_cava_random:ALT_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1_ANK3|ENSG00000151150.22|EH38E1470846|10-60353906-G-A']
alt_spdi_dict['cardiac_neuro_cava_random:ALT_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1_ANK3|ENSG00000151150.22|EH38E1470846|10-60353928-C-T']

'NC_000010.11:60353927:C:T'

In [39]:
ref_alt_dict

{'C_positive_heart_CAD:REF_rs1037169': ['C_positive_heart_CAD:ALT_rs1037169_rs1037169'],
 'C_positive_heart_CAD:REF_rs10418535': ['C_positive_heart_CAD:ALT_rs10418535_rs10418535'],
 'C_positive_heart_CAD:REF_rs1051338': ['C_positive_heart_CAD:ALT_rs1051338_rs1051338'],
 'C_positive_heart_CAD:REF_rs10750098': ['C_positive_heart_CAD:ALT_rs10750098_rs10750098'],
 'C_positive_heart_CAD:REF_rs10774625': ['C_positive_heart_CAD:ALT_rs10774625_rs10774625'],
 'C_positive_heart_CAD:REF_rs10811656': ['C_positive_heart_CAD:ALT_rs10811656_rs10811656'],
 'C_positive_heart_CAD:REF_rs1122608': ['C_positive_heart_CAD:ALT_rs1122608_rs1122608'],
 'C_positive_heart_CAD:REF_rs11556924': ['C_positive_heart_CAD:ALT_rs11556924_rs11556924'],
 'C_positive_heart_CAD:REF_rs12444113': ['C_positive_heart_CAD:ALT_rs12444113_rs12444113'],
 'C_positive_heart_CAD:REF_rs12721051': ['C_positive_heart_CAD:ALT_rs12721051_rs12721051'],
 'C_positive_heart_CAD:REF_rs12740374': ['C_positive_heart_CAD:ALT_rs12740374_rs12740374'

In [40]:
alt_spdi_dict

{'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C': 'NC_000001.11:2179590:T:C',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778490_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778490|1-2191444-G-A': 'NC_000001.11:2191443:G:A',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778492_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778492|1-2192015-G-T': 'NC_000001.11:2192014:G:T',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E1311587_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E1311587|1-2192366-T-G': 'NC_000001.11:2192365:T:G',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778494_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778494|1-2193142-G-A': 'NC_000001.11:2193141:G:A',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778498_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778498|1-2194344-T-G': 'NC_000001.11:2194343:T:G',
 'cardiac_neuro_cava_random:ALT_SKI|ENSG

In [41]:
variant_tested_meta_data_file = variant_tested_meta_data_file.apply(add_spdi_values_2_reference, axis=1)

In [ ]:
variant_tested_meta_data_file

In [42]:
# remove region:
variant_tested_meta_data_file.drop(columns=['Region'], inplace=True)

In [43]:
# tested_meta_data_file_ref_spdi_test = variant_tested_meta_data_file.apply(add_spdi_values_2_reference, axis=1)

variant_tested_meta_data_file.loc[variant_tested_meta_data_file['name'] == 'cardiac_neuro_cava_random:REF_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1']['SPDI'].to_list()

[['NC_000010.11:60353897:C:T',
  'NC_000010.11:60353905:G:A',
  'NC_000010.11:60353927:C:T']]

In [44]:
variant_tested_meta_data_file.loc[variant_tested_meta_data_file['name'] == 'cardiac_neuro_cava_random:REF_ANK3|ENSG00000151150.22|EH38E1470846_rev_tile1-1']['allele'].to_list()

[['ref', 'ref', 'ref']]

In [45]:
variant_tested_meta_data_file[col_SPDI].isna().sum() #0

0

In [ ]:
variant_tested_meta_data_file

Change SNP to SNV

In [254]:
snv_identifier = 'SNV'

# variant_class:
variant_tested_meta_data_file[col_variant_class] = variant_tested_meta_data_file.apply(lambda x: snv_identifier if x[col_category] == 'variant' else 'NA', axis=1)


Add array type to the alt rows as well (optional and might not be possible for the current state of the metadata file)

In [48]:
def make_list_allele_spdi_variant_pos_variant_class(row):
    """
    Make a list of the column entries for allele and SPDI
    """

    if row[col_category] == 'variant' and ':ALT_' in row[col_name]:
        row[col_variant_class] = [row[col_variant_class]]
        row[col_variant_pos] = [int(row[col_variant_pos])]
        row[col_SPDI] = [row[col_SPDI]]
        row[col_allele] = [row[col_allele]]
        return row
    row[col_variant_class] = [row[col_variant_class]]
    return row

# def make_list_allele_spdi_variant_pos_variant_class(row):
#     """
#     Make a list of the column entries for allele and SPDI
#     """
#     if ':ALT_' in row[col_name]:
#         row[col_variant_class] = [row[col_variant_class]]
#         return row
#     row[col_variant_class] = [row[col_variant_class]]
#     return row

variant_tested_meta_data_file = variant_tested_meta_data_file.apply(make_list_allele_spdi_variant_pos_variant_class, axis=1)

In [237]:
throw make ref with list of spdi and allele

SyntaxError: invalid syntax (171225028.py, line 1)

In [ ]:
variant_tested_meta_data_file

##### Focus on elements:
- Problem: we have REF sequences and ALT sequences which are not in the current variant region map and not in the region bed
1. Match all elements you can with the bed file directly
2. Report which are not in the bed file and if all of them have REF and ALT in the header
- all 8900 references could be matched with the region bed (94 (ALT/REF sequences are still not matchable with regions))

In [46]:
# sanity check: do allele == ref + allele = alt + category = element sum up to the total number of tested sequences?
# number of rows which have a list in allele column
# ref_number = tested_meta_data_file.loc[tested_meta_data_file['allele'].apply(lambda x: isinstance(x, list))].shape[0]
ref_number = tested_meta_data_file.loc[tested_meta_data_file['allele'].apply(hf.is_reference)].shape[0]
alt_number = tested_meta_data_file.loc[tested_meta_data_file['allele'].apply(hf.is_alternative)].shape[0]
element_number = tested_meta_data_file.loc[tested_meta_data_file['category'] == 'element'].shape[0]
print(f"Number sum: {ref_number + alt_number + element_number}")
print(f"Number of tested sequences: {tested_meta_data_file.shape[0]}") # 73940
# old style to count reference sequences:
# ref_number = tested_meta_data_file.loc[tested_meta_data_file['allele'] == 'ref'].shape[0]

Number sum: 73846
Number of tested sequences: 73846


In [47]:
tested_meta_data_file['allele'].value_counts()

alt    46374
ref    18572
Name: allele, dtype: int64

In [ ]:
element_tested_meta_data_file

In [50]:
element_tested_meta_data_file = tested_meta_data_file.loc[tested_meta_data_file['category'] == 'element']
element_tested_meta_data_file_region = element_tested_meta_data_file.merge(region_bed, left_on=col_name, right_on='region_name', how='left')

# element_tested_meta_data_file_region
# sanity check: how many matched:
print(f"Number of elements in the design: {element_tested_meta_data_file.shape[0]}")
print(f"Number of elements matched with region bed: {element_tested_meta_data_file_region.loc[~element_tested_meta_data_file_region['region_start'].isna()].shape[0]}")
matchable_element_tested_meta_data_file_region = element_tested_meta_data_file_region.loc[~element_tested_meta_data_file_region['region_start'].isna()] # 8900

Number of elements in the design: 8900
Number of elements matched with region bed: 8900


In [51]:
# clean the column names up
matchable_element_tested_meta_data_file_region = matchable_element_tested_meta_data_file_region.drop(columns=['region_score'])
# rename the column names: 'region_chr', 'region_start', 'region_end', 'region_strand' chr, start, end, strand
# drop the already existing columns: chr, start, end, strand
matchable_element_tested_meta_data_file_region.drop(columns=[col_chr, col_start, col_end, col_strand], inplace=True)
matchable_element_tested_meta_data_file_region = matchable_element_tested_meta_data_file_region.rename(columns={'region_chr': col_chr, 'region_start': col_start, 'region_end': col_end, 'region_strand': col_strand})

# make start and end column int
matchable_element_tested_meta_data_file_region[col_start] = matchable_element_tested_meta_data_file_region[col_start].astype(int)
matchable_element_tested_meta_data_file_region[col_end] = matchable_element_tested_meta_data_file_region[col_end].astype(int)

In [52]:
# investigate genomic position
print(matchable_element_tested_meta_data_file_region.columns)
matchable_element_tested_meta_data_file_region.loc[matchable_element_tested_meta_data_file_region['category'] == 'element'][[col_name, col_chr, col_start, col_end, col_strand, col_SPDI, col_allele]]
# matchable_element_tested_meta_data_file_region.loc[matchable_element_tested_meta_data_file_region['category'] == 'element'][[col_name, col_chr, col_start, col_end, col_strand, col_SPDI, col_allele]][col_name].to_list()

Index(['name', 'sequence', 'category', 'class', 'source', 'ref',
       'variant_class', 'variant_pos', 'SPDI', 'allele', 'info', 'chr',
       'start', 'end', 'region_name', 'strand'],
      dtype='object')


,name,chr,start,end,strand,SPDI,allele
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,chr1,2181843,2182113,+,None,None
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,chr1,2182439,2182709,+,None,None
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,chr1,2182830,2183100,+,None,None
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,chr1,2185027,2185297,+,None,None
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,chr1,2188389,2188659,+,None,None
...,...,...,...,...,...,...,...
8895,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,chrX,154376832,154377102,-,None,None
8896,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,chrX,154383755,154384025,-,None,None
8897,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,chrX,154402403,154402673,-,None,None
8898,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,chrX,154409693,154409963,-,None,None


##### Investigate which elements are not in the bed file and why
- 94 elements are not in the bed file 
  - 84 have ALT_ in their name
  - 10 have REF_ in their name
- Option: look in earlier version of variant region map

In [53]:
element_tested_meta_data_file_region.loc[element_tested_meta_data_file_region['region_start'].isna()].shape[0] # 94
element_tested_meta_data_file_region.loc[element_tested_meta_data_file_region['region_start'].isna()][col_name].to_list() # 94

[]

In [54]:
element_tested_meta_data_file_region.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref', 'chr',
       'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI',
       'allele', 'info', 'region_chr', 'region_start', 'region_end',
       'region_name', 'region_score', 'region_strand'],
      dtype='object')

In [296]:
meta_data_file.loc[meta_data_file['tmp_open_NGN2_WTC11_NP_screen_regions']]['tmp_region_id'].nunique() # 355
# meta_data_file.loc[meta_data_file['tmp_open_NGN2_WTC11_NP']]['tmp_region_id'].nunique() # 367
# ## investigate element regions:
# element_tested_meta_data_file_region['region_name'].nunique() # 8900
# element_tested_meta_data_file_region['tmp_is_neuro'].value_counts()
# # True     6580
# # False    2320

# # open:
# # h1:
# element_tested_meta_data_file_region['tmp_open_h1_neuro_dnase_screen'].value_counts()
# # False    8282
# # True      618
# open_h1_neuro_dnas_screen = element_tested_meta_data_file_region.loc[element_tested_meta_data_file_region['tmp_open_h1_neuro_dnase_screen']][col_name].to_list()
# # tmp_open_NGN2_WTC11_NP_screen_regions
# element_tested_meta_data_file_region['tmp_open_NGN2_WTC11_NP_screen_regions'].value_counts()
# # False    8788
# # True      112
# open_NGN2_WTC11_NP_screen_regions = element_tested_meta_data_file_region.loc[element_tested_meta_data_file_region['tmp_open_NGN2_WTC11_NP_screen_regions']][col_name].to_list()
# # tmp_open_NGN2_WTC11_NP
# element_tested_meta_data_file_region['tmp_open_NGN2_WTC11_NP'].value_counts()
# # False    8783
# # True      117
# open_NGN2_WTC11_NP = element_tested_meta_data_file_region.loc[element_tested_meta_data_file_region['tmp_open_NGN2_WTC11_NP']][col_name].to_list()

# # plot venn diagram of NGN2_WTC11_NP_screen_regions and NGN2_WTC11_NP and open_h1_neuro_dnase_screen
# import matplotlib.pyplot as plt
# from matplotlib_venn import venn3

# venn3([set(open_h1_neuro_dnas_screen), set(open_NGN2_WTC11_NP_screen_regions), set(open_NGN2_WTC11_NP)], ('open_h1_neuro_dnase_screen', 'open_NGN2_WTC11_NP_screen_regions', 'open_NGN2_WTC11_NP'))
# plt.show()

355

In [264]:
throw focused on varaints and elements

SyntaxError: invalid syntax (1679214442.py, line 1)

#### Concatenate tested metadata file
- varifying the missing numbers
- => finished for tested sequences (except for the 94 elements which we have no information about)
- TODO: find out if these sequences are in the significant set of activators or repressors

##### Check if both dfs have the same columns

In [55]:
variant_tested_meta_data_file.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref',
       'variant_class', 'variant_pos', 'SPDI', 'allele', 'info', 'chr',
       'start', 'end', 'region_name', 'strand'],
      dtype='object')

In [56]:
matchable_element_tested_meta_data_file_region.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref',
       'variant_class', 'variant_pos', 'SPDI', 'allele', 'info', 'chr',
       'start', 'end', 'region_name', 'strand'],
      dtype='object')

In [57]:
tested_meta_data_file_region = pd.concat([variant_tested_meta_data_file, matchable_element_tested_meta_data_file_region], ignore_index=True)
tested_meta_data_file_region.shape[0] # 73846 (94 missing (ALT / REF without entry in variant_region_map))

73846

In [58]:
print("Number of na values in the Metadata file")
print(tested_meta_data_file_region.isna().sum())

Number of na values in the Metadata file
name                 0
sequence             0
category             0
class                0
source               0
ref                  0
variant_class     8900
variant_pos      27472
SPDI              8900
allele            8900
info                 0
chr                  0
start                0
end                  0
region_name          0
strand               0
dtype: int64


In [59]:
# investigate SPDI missing values: reference and element sequences
no_SPDI = tested_meta_data_file_region.loc[tested_meta_data_file_region[col_SPDI].isna()] # 8900 elements
no_SPDI

no_variant_pos = tested_meta_data_file_region.loc[tested_meta_data_file_region[col_variant_pos].isna()] # 27472
no_variant_pos['allele'].isna().sum() # 8900 => elements
no_variant_pos['allele'].value_counts() # no 'alt' => only ref; 27472 - 8900 = 18572

# number of references in the metadata file
tested_meta_data_file_region.loc[(tested_meta_data_file_region[col_name].str.contains(':REF_')) & (tested_meta_data_file_region[col_category] == 'variant')].shape[0] # 18572

18572

In [60]:
tested_meta_data_file_region.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref',
       'variant_class', 'variant_pos', 'SPDI', 'allele', 'info', 'chr',
       'start', 'end', 'region_name', 'strand'],
      dtype='object')

In [61]:
interesting_columns = ['name', 'sequence', 'category', 'class',
       'source', 'ref', 'chr', 'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info']

In [63]:
# only if you are sure tested_meta_data_file_region[interesting_columns].to_csv(config['files']['creating']['region_metadata_table_local_tested_juli'], sep="\t", index=False)
tested_meta_data_file_region[interesting_columns].to_csv(config['files']['creating']['region_metadata_table_local_tested_oktober'], sep="\t", index=False)

#### Validate tested metadata file:
- checkfiles github: https://github.com/IGVF-DACC/checkfiles.git (src/schemas/table_schemas)
- easily check if metadata file is fine => only tested is fine
```
from frictionless import validate
schema_path = 'designed_sequences.json'
file_path = 'path/to/your/file/to/validate'
report = validate(file_path, schema = schema_path)
```

In [64]:
tested_meta_data_file_region[interesting_columns].dtypes

name             object
sequence         object
category         object
class            object
source           object
ref              object
chr              object
start             int64
end               int64
strand           object
variant_class    object
variant_pos      object
SPDI             object
allele           object
info             object
dtype: object

In [65]:
import json

def write_data_with_json(data_df, file_path, interesting_columns=['name', 'sequence', 'category', 'class',
       'source', 'ref', 'chr', 'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info']):
    """
    Write data with json dumps (for lists)
    """
    # Convert lists/arrays to JSON strings
    for column in data_df[interesting_columns].columns:
        if data_df[column].dtype == 'object' and column in ['allele', 'SPDI', 'variant_class']:
            data_df[column] = data_df[column].apply(lambda x: json.dumps(x) if isinstance(x, (list, np.ndarray)) else x)
    data_df[interesting_columns].to_csv(file_path, sep='\t', index=False, na_rep='NA')
    return data_df

In [66]:
# try making lists to json strings


# Convert lists/arrays to JSON strings
for column in tested_meta_data_file_region[interesting_columns].columns:
    if tested_meta_data_file_region[column].dtype == 'object' and column in ['allele', 'SPDI', 'variant_class']:
        tested_meta_data_file_region[column] = tested_meta_data_file_region[column].apply(lambda x: json.dumps(x) if isinstance(x, (list, np.ndarray)) else x)

# Write to TSV
# file_path = 'metadata_tested_region_2307.tsv'
# tested_meta_data_file_region[interesting_columns].to_csv(file_path, sep='\t', index=False)

In [67]:
write_data_with_json(tested_meta_data_file_region, 'metadata_tested_region_2010_json.tsv', interesting_columns=interesting_columns)

,name,sequence,category,class,source,ref,variant_class,variant_pos,SPDI,allele,info,chr,start,end,region_name,strand
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,"[""SNV""]",[83],"[""NC_000001.11:2179590:T:C""]","[""alt""]",cardiac_neuro_cava_random,chr1,2179507,2179777,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCT...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,"[""SNV""]",[181],"[""NC_000001.11:2191443:G:A""]","[""alt""]",cardiac_neuro_cava_random,chr1,2191262,2191532,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCTGGGTGACCCGGAGAACACCAAGGCTG...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,"[""SNV""]",[43],"[""NC_000001.11:2192014:G:T""]","[""alt""]",cardiac_neuro_cava_random,chr1,2191971,2192241,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCATGCGGTGGCCACAGCCTCGGGTGAGTTC...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,"[""SNV""]",[116],"[""NC_000001.11:2192365:T:G""]","[""alt""]",cardiac_neuro_cava_random,chr1,2192249,2192519,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGGACTCCGGTGCCTTCGCATTCCCGAGCTGT...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,"[""SNV""]",[205],"[""NC_000001.11:2193141:G:A""]","[""alt""]",cardiac_neuro_cava_random,chr1,2192936,2193206,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73841,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGACTCTGGGCTGCTCAGAGGCTGCCTTG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,None,None,cardiac_neuro_cava_random,chrX,154376832,154377102,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,-
73842,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGAGCCCTGGGGAACGCCATGAGCCCTCAGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,None,None,cardiac_neuro_cava_random,chrX,154383755,154384025,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,-
73843,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGGACTTAAACCCCAGCCTCCCCCGTCCA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,None,None,cardiac_neuro_cava_random,chrX,154402403,154402673,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,-
73844,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGCCCATAATTTATTGATTTTTTAAAATTTG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,None,None,cardiac_neuro_cava_random,chrX,154409693,154409963,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,-


In [258]:
file_path = 'metadata_tested_region_2307.tsv'
# tested_meta_data_file_region[interesting_columns].to_csv(file_path, sep="\t", index=False, na_rep='NA')

In [259]:
tested_meta_data_file_region[col_allele]

0        ["alt"]
1        ["alt"]
2        ["alt"]
3        ["alt"]
4        ["alt"]
          ...   
73841        NaN
73842        NaN
73843        NaN
73844        NaN
73845        NaN
Name: allele, Length: 73846, dtype: object

In [260]:
import ast

# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

# list columns col_variant_class, col_variant_pos, col_SPDI, col_allele,
list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]
# Apply the safe_eval function to the specified columns
for col in list_columns:
    tested_meta_data_file_region[col] = tested_meta_data_file_region[col].apply(safe_eval)

In [261]:
tested_meta_data_file_region[interesting_columns].loc[tested_meta_data_file_region[col_allele].apply(hf.is_reference)].head()
tested_meta_data_file_region[interesting_columns].loc[tested_meta_data_file_region[col_name]=='cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1']

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
64946,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,chr1,2181843,2182113,+,None,None,None,None,cardiac_neuro_cava_random


In [262]:
# from pprint import pprint
# from frictionless import extract

# rows = extract('metadata_tested_region_2307.tsv')
# pprint(rows)


In [18]:
from frictionless import validate
schema_path = config['files']['creating']['metadata_file_schema']
schema_path = '../../../../80K_MPRA/igvf_checkfiles_metadata/src/schemas/table_schemas/designed_sequences.json' # file path needs to be relative
schema_path = 'designed_sequences.json' # file path needs to be relative
schema_path = 'mpra_sequence_designs.json'
file_path = 'metadata_tested_region_2307.tsv' # config['files']['creating']['region_metadata_table_local_tested_juli'] # file path needs to be relative
file_path = 'metadata_tested_region_2208.tsv'
file_path = 'metadata_tested_region_2010_validate.tsv'
file_path = 'metadata_tested_region_2010_json.tsv'
file_path = 'metadata_tested_region_2010.tsv'
file_path = 'metadata_tested_region_2110_doubleQuotes.tsv'
file_path = 'metadata_tested_region_2110_doubleQuotes_test.tsv'
file_path = 'metadata_tested_region.tsv'
file_path = 'MK.metadata.tsv'
file_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/MPRA_80215.metadata.tsv.gz'
file_path = "control_metadata/cardiac_neuro_cava_random.metadata.tsv"
file_path = "control_metadata/MPRA_80215.metadata.test.tsv"
file_path = "control_metadata/cardiac_neuro_cava_random.metadata.tsv.gz"
file_path = "control_metadata/MPRA_80215.metadata.tsv.gz"

# file_path = os.path.relpath(config['files']['creating']['region_metadata_table_local_tested_juli'])
report = validate(file_path, schema = schema_path)
report
# /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks
#
# /home/kisa/coding/80K_MPRA/metadata_info/metadata_tested_region_2307.tsv
#

{'valid': True,
 'stats': {'tasks': 1, 'errors': 0, 'warnings': 0, 'seconds': 2.787},
 'warnings': [],
 'errors': [],
 'tasks': [{'name': 'mpra_80215.metadata',
            'type': 'table',
            'valid': True,
            'place': 'control_metadata/MPRA_80215.metadata.tsv.gz',
            'labels': ['name',
                       'sequence',
                       'category',
                       'class',
                       'source',
                       'ref',
                       'chr',
                       'start',
                       'end',
                       'strand',
                       'variant_class',
                       'variant_pos',
                       'SPDI',
                       'allele',
                       'info'],
            'stats': {'errors': 0,
                      'warnings': 0,
                      'seconds': 2.787,
                      'md5': '82a5f222ccb154644caa9c61c3646062',
                      'sha256': '26e5f67a68

In [5]:
file_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/MPRA_80215.metadata.tsv.gz'
file_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/MPRA_80215.metadata.tsv.test.gz'
metadata_file = pd.read_csv(file_path,sep="\t")

metadata_file


# write_data_with_json()

FileNotFoundError: [Errno 2] No such file or directory: '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/MPRA_80215.metadata.tsv.test.gz'

In [71]:
# read json file
metadata = pd.read_csv(file_path, sep="\t", low_memory=False)

/tmp/ipykernel_266775/3044895676.py:2: DtypeWarning: Columns (10,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(file_path, sep="\t")


In [ ]:
metadata

In [264]:
# matchable_element_tested_meta_data_file_region.columns

# tested_meta_data_file_region = pd.concat([variant_tested_meta_data_file, matchable_element_tested_meta_data_file_region], ignore_index=True)

# # write the metadata table to file
# tested_meta_data_file_region.to_csv(config['files']['creating']['region_metadata_table_local_tested_juli'], sep="\t", index=False)
# tested_meta_data_file_region.to_csv(config['files']['creating']['datafreeze_table'], sep="\t", index=False, na_rep='NA')


# meta_data_file_juli = pd.read_csv(config['files']['creating']['region_metadata_table_local_tested_juli'], sep="\t")
# meta_data_file_april = pd.read_csv(config['files']['creating']['metadata_table'], sep="\t")
# meta_data_file_april.shape[0]
# meta_data_file_juli.shape[0]

In [265]:
throw end of validation

SyntaxError: invalid syntax (3734694893.py, line 1)

#### Add have_readout_ columns
- add have_readout_MPRAsnakeflow_bc_10: add if we have readout from MPRAsnakeflow with barcode threshold 10
- add DNA count 1, 2, 3 and RNA count 1, 2, 3 and n_barcodes 1, 2, 3
- add `have_readout_BE_CALM_variants`: add if we have readout from BE-CALM for variants: add `variant_effect_logfc` (logfc) and `variant_adjusted_pvalue` adjusted-pvalue 
  - from: 
- add `have_readout_BE_CALM_elements`: add if we have readout from BE-CALM for elements 
- alternative_df + ref_id column
- ref_df 
- join ref_df with alternative_df on ref_id + 
- return merged_df.groupby('REF')[column_name].apply(list).reset_index(name=column_name)
- .loc[df.ref_id == "ref_id", 'column_variant_effect']


In [266]:
tested_meta_data_file_region.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand'],
      dtype='object')

Preprocess the variant bcalm file

In [267]:
# 1. find the data from BE-CALM variants default: files, creating, toptable_bcMPRAlm_final_resequencing_umi
variant_becalm_raw = pd.read_csv(config['files']['creating']['toptable_bcMPRAlm_final_resequencing_umi'], sep="\t")
print(variant_becalm_raw.shape[0]) # 29781
variant_becalm_raw.head()
variant_becalm_raw.columns = ['variant_BECALM_logFC', 'variant_BECALM_AveExpr', 'variant_BECALM_t', 'variant_BECALM_P.Value', 'variant_BECALM_adj.P.Val', 'variant_BECALM_B', 'variant_id']
# important columns:
becalm_variant_important_columns = ['variant_id', 'variant_BECALM_logFC', 'variant_BECALM_adj.P.Val']
variant_becalm = variant_becalm_raw[becalm_variant_important_columns]
variant_becalm.head()

29781


,variant_id,variant_BECALM_logFC,variant_BECALM_adj.P.Val
0,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,1.319700,6.392866e-112
1,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,1.200951,3.117373e-66
2,cardiac_neuro_cava_random:DISC1|ENSG0000016294...,-0.981155,1.483143e-55
3,cardiac_neuro_cava_random:ANKZF1|ENSG000001635...,1.241950,4.272807e-53
4,cardiac_neuro_cava_random:SLC1A2|ENSG000001104...,0.884075,8.103250e-46


Preprocess the element bcalm file

In [268]:
from collections import defaultdict


def get_gene_lookup_dict(gene_lists):
    """
    prepare a dict which has gene_name: (list of associated gene_sets)
    :gene_lists - list of files containing gene names and have useful names
    """
    gene_lookup_dict = defaultdict(list)
    for file in gene_lists:
        with open(os.path.join(gene_list_dir, file), 'r') as f:
            gene_list = f.read().splitlines()
            gene_list_name = file.split('.')[0]
            for gene in gene_list:
                gene_lookup_dict[gene].append(gene_list_name)
    return gene_lookup_dict


def get_gene_name_from_element(header):
    """Returns the gene name: cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778533_fwd_tile1-1 => SKI"""
    if 'ALT_' in header:
        print(header)
        raise ValueError('This function is only for element sequences')

    if 'REF_' in header:
        return header.split(":REF_")[1].split('|')[0]

    return header.split(':')[1].split('|')[0]


def get_info_for_ref(metadata_file, references_of_interest, columns_of_interest, variant_region_map, additional_info_df=None, additional_matching_column=None):
    """
    Returns the values of columns of interest for the given reference sequence from their associated alternative sequences
    @params: additional_info_df: df with additional_matching_column + columns of interest values
    """
    # 1. check for tested sequences
    metadata_file_tested = metadata_file.loc[metadata_file[col_name].str.startswith('cardiac_neuro_cava_random')]
    # 2. split in variant and not variant table
    variant_tested_meta_data_file_region = metadata_file_tested.loc[metadata_file_tested[col_category] == 'variant']
    variant_tested_meta_data_file_region # 64946
    # 2a. split in reference and alternative
    ref_variant_tested_meta_data_file_region = variant_tested_meta_data_file_region.loc[variant_tested_meta_data_file_region[col_allele].apply(hf.is_reference)]
    alt_variant_tested_meta_data_file_region = variant_tested_meta_data_file_region.loc[variant_tested_meta_data_file_region[col_allele].apply(hf.is_alternative)]

    # change column names of variant region map
    variant_region_map.columns = ['tmp_variant_map_Variant_id', 'tmp_variant_map_Region', 'tmp_variant_map_REF_ID', 'tmp_variant_map_ALT_ID']
    # 2b. add variant id information using REF_ID and ALT_ID from variant_region_map
    alt_variant_id_tested_meta_data_file_region = alt_variant_tested_meta_data_file_region.merge(variant_region_map[['tmp_variant_map_Variant_id', 'tmp_variant_map_ALT_ID', 'tmp_variant_map_REF_ID']], left_on=col_name, right_on='tmp_variant_map_ALT_ID', how='left')
    if additional_matching_column != None:
        alt_variant_id_tested_meta_data_file_region = alt_variant_id_tested_meta_data_file_region.merge(additional_info_df, left_on='tmp_variant_map_Variant_id', right_on=additional_matching_column, how='left')

    # left join on name and REF_ID
    ref_alt_tested_metadata_merged = ref_variant_tested_meta_data_file_region.merge(alt_variant_id_tested_meta_data_file_region[['tmp_variant_map_ALT_ID']+columns_of_interest], left_on=col_name, right_on='tmp_variant_map_REF_ID', how='left')
    # get results df
    reference_result = ref_alt_tested_metadata_merged.loc[ref_alt_tested_metadata_merged['tmp_variant_map_REF_ID'].isin(references_of_interest), columns_of_interest]
    return reference_result


# get gene_list files:
gene_list_dir = '/home/kisa/coding/80K_MPRA/MPRA_design/resources/gene_lists'
gene_lists = os.listdir(gene_list_dir)

# prepare a dict which has gene_name: (tuple of associated gene_sets)
gene_lookup_dict = get_gene_lookup_dict(gene_lists=gene_lists)

In [269]:
element_analysis_r_result.columns

Index(['logFC', 'AveExpr', 't', 'P.Value', 'adj.P.Val', 'variant_id',
       'gene_name', 'gene_set', 'is_neuro', 'is_cardiac', 'is_cava',
       'is_random', 'is_significant'],
      dtype='object')

In [270]:
# negative neuron np vs tested (left sided)
element_analysis_r_result = pd.read_csv(config['files']['creating']['element_analysis_np_negative_neuron_vs_tested'], sep="\t")
element_analysis_r_result
# # get gene name from variant id
element_analysis_r_result['gene_name'] = element_analysis_r_result['variant_id'].apply(get_gene_name_from_element)
# add gene_set annotation
element_analysis_r_result['gene_set'] = element_analysis_r_result['gene_name'].apply(lambda gene: gene_lookup_dict[gene])

# add gene info boolean columns
element_analysis_r_result['is_neuro'] = element_analysis_r_result['gene_set'].apply(lambda x: 'neuro' in x)
element_analysis_r_result['is_cardiac'] = element_analysis_r_result['gene_set'].apply(lambda x: 'cardiac' in x)
element_analysis_r_result['is_cava'] = element_analysis_r_result['gene_set'].apply(lambda x: 'cava' in x)
element_analysis_r_result['is_random'] = element_analysis_r_result['gene_set'].apply(lambda x: 'random' in x)

# add significant column
element_analysis_r_result['is_significant'] = element_analysis_r_result['adj.P.Val'] < 0.05

# rename columns: logFC: element_BCALM_logFC, adj.P.Val: element_BCALM_adj.P.Val
element_analysis_r_result_renamed = element_analysis_r_result.rename(columns={'logFC': 'element_BCALM_logFC', 'adj.P.Val': 'element_BCALM_adj.P.Val'})

# columns of interest
columns_of_interest = ['variant_id', 'element_BCALM_logFC', 'element_BCALM_adj.P.Val']
element_analysis_r_result_renamed = element_analysis_r_result_renamed[columns_of_interest]

In [282]:
tested_meta_data_file_region_bcalm.columns


Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC',
       'variant_id', 'element_BCALM_logFC', 'element_BCALM_adj.P.Val'],
      dtype='object')

Add the variant_map information for joining the tables 
- variant bcalm: variant_id => alt_id
- element bcalm: variant_id => ref_id

In [271]:
all_ref_of_interest = tested_meta_data_file_region.loc[tested_meta_data_file_region[col_allele].apply(hf.is_reference)][col_name].to_list()
variant_bcalm_all_references = get_info_for_ref(metadata_file=tested_meta_data_file_region, additional_info_df=variant_becalm, additional_matching_column='variant_id', references_of_interest=all_ref_of_interest, columns_of_interest=['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC'], variant_region_map=variant_region_map)
# group by ref_id and make variant_BECALM and variant_BECALM_logFC to list
list_of_listing_columns = ['variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']
list_of_listing_cols_dict = {col: list for col in list_of_listing_columns}
variant_bcalm_all_references_renamed = variant_bcalm_all_references.groupby('tmp_variant_map_REF_ID').agg(list_of_listing_cols_dict).reset_index() # 18572

# rename tmp_variant_map_REF_ID to name
variant_bcalm_all_references_renamed = variant_bcalm_all_references_renamed.rename(columns={'tmp_variant_map_REF_ID': 'name'}) # 18572
variant_bcalm_all_references_renamed['name'].nunique() # 18572

# prepare the data for the alternative sequences: goal is a dataframe of the alternative sequences: name, variant_BECALM_adj.P.Val, variant_BECALM_logFC
# merge tested_meta_data_file_region with variant_region_map => to get the alt_id
alt_ids_with_variant_bcalm = tested_meta_data_file_region.loc[tested_meta_data_file_region[col_allele].apply(hf.is_alternative)].merge(variant_region_map[['tmp_variant_map_ALT_ID', 'tmp_variant_map_Variant_id']], left_on=col_name, right_on='tmp_variant_map_ALT_ID', how='left')
alt_ids_with_variant_bcalm # 46374
alt_ids_with_variant_bcalm['tmp_variant_map_Variant_id'].isna().sum() # 0

# merge with variant_becalm
alt_ids_with_variant_bcalm = alt_ids_with_variant_bcalm.merge(variant_becalm, left_on='tmp_variant_map_Variant_id', right_on='variant_id', how='left')
alt_ids_with_variant_bcalm['variant_id'].isna().sum() # 17091 => 29283 not na
alt_ids_with_variant_bcalm.columns

# subset of the columns
all_alt_variant_becalm = alt_ids_with_variant_bcalm[['name', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']]
all_alt_variant_becalm # 46374
all_alt_variant_becalm['name'].nunique() # 46374
# concat ref and alt dataframes (same column names )
tested_metadata_bcalm = pd.concat([variant_bcalm_all_references_renamed, all_alt_variant_becalm], ignore_index=True)
tested_metadata_bcalm # 64946 = 18572 + 46374

# numbers without na values
tested_metadata_bcalm['variant_BECALM_adj.P.Val'].isna().sum() # 17091

17091

In [272]:
tested_meta_data_file_region_bcalm = tested_meta_data_file_region.merge(tested_metadata_bcalm, on='name', how='left')
# print results:
print(f"Number of rows in the metadata file: {tested_meta_data_file_region_bcalm.shape[0]}")
print(f"Number of variants within the metadata file: {tested_meta_data_file_region_bcalm.loc[tested_meta_data_file_region_bcalm[col_allele].apply(hf.is_alternative)].shape[0]}") # 46374
print(f"Number of rows in the metadata file with variant BECALM results: {tested_meta_data_file_region_bcalm.loc[tested_meta_data_file_region_bcalm[col_allele].apply(hf.is_alternative)]['variant_BECALM_adj.P.Val'].notna().sum()}") # 29283

Number of rows in the metadata file: 73846
Number of variants within the metadata file: 46374
Number of rows in the metadata file with variant BECALM results: 29283


In [280]:
tested_meta_data_file_region_bcalm.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC',
       'variant_id', 'element_BCALM_logFC', 'element_BCALM_adj.P.Val'],
      dtype='object')

In [273]:
# add element information to the dataframe:

In [274]:
element_analysis_r_result_renamed

,variant_id,element_BCALM_logFC,element_BCALM_adj.P.Val
0,cardiac_neuro_cava_random:RBM20|ENSG0000020386...,1.651677,0.0
1,cardiac_neuro_cava_random:REF_PSMD12|ENSG00000...,2.072247,0.0
2,cardiac_neuro_cava_random:REF_WWOX|ENSG0000018...,1.852506,0.0
3,cardiac_neuro_cava_random:RERE|ENSG00000142599...,1.765497,0.0
4,cardiac_neuro_cava_random:TCF4|ENSG00000196628...,1.827335,0.0
...,...,...,...
22908,cardiac_neuro_cava_random:REF_GNAI1|ENSG000001...,-0.969547,1.0
22909,cardiac_neuro_cava_random:REF_CREBBP|ENSG00000...,-1.001650,1.0
22910,cardiac_neuro_cava_random:NCOA1|ENSG0000008467...,-1.010178,1.0
22911,cardiac_neuro_cava_random:NLGN4X|ENSG000001469...,-0.983895,1.0


In [275]:
tested_meta_data_file_region_bcalm = tested_meta_data_file_region_bcalm.merge(element_analysis_r_result_renamed, left_on=col_name, right_on='variant_id', how='left')
# tested_meta_data_file_region_bcalm # 73846

In [276]:
tested_meta_data_file_region_bcalm['element_BCALM_logFC'].isna().sum() # 50937

50937

In [277]:
# print the results
print(f"Number of rows in the metadata file: {tested_meta_data_file_region_bcalm.shape[0]}") # 73846
print(f"Number of elements and references within the metadata file: {tested_meta_data_file_region_bcalm.loc[~tested_meta_data_file_region_bcalm[col_allele].apply(hf.is_alternative)].shape[0]}") # 27472
print(f"Number of elements and references within the metadata file with readout from bcalm: {tested_meta_data_file_region_bcalm.loc[~tested_meta_data_file_region_bcalm[col_allele].apply(hf.is_alternative)]['element_BCALM_logFC'].notna().sum()}") # 22909
# tested_meta_data_file_region_bcalm.to_csv('/home/kisa/coding/80K_MPRA/metadata_info/metadata_file_with_bcalm_results.tsv', sep="\t", index=False)

Number of rows in the metadata file: 73846
Number of elements and references within the metadata file: 27472
Number of elements and references within the metadata file with readout from bcalm: 22909


In [314]:
print('summary of metadata file with bcalm info for elements and variants:\n', tested_meta_data_file_region_bcalm.columns)
tested_meta_data_file_region_bcalm.head()

summary of metadata file with bcalm info for elements and variants:
 Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC',
       'variant_id', 'element_BCALM_logFC', 'element_BCALM_adj.P.Val'],
      dtype='object')


,header,sequence,tmp_label,name,category,class,source,ref,variant_class,variant_pos,...,chr,start,end,region_name,strand,variant_BECALM_adj.P.Val,variant_BECALM_logFC,variant_id,element_BCALM_logFC,element_BCALM_adj.P.Val
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[83],...,chr1,2179507,2179777,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+,NaN,NaN,NaN,NaN,NaN
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[181],...,chr1,2191262,2191532,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+,NaN,NaN,NaN,NaN,NaN
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCTGGGTGACCCGGAGAACACCAAGGCTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[43],...,chr1,2191971,2192241,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+,NaN,NaN,NaN,NaN,NaN
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCATGCGGTGGCCACAGCCTCGGGTGAGTTC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[116],...,chr1,2192249,2192519,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+,0.946869,-0.045242,NaN,NaN,NaN
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGGACTCCGGTGCCTTCGCATTCCCGAGCTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[205],...,chr1,2192936,2193206,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+,0.816996,-0.077985,NaN,NaN,NaN


In [294]:
variant_region_map

,tmp_variant_map_Variant_id,tmp_variant_map_Region,tmp_variant_map_REF_ID,tmp_variant_map_ALT_ID
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
...,...,...,...,...
47039,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618
47040,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757
47041,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373
47042,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656


In [297]:
# rename variant region map columns Variant => tmp_variant_map_id, REF_ID => tmp_variant_map_ref_id, ALT_ID => tmp_variant_map_alt_id
# variant_map_columns = ['Variant', 'REF_ID', 'ALT_ID']
# tested_variant_map = variant_region_map.rename(columns={variant_map_columns[0]: 'tmp_variant_map_Variant_id', variant_map_columns[1]: 'tmp_variant_map_REF_ID', variant_map_columns[2]: 'tmp_variant_map_ALT_ID'})
tested_variant_map = tested_variant_map.loc[tested_variant_map['tmp_variant_map_Variant_id'].str.startswith('cardiac_neuro_cava_random')]
tested_variant_map['tmp_variant_map_ALT_ID'].nunique() # 46374

# split the tested_meta_data_file_region into variant and element dataframes
all_alt_tested_meta_data_file_region = tested_meta_data_file_region.loc[tested_meta_data_file_region[col_allele].apply(hf.is_alternative)]
all_alt_tested_meta_data_file_region.shape[0] # 46374


46374

In [298]:
# use for variant df the ALT_ID,
all_alt_tested_meta_data_file_region_variant_map = all_alt_tested_meta_data_file_region.merge(tested_variant_map, left_on=col_name, right_on='tmp_variant_map_ALT_ID', how='left')
all_alt_tested_meta_data_file_region_variant_map

# add the variant becalm info
all_alt_tested_meta_data_file_region_variant_map_bcalm = all_alt_tested_meta_data_file_region_variant_map.merge(variant_becalm, left_on='tmp_variant_map_Variant_id', right_on='variant_id', how='left')
all_alt_tested_meta_data_file_region_variant_map_bcalm['variant_BECALM_adj.P.Val'].isna().sum() # 17091 without readout from bcalm
# concatenate in the end

17091

In [299]:
# 2. split in variant and not variant table
variant_tested_meta_data_file_region = tested_meta_data_file_region.loc[tested_meta_data_file_region[col_category] == 'variant']
variant_tested_meta_data_file_region # 64946
# 2a. split in reference and alternative
ref_variant_tested_meta_data_file_region = variant_tested_meta_data_file_region.loc[variant_tested_meta_data_file_region[col_allele].apply(hf.is_reference)]
alt_variant_tested_meta_data_file_region = variant_tested_meta_data_file_region.loc[variant_tested_meta_data_file_region[col_allele].apply(hf.is_alternative)]
print(alt_variant_tested_meta_data_file_region.shape[0]) # 46374
# change column names of variant region map
variant_region_map.columns = ['tmp_variant_map_Variant_id', 'tmp_variant_map_Region', 'tmp_variant_map_REF_ID', 'tmp_variant_map_ALT_ID']
# remove the unnecessary duplicates
reference_variant_region_map = variant_region_map[['tmp_variant_map_REF_ID', 'tmp_variant_map_Region']].drop_duplicates()
# 2b. add variant id information using REF_ID and ALT_ID from variant_region_map
alt_variant_id_tested_meta_data_file_region = alt_variant_tested_meta_data_file_region.merge(variant_region_map[['tmp_variant_map_Variant_id', 'tmp_variant_map_ALT_ID', 'tmp_variant_map_REF_ID']], left_on=col_name, right_on='tmp_variant_map_ALT_ID', how='left')
alt_variant_id_tested_meta_data_file_region #46374
alt_variant_id_tested_meta_data_file_region['tmp_variant_map_Variant_id'].isna().sum() # 0

# 2c. add results from BE-CALM
alt_variants_tested_be_calm_results = alt_variant_id_tested_meta_data_file_region.merge(variant_becalm, left_on='tmp_variant_map_Variant_id', right_on='variant_id', how='left')
alt_variants_tested_be_calm_results # 46374
alt_variants_tested_be_calm_results['variant_BECALM_adj.P.Val'].isna().sum() # 17091 => 29283 not na
# 2n. combine variant and not variant table


46374


17091

In [300]:
alt_variants_tested_be_calm_results.head()['tmp_variant_map_REF_ID'].to_list()

['cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778490_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778492_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E1311587_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778494_fwd_tile1-1']

In [301]:
# find ref with length of SPDI column >= 2
ref_variant_tested_meta_data_file_region.loc[ref_variant_tested_meta_data_file_region[col_SPDI].apply(lambda x: len(x) >= 2)][col_name].to_list()

['cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778506_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778530_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778534_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778535_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778544_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778574_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778579_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778585_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778586_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778588_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778590_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778592_fwd_tile1-1',
 'cardiac_neuro_

In [305]:
alt_variants_tested_be_calm_results.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_variant_map_Variant_id', 'tmp_variant_map_ALT_ID',
       'tmp_variant_map_REF_ID', 'variant_id', 'variant_BECALM_logFC',
       'variant_BECALM_adj.P.Val'],
      dtype='object')

In [306]:
ref_variant_tested_meta_data_file_region.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand'],
      dtype='object')

In [307]:
# now get the values for the reference sequences
# examples with results from variant BE-CALM: 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E1311587_fwd_tile1-1', 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778494_fwd_tile1-1'
# example with multiple alternatives: cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778506_fwd_tile1-1
columns_of_interest_variant_info = ['tmp_variant_map_Variant_id', 'tmp_variant_map_ALT_ID', 'tmp_variant_map_REF_ID', 'variant_BECALM_logFC', 'variant_BECALM_adj.P.Val']
# left join on name and REF_ID
ref_alt_tested_metadata_merged = ref_variant_tested_meta_data_file_region.merge(
    alt_variants_tested_be_calm_results[columns_of_interest_variant_info],
    left_on=col_name,
    right_on='tmp_variant_map_REF_ID', how='left')

ref_alt_tested_metadata_merged.shape[0] # 46374 increased
# A: groupby ref id and apply(list) if one column
ref_alt_tested_metadata_merged.groupby('tmp_variant_map_REF_ID')['variant_BECALM_adj.P.Val'].apply(list).to_dict()
# # B: .loc[(condition), [columns]]
# ref_alt_tested_metadata_merged.loc[ref_alt_tested_metadata_merged['tmp_variant_map_REF_ID'] == 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E1311587_fwd_tile1-1'][['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']]
# ref_alt_tested_metadata_merged.loc[ref_alt_tested_metadata_merged['tmp_variant_map_REF_ID'] == 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778506_fwd_tile1-1'][['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']]
# ref_alt_tested_metadata_merged.loc[ref_alt_tested_metadata_merged['tmp_variant_map_REF_ID'].isin(['cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E1311587_fwd_tile1-1', 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778506_fwd_tile1-1'])][['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']]
# ref_alt_tested_metadata_merged.loc[ref_alt_tested_metadata_merged['tmp_variant_map_REF_ID'].isin(['cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E1311587_fwd_tile1-1', 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778506_fwd_tile1-1']), ['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']]
# ref_alt_tested_metadata_merged.loc[ref_alt_tested_metadata_merged['tmp_variant_map_REF_ID'].isin(['cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778506_fwd_tile1-1'])][['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']]
# ref_alt_tested_metadata_merged.loc[ref_alt_tested_metadata_merged['tmp_variant_map_REF_ID'].isin(['cardiac_neuro_cava_random:REF_AARS1|ENSG00000090861.17|EH38E3188759_rev_tile1-1'])][['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']]

{'cardiac_neuro_cava_random:REF_AARS1|ENSG00000090861.17|EH38E1825123_rev_tile1-1': [0.859317452516081],
 'cardiac_neuro_cava_random:REF_AARS1|ENSG00000090861.17|EH38E3188744_rev_tile1-1': [0.534125925260689],
 'cardiac_neuro_cava_random:REF_AARS1|ENSG00000090861.17|EH38E3188759_rev_tile1-1': [0.70510375248836,
  0.982066266302263,
  0.92967765622702],
 'cardiac_neuro_cava_random:REF_AARS1|ENSG00000090861.17|EH38E3188760_rev_tile1-1': [0.76368093010349,
  0.814549845836823],
 'cardiac_neuro_cava_random:REF_AARS1|ENSG00000090861.17|EH38E3188763_rev_tile1-1': [0.879813515178025],
 'cardiac_neuro_cava_random:REF_AARS1|ENSG00000090861.17|EH38E3188770_rev_tile1-1': [0.966985456255873],
 'cardiac_neuro_cava_random:REF_AARS1|ENSG00000090861.17|EH38E3188772_rev_tile1-1': [0.837600551452658],
 'cardiac_neuro_cava_random:REF_AARS1|ENSG00000090861.17|EH38E3188775_rev_tile1-1': [0.942992257052366],
 'cardiac_neuro_cava_random:REF_ABCC9|ENSG00000069431.14|EH38E1598027_rev_tile1-1': [0.9280741183479

In [308]:
ref_alt_tested_metadata_merged.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_variant_map_Variant_id', 'tmp_variant_map_ALT_ID',
       'tmp_variant_map_REF_ID', 'variant_BECALM_logFC',
       'variant_BECALM_adj.P.Val'],
      dtype='object')

In [ ]:
throw end of adding variant and element information to the metadata table

##### Fast code snippet to get the values of the associated alternative sequences for a set of reference sequences

In [93]:
def get_info_for_ref(metadata_file, references_of_interest, columns_of_interest, variant_region_map, additional_info_df=None, additional_matching_column=None):
    """
    Returns the values of columns of interest for the given reference sequence from their associated alternative sequences
    @params: additional_info_df: df with additional_matching_column + columns of interest values
    """
    # 1. check for tested sequences
    metadata_file_tested = metadata_file.loc[metadata_file[col_name].str.startswith('cardiac_neuro_cava_random')]
    # 2. split in variant and not variant table
    variant_tested_meta_data_file_region = metadata_file_tested.loc[metadata_file_tested[col_category] == 'variant']
    variant_tested_meta_data_file_region # 64946
    # 2a. split in reference and alternative
    ref_variant_tested_meta_data_file_region = variant_tested_meta_data_file_region.loc[variant_tested_meta_data_file_region[col_allele].apply(hf.is_reference)]
    alt_variant_tested_meta_data_file_region = variant_tested_meta_data_file_region.loc[variant_tested_meta_data_file_region[col_allele].apply(hf.is_alternative)]

    # change column names of variant region map
    variant_region_map.columns = ['tmp_variant_map_Variant_id', 'tmp_variant_map_Region', 'tmp_variant_map_REF_ID', 'tmp_variant_map_ALT_ID']
    # 2b. add variant id information using REF_ID and ALT_ID from variant_region_map
    alt_variant_id_tested_meta_data_file_region = alt_variant_tested_meta_data_file_region.merge(variant_region_map[['tmp_variant_map_Variant_id', 'tmp_variant_map_ALT_ID', 'tmp_variant_map_REF_ID']], left_on=col_name, right_on='tmp_variant_map_ALT_ID', how='left')
    if additional_matching_column != None:
        alt_variant_id_tested_meta_data_file_region = alt_variant_id_tested_meta_data_file_region.merge(additional_info_df, left_on='tmp_variant_map_Variant_id', right_on=additional_matching_column, how='left')

    # left join on name and REF_ID
    ref_alt_tested_metadata_merged = ref_variant_tested_meta_data_file_region.merge(alt_variant_id_tested_meta_data_file_region[['tmp_variant_map_ALT_ID']+columns_of_interest], left_on=col_name, right_on='tmp_variant_map_REF_ID', how='left')
    # get results df
    reference_result = ref_alt_tested_metadata_merged.loc[ref_alt_tested_metadata_merged['tmp_variant_map_REF_ID'].isin(references_of_interest), columns_of_interest]
    return reference_result


In [94]:
variant_becalm_raw = pd.read_csv(config['files']['creating']['toptable_bcMPRAlm_final_resequencing_umi'], sep="\t")
variant_becalm_raw.columns = ['variant_BECALM_logFC', 'variant_BECALM_AveExpr', 'variant_BECALM_t', 'variant_BECALM_P.Value', 'variant_BECALM_adj.P.Val', 'variant_BECALM_B', 'variant_id']

# important columns:
becalm_variant_important_columns = ['variant_id', 'variant_BECALM_logFC', 'variant_BECALM_adj.P.Val']
variant_becalm = variant_becalm_raw[becalm_variant_important_columns]

In [95]:
tested_meta_data_file_region_references = tested_meta_data_file_region.loc[tested_meta_data_file_region[col_allele].apply(hf.is_reference)]
tested_meta_data_file_region_references.loc[tested_meta_data_file_region_references[col_allele].isna()]

,header,sequence,tmp_label,name,category,class,source,ref,variant_class,variant_pos,SPDI,allele,info,tmp_matching_header,chr,start,end,region_name,strand


In [96]:
variant_becalm

,variant_id,variant_BECALM_logFC,variant_BECALM_adj.P.Val
0,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,1.319700,6.392866e-112
1,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,1.200951,3.117373e-66
2,cardiac_neuro_cava_random:DISC1|ENSG0000016294...,-0.981155,1.483143e-55
3,cardiac_neuro_cava_random:ANKZF1|ENSG000001635...,1.241950,4.272807e-53
4,cardiac_neuro_cava_random:SLC1A2|ENSG000001104...,0.884075,8.103250e-46
...,...,...,...
29776,cardiac_neuro_cava_random:TCF4|ENSG00000196628...,-0.002468,9.926864e-01
29777,cardiac_neuro_cava_random:DDX3X|ENSG0000021530...,0.003859,9.883899e-01
29778,cardiac_neuro_cava_random:GRIN2A|ENSG000001834...,0.001549,9.965500e-01
29779,cardiac_neuro_cava_random:DDX3X|ENSG0000021530...,-0.001804,9.958936e-01


In [107]:
['tmp_variant_map_ALT_ID']+['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']

['tmp_variant_map_ALT_ID',
 'tmp_variant_map_REF_ID',
 'variant_BECALM_adj.P.Val',
 'variant_BECALM_logFC']

In [104]:
ref_of_interest=['cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778506_fwd_tile1-1']
# ref_of_interest=['cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E1311587_fwd_tile1-1']
# ref_of_interest=['cardiac_neuro_cava_random:REF_AARS1|ENSG00000090861.17|EH38E3188759_rev_tile1-1']
get_info_for_ref(metadata_file=tested_meta_data_file_region, additional_info_df=variant_becalm, additional_matching_column='variant_id', references_of_interest=ref_of_interest, columns_of_interest=['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC'], variant_region_map=variant_region_map)

,tmp_variant_map_REF_ID,variant_BECALM_adj.P.Val,variant_BECALM_logFC
6,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.856513,-0.099725
7,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.915830,-0.061454
8,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.896808,-0.070486
9,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.912745,-0.060166


make dataframe with list of the results from the alternative sequences

In [309]:
all_ref_of_interest = tested_meta_data_file_region.loc[tested_meta_data_file_region[col_allele].apply(hf.is_reference)][col_name].to_list()
variant_bcalm_all_references = get_info_for_ref(metadata_file=tested_meta_data_file_region, additional_info_df=variant_becalm, additional_matching_column='variant_id', references_of_interest=all_ref_of_interest, columns_of_interest=['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC'], variant_region_map=variant_region_map)
# group by ref_id and make variant_BECALM and variant_BECALM_logFC to list
list_of_listing_columns = ['variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']
list_of_listing_cols_dict = {col: list for col in list_of_listing_columns}
variant_bcalm_all_references_renamed = variant_bcalm_all_references.groupby('tmp_variant_map_REF_ID').agg(list_of_listing_cols_dict).reset_index() # 18572

# rename tmp_variant_map_REF_ID to name
variant_bcalm_all_references_renamed = variant_bcalm_all_references_renamed.rename(columns={'tmp_variant_map_REF_ID': 'name'}) # 18572
variant_bcalm_all_references_renamed['name'].nunique() # 18572

# prepare the data for the alternative sequences: goal is a dataframe of the alternative sequences: name, variant_BECALM_adj.P.Val, variant_BECALM_logFC
# merge tested_meta_data_file_region with variant_region_map => to get the alt_id
alt_ids_with_variant_bcalm = tested_meta_data_file_region.loc[tested_meta_data_file_region[col_allele].apply(hf.is_alternative)].merge(variant_region_map[['tmp_variant_map_ALT_ID', 'tmp_variant_map_Variant_id']], left_on=col_name, right_on='tmp_variant_map_ALT_ID', how='left')
alt_ids_with_variant_bcalm # 46374
alt_ids_with_variant_bcalm['tmp_variant_map_Variant_id'].isna().sum() # 0

# merge with variant_becalm
alt_ids_with_variant_bcalm = alt_ids_with_variant_bcalm.merge(variant_becalm, left_on='tmp_variant_map_Variant_id', right_on='variant_id', how='left')
alt_ids_with_variant_bcalm['variant_id'].isna().sum() # 17091 => 29283 not na
alt_ids_with_variant_bcalm.columns

# subset of the columns
all_alt_variant_becalm = alt_ids_with_variant_bcalm[['name', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']]
all_alt_variant_becalm # 46374
all_alt_variant_becalm['name'].nunique() # 46374
# concat ref and alt dataframes (same column names )
tested_metadata_bcalm = pd.concat([variant_bcalm_all_references_renamed, all_alt_variant_becalm], ignore_index=True)
tested_metadata_bcalm # 64946 = 18572 + 46374

# numbers without na values
tested_metadata_bcalm['variant_BECALM_adj.P.Val'].isna().sum() # 17091

17091

add to the metadata file: tested_meta_data_file_region

In [165]:
tested_meta_data_file_region_bcalm = tested_meta_data_file_region.merge(tested_metadata_bcalm, on='name', how='left')
# print results:
print(f"Number of rows in the metadata file: {tested_meta_data_file_region_bcalm.shape[0]}")
print(f"Number of variants within the metadata file: {tested_meta_data_file_region_bcalm.loc[tested_meta_data_file_region_bcalm[col_allele].apply(hf.is_alternative)].shape[0]}") # 46374
print(f"Number of rows in the metadata file with variant BECALM results: {tested_meta_data_file_region_bcalm.loc[tested_meta_data_file_region_bcalm[col_allele].apply(hf.is_alternative)]['variant_BECALM_adj.P.Val'].notna().sum()}") # 29283

Number of rows in the metadata file: 73846
Number of variants within the metadata file: 46374
Number of rows in the metadata file with variant BECALM results: 29283


add element information to the elements: 

In [154]:
variant_bcalm_all_references_renamed

,name,variant_BECALM_adj.P.Val,variant_BECALM_logFC
0,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,NaN,NaN
1,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,NaN,NaN
2,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,NaN,NaN
3,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.946869,-0.045242
4,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.816996,-0.077985
...,...,...,...
46369,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,0.910014,-0.086923
46370,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,0.861277,-0.105171
46371,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,0.923302,0.050027
46372,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,0.741912,-0.150336


In [153]:
tested_metadata_bcalm

,name,variant_BECALM_adj.P.Val,variant_BECALM_logFC
0,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,NaN,NaN
1,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,NaN,NaN
2,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,NaN,NaN
3,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.946869,-0.045242
4,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.816996,-0.077985
...,...,...,...
92743,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,0.910014,-0.086923
92744,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,0.861277,-0.105171
92745,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,0.923302,0.050027
92746,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,0.741912,-0.150336


In [141]:
tested_meta_data_file_region.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand'],
      dtype='object')

In [105]:
ref_tmp_results = get_info_for_ref(metadata_file=tested_meta_data_file_region, additional_info_df=variant_becalm, additional_matching_column='variant_id', references_of_interest=ref_of_interest, columns_of_interest=['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC'], variant_region_map=variant_region_map)

In [113]:
get_info_for_ref(metadata_file=tested_meta_data_file_region, additional_info_df=variant_becalm, additional_matching_column='variant_id', references_of_interest=ref_of_interest, columns_of_interest=['tmp_variant_map_REF_ID', 'variant_BECALM_adj.P.Val', 'variant_BECALM_logFC'], variant_region_map=variant_region_map)

,tmp_variant_map_REF_ID,variant_BECALM_adj.P.Val,variant_BECALM_logFC
6,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.856513,-0.099725
7,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.915830,-0.061454
8,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.896808,-0.070486
9,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,0.912745,-0.060166


In [117]:
# group by ref_id and make variant_BECALM and variant_BECALM_logFC to list
list_of_listing_columns = ['variant_BECALM_adj.P.Val', 'variant_BECALM_logFC']
list_of_listing_cols_dict = {col: list for col in list_of_listing_columns}
ref_tmp_results.groupby('tmp_variant_map_REF_ID').agg(list_of_listing_cols_dict).reset_index()


,tmp_variant_map_REF_ID,variant_BECALM_adj.P.Val,variant_BECALM_logFC
0,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,"[0.856513151561757, 0.915829919536914, 0.89680...","[-0.0997252173527838, -0.0614538468100273, -0...."


#### Add if open 
- according to NGN2_WTC11_NP data ATAC: 'tmp_open_NGN2_WTC11_NP'
    - two different datasets: considering the original regions of the open chromatin regions from screen (416 uniqe regions overlapping atac peaks) or the in reality tested regions which were modified to fit the variants better (368 unique regions overlapping atac peaks)
    - add them and create new columns: 'tmp_open_NGN2_WTC11_NP' and 'tmp_open_NGN2_WTC11_NP_screen_regions'
    - screen overlapping regions: /home/kisa/coding/80K_MPRA/WTC11_ATAC_Ahituv/tested_elements_NGN2_wtc11_NP_overlapping_regions.bed
    - in reality tested regions overlap: /home/kisa/coding/80K_MPRA/WTC11_ATAC_Ahituv/in_reallife_ngn2_open_NP_tested_overlapping_regions.bed
- according to H1_neural_progenitor_screen: 'tmp_open_H1_neural_progenitor_screen'


In [109]:
meta_data_file.shape[0]

80215

In [110]:
meta_data_file.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'chr', 'start', 'end', 'strand', 'variant_class',
       'variant_pos', 'SPDI', 'allele', 'info', 'tmp_matching_header'],
      dtype='object')

In [297]:
# make function which gets a row from a pandas dataframe and checks if the given column has a value != NA and if so returns True else False

def column_not_na(row, column):
    return not pd.isna(row[column])

In [298]:
import ast

# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

In [299]:
# read metadata file: config['files']['creating']['region_metadata_table_local_tested_juli']
meta_data_file = pd.read_csv(config['files']['creating']['region_metadata_table_local_tested_juli'], sep="\t", low_memory=False)
meta_data_file = pd.read_csv(config['files']['creating']['datafreeze_table'], sep="\t", low_memory=False)

In [300]:
# read columns from tsv
# list columns col_variant_class, col_variant_pos, col_SPDI, col_allele,
list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]
# Apply the safe_eval function to the specified columns
for col in list_columns:
    meta_data_file[col] = meta_data_file[col].apply(safe_eval)

In [301]:
meta_data_file.loc[meta_data_file[col_name].str.contains('REF_')][col_variant_pos]

46374    None
46375    None
46376    None
46377    None
46378    None
         ... 
64941    None
64942    None
64943    None
64944    None
64945    None
Name: variant_pos, Length: 18572, dtype: object

In [302]:
meta_data_file['tmp_region_id'] = meta_data_file[col_name].apply(hf.get_region_info)
meta_data_file['tmp_label'] = meta_data_file[col_name].apply(hf.get_label)

# read bed file with open enhancer ids
wtc11_open_screen_element_region_bed = pd.read_csv(config['files']['creating']['wtc11_ngn2_atac_tested_elements_overlap'], sep="\t", header=None)
wtc11_open_screen_element_region_bed.columns = ['ngn2_open_chr', 'ngn2_open_start', 'ngn2_open_end', 'ngn2_open_name', 'ngn2_open_score', 'ngn2_open_strand']
# remove duplicated rows
print(wtc11_open_screen_element_region_bed.shape[0]) # 688
wtc11_open_screen_element_region_bed = wtc11_open_screen_element_region_bed.drop_duplicates()
print(wtc11_open_screen_element_region_bed.shape[0]) # 416
# add tmp_open_NGN2_WTC11_NP_screen_regions column if ngn2_open_start is not na
meta_data_file = meta_data_file.merge(wtc11_open_screen_element_region_bed, left_on='tmp_region_id', right_on='ngn2_open_name', how='left')
print(f'Dataframe size after left join: {meta_data_file.shape[0]}')
# get the number of unique ngn2_open_name
print(f'Unique number of matched screen ids: {meta_data_file["ngn2_open_name"].nunique()}') # 355 (61 open chromatin regions around regions open in wtc 11 but not in the final design)
# check if all elements are matched
meta_data_file.loc[~meta_data_file['ngn2_open_start'].isna()].shape[0] # 990

meta_data_file['tmp_open_NGN2_WTC11_NP_screen_regions'] = meta_data_file.apply(lambda row: column_not_na(row, 'ngn2_open_start'), axis=1)

# drop all columns starting with 'ngn2_open'
columns_to_drop = [col for col in meta_data_file.columns if col.startswith('ngn2_open')]
meta_data_file.drop(columns=columns_to_drop, inplace=True)

# check remaining columns
meta_data_file.columns

688
416
Dataframe size after left join: 73846
Unique number of matched screen ids: 355


Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_region_id', 'tmp_open_NGN2_WTC11_NP_screen_regions',
       'tmp_chr_start_end', 'tmp_open_NGN2_WTC11_NP',
       'tmp_open_h1_neuro_dnase_screen', 'tmp_MPRA_primary_region_overlap',
       'tmp_MPRA_organoid_region_overlap', 'tmp_gene_name', 'tmp_gene_set',
       'tmp_is_neuro', 'tmp_is_cardiac', 'tmp_is_cava', 'tmp_is_random'],
      dtype='object')

In [303]:
# check numbers:
meta_data_file['tmp_open_NGN2_WTC11_NP_screen_regions'].value_counts()

False    72856
True       990
Name: tmp_open_NGN2_WTC11_NP_screen_regions, dtype: int64

In [304]:
# add in reality open regions: config files, creating, in_reality_wtc11_ngn2_atac_tested_elements_overlap
in_reality_wtc11_open_region_overlap_bed = pd.read_csv(config['files']['creating']['in_reality_wtc11_ngn2_atac_tested_elements_overlap'], sep="\t", header=None)
in_reality_wtc11_open_region_overlap_bed.columns = ['ngn2_open_chr', 'ngn2_open_start', 'ngn2_open_end', 'ngn2_open_name', 'ngn2_open_score', 'ngn2_open_strand']
in_reality_wtc11_open_region_overlap_bed

def make_region_identifier(row, prefix=''):
    return f"{row[f'{prefix}chr']}_{row[f'{prefix}start']}_{row[f'{prefix}end']}"

# add region_id from header
# in_reality_wtc11_open_region_overlap_bed['ngn2_open_region_id'] = in_reality_wtc11_open_region_overlap_bed['ngn2_open_name'].apply(hf.get_region_info)
in_reality_wtc11_open_region_overlap_bed['ngn2_chr_start_end'] = in_reality_wtc11_open_region_overlap_bed.apply(lambda row: make_region_identifier(row, 'ngn2_open_'), axis=1)
# remove duplicated rows
print(in_reality_wtc11_open_region_overlap_bed.shape[0]) # 583
in_reality_wtc11_open_region_overlap_bed = in_reality_wtc11_open_region_overlap_bed.drop_duplicates()
print(in_reality_wtc11_open_region_overlap_bed.shape[0]) # 368

# add chr start end identifier
meta_data_file['tmp_chr_start_end'] = meta_data_file.apply(lambda row: make_region_identifier(row), axis=1)

# merge
meta_data_file = meta_data_file.merge(in_reality_wtc11_open_region_overlap_bed, left_on='tmp_chr_start_end', right_on='ngn2_chr_start_end', how='left')
print(f'Dataframe size after left join: {meta_data_file.shape[0]} (expected: 73846 because only tested and missing regions for 84 sequences (ALT and REF in header but not in variant region map))')

# get the number of unique ngn2_open_region_id
print(f'Unique number of matched screen ids: {meta_data_file["ngn2_chr_start_end"].nunique()}') # 367 (1 open screen element in region bed but not in final design table)
meta_data_file['tmp_open_NGN2_WTC11_NP'] = meta_data_file.apply(lambda row: column_not_na(row, 'ngn2_open_start'), axis=1)

# drop all columns starting with 'ngn2_open'
columns_to_drop = [col for col in meta_data_file.columns if col.startswith('ngn2_')]
meta_data_file.drop(columns=columns_to_drop, inplace=True)

# check remaining columns
meta_data_file.columns

583
368
Dataframe size after left join: 73846 (expected: 73846 because only tested and missing regions for 84 sequences (ALT and REF in header but not in variant region map))
Unique number of matched screen ids: 367


Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_region_id', 'tmp_open_NGN2_WTC11_NP_screen_regions',
       'tmp_chr_start_end', 'tmp_open_NGN2_WTC11_NP',
       'tmp_open_h1_neuro_dnase_screen', 'tmp_MPRA_primary_region_overlap',
       'tmp_MPRA_organoid_region_overlap', 'tmp_gene_name', 'tmp_gene_set',
       'tmp_is_neuro', 'tmp_is_cardiac', 'tmp_is_cava', 'tmp_is_random'],
      dtype='object')

In [305]:
# check the numbers:
meta_data_file['tmp_open_NGN2_WTC11_NP'].value_counts() # 1044 even more than the open in screen?

False    72802
True      1044
Name: tmp_open_NGN2_WTC11_NP, dtype: int64

In [306]:
# sanity check if tmp_open_NGN2_WTC11_NP_screen_regions and tmp_open_NGN2_WTC11_NP are the same (92 are not the same)
meta_data_file.loc[meta_data_file['tmp_open_NGN2_WTC11_NP_screen_regions'] != meta_data_file['tmp_open_NGN2_WTC11_NP']]


,header,sequence,tmp_label,name,category,class,source,ref,variant_class,variant_pos,...,tmp_open_NGN2_WTC11_NP,tmp_open_h1_neuro_dnase_screen,tmp_MPRA_primary_region_overlap,tmp_MPRA_organoid_region_overlap,tmp_gene_name,tmp_gene_set,tmp_is_neuro,tmp_is_cardiac,tmp_is_cava,tmp_is_random
1840,cardiac_neuro_cava_random:ALT_AHDC1|ENSG000001...,AGGACCGGATCAACTTCTCTGGGCCTTGGTTTCCTTCCTCCTGCAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_AHDC1|ENSG000001...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[164],...,True,True,False,False,AHDC1,"['neuro', 'cava']",True,False,True,False
1841,cardiac_neuro_cava_random:ALT_AHDC1|ENSG000001...,AGGACCGGATCAACTTCTCTGGGCCTTGGTTTCCTTCCTCCTGCAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_AHDC1|ENSG000001...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[85],...,True,True,False,False,AHDC1,"['neuro', 'cava']",True,False,True,False
3571,cardiac_neuro_cava_random:ALT_ASH1L|ENSG000001...,AGGACCGGATCAACTGTGTGTGTTTAATTAAGGGATAAGAGTGGTC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_ASH1L|ENSG000001...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[191],...,True,False,False,False,ASH1L,['neuro'],True,False,False,False
10499,cardiac_neuro_cava_random:ALT_NECAP1|ENSG00000...,AGGACCGGATCAACTAAGTGTGTGGATCTGTTGTATTTTATGGATA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_NECAP1|ENSG00000...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[180],...,True,False,False,False,NECAP1,['neuro'],True,False,False,False
13771,cardiac_neuro_cava_random:ALT_IRF2BPL|ENSG0000...,AGGACCGGATCAACTCCCGTAGAAAACTGCGAGACAACGAAACAGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_IRF2BPL|ENSG0000...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[161],...,True,False,False,False,IRF2BPL,['neuro'],True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72554,cardiac_neuro_cava_random:SMARCA4|ENSG00000127...,AGGACCGGATCAACTCAATTCCTAGAAAGGAAAAGGCAAATATAGA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SMARCA4|ENSG00000127...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,True,True,False,False,SMARCA4,"['neuro', 'cardiac']",True,True,False,False
72648,cardiac_neuro_cava_random:CNOT3|ENSG0000008803...,AGGACCGGATCAACTTCTAAATGAGACGTTCTCAACCCTGCTTATG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:CNOT3|ENSG0000008803...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,False,True,False,False,CNOT3,['neuro'],True,False,False,False
72720,cardiac_neuro_cava_random:TGFB1|ENSG0000010532...,AGGACCGGATCAACTAATGTGGCTGTCTGTGTTCCCTGAACCCTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:TGFB1|ENSG0000010532...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,True,False,False,False,TGFB1,['cardiac'],False,True,False,False
73622,cardiac_neuro_cava_random:BCOR|ENSG00000183337...,AGGACCGGATCAACTGAAGCAAGGGTGGCAGAATTTATTCTTTCCC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:BCOR|ENSG00000183337...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,True,False,False,False,BCOR,['cardiac'],False,True,False,False


990 sequences are open in WTC11 NGN2 
- how many variants and how many elements?

In [307]:
def is_reference(allele):
    """
    Since allele is list: returns if 'ref' in allele list
    """
    # check case if allele is NA
    if pd.isna(allele):
        return False
    if 'ref' in allele:
        return True
    return False

In [124]:
# !! left_join_meta_data_tested_open_wtc11 was changed to meta_data_file
# open_tested_oligos = left_join_meta_data_tested_open_wtc11.loc[~left_join_meta_data_tested_open_wtc11['ngn2_open_start'].isna()] # 990
# open_tested_oligos.loc[open_tested_oligos['allele'] == 'alt'].shape[0] # 631
# open_tested_oligos.loc[open_tested_oligos['allele'].apply(is_reference)].shape[0] # 247
# open_tested_oligos.loc[open_tested_oligos['category'] == 'element'].shape[0] # 112

In [125]:
# meta_data_file['tmp_label'].value_counts()

##### Investigate H1 neuro dnase 
- /home/kisa/coding/80K_MPRA/DNase_H1_neural_progenitor_screen/in_reallife_tested_regions_and_h1_neural_dnase_open_screen_overlapping_regions_verbose.bed
- /home/kisa/coding/80K_MPRA/DNase_H1_neural_progenitor_screen/in_reallife_tested_regions_and_h1_neural_dnase_open_screen_overlapping_regions.bed

In [126]:
# add in reality open regions: config files, creating, h1_neuro_dnase_tested_elements_overlap
in_reality_h1_dnase_open_region_overlap_bed = pd.read_csv(config['files']['creating']['h1_neuro_dnase_tested_elements_overlap'], sep="\t", header=None)
in_reality_h1_dnase_open_region_overlap_bed.columns = ['h1_open_chr', 'h1_open_start', 'h1_open_end', 'h1_open_name', 'h1_open_score', 'h1_open_strand']

# add region_id from header
in_reality_h1_dnase_open_region_overlap_bed['h1_open_region_id'] = in_reality_h1_dnase_open_region_overlap_bed['h1_open_name'].apply(hf.get_region_info)
in_reality_h1_dnase_open_region_overlap_bed

# remove duplicated rows
print(in_reality_h1_dnase_open_region_overlap_bed.shape[0]) # 2008
in_reality_h1_dnase_open_region_overlap_bed = in_reality_h1_dnase_open_region_overlap_bed.drop_duplicates()
print(f'Number of unique regions with overlap in H1 neural progenitor DNase seq data from SCREEN: {in_reality_h1_dnase_open_region_overlap_bed["h1_open_region_id"].nunique()}') # 1763

meta_data_file = meta_data_file.merge(in_reality_h1_dnase_open_region_overlap_bed, left_on='tmp_region_id', right_on='h1_open_region_id', how='left')
print(f'Dataframe size after left join: {meta_data_file.shape[0]} (expected: 73846 because only tested and missing regions for 84 sequences (ALT and REF in header but not in variant region map))')

# get the number of unique h1_open_region_id
print(f'Unique number of matched screen ids: {meta_data_file["h1_open_region_id"].nunique()}') # 1755 (8 in H1 dnase seq open tested regions but not in final design table)
meta_data_file['tmp_open_h1_neuro_dnase_screen'] = meta_data_file.apply(lambda row: column_not_na(row, 'h1_open_start'), axis=1)

# drop all columns starting with 'h1_open'
columns_to_drop = [col for col in meta_data_file.columns if col.startswith('h1_open')]
meta_data_file.drop(columns=columns_to_drop, inplace=True)

# check remaining columns
meta_data_file.columns


2008
Number of unique regions with overlap in H1 neural progenitor DNase seq data from SCREEN: 1763
Dataframe size after left join: 73846 (expected: 73846 because only tested and missing regions for 84 sequences (ALT and REF in header but not in variant region map))
Unique number of matched screen ids: 1755


Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_region_id', 'tmp_open_NGN2_WTC11_NP_screen_regions',
       'tmp_chr_start_end', 'tmp_open_NGN2_WTC11_NP',
       'tmp_open_h1_neuro_dnase_screen'],
      dtype='object')

In [127]:
# check the numbers:
meta_data_file['tmp_open_h1_neuro_dnase_screen'].value_counts() # 4920 way more than these open in WTC11 (difference DNase to ATAC?)

False    68926
True      4920
Name: tmp_open_h1_neuro_dnase_screen, dtype: int64

difference between wtc 11 and h1? 
- filter for h1 neuro and count how many are wtc11 regions / wtc11 screen regions overlaps
- 0.407 of actually tested sequences open in wtc 11 - ngn2 are open in H1 neural progenitor
- 0.388 of tested sequences associated to SCREEN open chromatin regions open in wtc11 - ngn2 atac are open in H1 neural progenitor 


In [128]:
h1_dnase_open = meta_data_file.loc[meta_data_file['tmp_open_h1_neuro_dnase_screen']]
h1_dnase_open.loc[h1_dnase_open['tmp_open_NGN2_WTC11_NP_screen_regions']].shape[0] # 384
h1_dnase_open.loc[h1_dnase_open['tmp_open_NGN2_WTC11_NP']].shape[0] # 425

# overall in the data:
# meta_data_file
# meta_data_file.loc[meta_data_file['tmp_open_NGN2_WTC11_NP_screen_regions']].shape[0] # 990
# meta_data_file.loc[meta_data_file['tmp_open_NGN2_WTC11_NP']].shape[0] # 1044

h1_overlap_wtc_atac_screen_regions = h1_dnase_open.loc[h1_dnase_open['tmp_open_NGN2_WTC11_NP_screen_regions']].shape[0] / meta_data_file.loc[meta_data_file['tmp_open_NGN2_WTC11_NP_screen_regions']].shape[0]
print(f'Number of open regions in H1 DNase seq data overlapping with WTC11 ATAC screen regions: {round(h1_overlap_wtc_atac_screen_regions,3)}')
h1_overlap_wtc_atac_actually_tested_regions = h1_dnase_open.loc[h1_dnase_open['tmp_open_NGN2_WTC11_NP']].shape[0] / meta_data_file.loc[meta_data_file['tmp_open_NGN2_WTC11_NP']].shape[0]
print(f'Number of open regions in H1 DNase seq data overlapping with WTC11 ATAC actually tested regions: {round(h1_overlap_wtc_atac_actually_tested_regions,3)}')

Number of open regions in H1 DNase seq data overlapping with WTC11 ATAC screen regions: 0.388
Number of open regions in H1 DNase seq data overlapping with WTC11 ATAC actually tested regions: 0.407


In [129]:
990/73846
1044/73846

0.014137529453186361

##### Investigate regions from paper: 
- Science paper: Single-cell genomics and regulatory networks for 388 human brains https://www.science.org/doi/10.1126/science.adi5199#supplementary-materials
- Overlap with eQTLs of variants /home/kisa/coding/80K_MPRA/cCREs_overlap/single_cell_genomics_regulatory_network/science.adi5199_data_s1_to_s33.xlsx 
    - /home/kisa/coding/80K_MPRA/cCREs_overlap/single_cell_genomics_regulatory_network
    - Data S16
        - our SPDI: NC_000001.11:2179590:T:C
        - their: chr10:125795634:C:T
    - Data S17
        - Gene_name	ASD	SCZ	BPD	AD/Aging (overlap Gene names) => has annotation from eQTLs to neuro disorder  
            - ASD: autism spectrum disorder
            - SCZ: schizophrenia      
            - BPD: borderline
            - AD/Aging: Alzheimer's disease
    - config: files, creating, from_other_paper, SC_reg_network_human_brains_emani_et_al_2024
        - genes_disease_annotation: /home/kisa/coding/80K_MPRA/cCREs_overlap/single_cell_genomics_regulatory_network/genes_neuro_dev_disease_annotation_sc_reg_network_human_brains.csv
        - identified_variants: TODO: 
- Science paper: MPRA regulatory elements in developing human cortex: https://www.science.org/doi/10.1126/science.adh0559#supplementary-materials
    - two csv (bed like tables)
    - `/home/kisa/coding/80K_MPRA/cCREs_overlap/MPRA_neuro_paper`
    - config: files, creating, from_other_paper, MPRA_dev_cortex_deng_et_al_2024
        - differentiall_expressed_genes (not too important for the overview? first column gene identifiers: merge lower case to my lower case)
        - cCREs_primary
        - cCREs_organoid
        - read in and safe as bed like format => bed tools as always 

###### Read csv data and write tsv data: changed config file inbetween

In [ ]:
# TODO: make csv from Deng et al2024 => tsv: /home/kisa/coding/80K_MPRA/cCREs_overlap/MPRA_neuro_paper/Oranoids_regions_Deng_et_al_2024.csv
# read of SC paper identified variants
# emani_et_al_2024_reg_net_human_brains_identified_variants = pd.read_csv(config['files']['creating']['from_other_paper']['SC_reg_network_human_brains_emani_et_al_2024']['identified_variants'], sep=";")
# emani_et_al_2024_reg_net_human_brains_identified_variants # 1389

# emani_et_al_2024_reg_net_human_brains_genes_disease_annotation = pd.read_csv(config['files']['creating']['from_other_paper']['SC_reg_network_human_brains_emani_et_al_2024']['genes_disease_annotation'], sep=";")
# emani_et_al_2024_reg_net_human_brains_genes_disease_annotation # 330

# # fix job: load all data into dataframes => change the config to (.tsv) and write to same file (i want all files to be tsv to be consistent)
# cCREs_primary = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_primary'], sep=";")
# cCREs_primary # 46370

# cCREs_organoid = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_organoid'], sep=";")
# cCREs_organoid # 43902
# differentiall_expressed_genes = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['differentiall_expressed_genes'], sep=";")
# differentiall_expressed_genes # 29637

# # read as tsv
# emani_et_al_2024_reg_net_human_brains_identified_variants = pd.read_csv(config['files']['creating']['from_other_paper']['SC_reg_network_human_brains_emani_et_al_2024']['identified_variants'], sep="\t")
# emani_et_al_2024_reg_net_human_brains_identified_variants # 1389

# emani_et_al_2024_reg_net_human_brains_genes_disease_annotation = pd.read_csv(config['files']['creating']['from_other_paper']['SC_reg_network_human_brains_emani_et_al_2024']['genes_disease_annotation'], sep="\t")
# emani_et_al_2024_reg_net_human_brains_genes_disease_annotation # 330

# # fix job: load all data into dataframes => change the config to (.tsv) and write to same file (i want all files to be tsv to be consistent)
# cCREs_primary = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_primary'], sep="\t")
# cCREs_primary # 46370

# cCREs_organoid = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_organoid'], sep="\t")
# cCREs_organoid # 43902

# differentiall_expressed_genes = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['differentiall_expressed_genes'], sep="\t")
# differentiall_expressed_genes # 29637


# # write csvs
# emani_et_al_2024_reg_net_human_brains_identified_variants.to_csv(config['files']['creating']['from_other_paper']['SC_reg_network_human_brains_emani_et_al_2024']['identified_variants'], sep=";", index=False)
# emani_et_al_2024_reg_net_human_brains_genes_disease_annotation.to_csv(config['files']['creating']['from_other_paper']['SC_reg_network_human_brains_emani_et_al_2024']['genes_disease_annotation'], sep=";", index=False)
# cCREs_primary.to_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_primary'], sep=";", index=False)
# cCREs_organoid.to_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_organoid'], sep=";", index=False)
# differentiall_expressed_genes.to_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['differentiall_expressed_genes'], sep=";", index=False)
# write tsv
# emani_et_al_2024_reg_net_human_brains_identified_variants.to_csv(config['files']['creating']['from_other_paper']['SC_reg_network_human_brains_emani_et_al_2024']['identified_variants'], sep="\t", index=False)
# emani_et_al_2024_reg_net_human_brains_genes_disease_annotation.to_csv(config['files']['creating']['from_other_paper']['SC_reg_network_human_brains_emani_et_al_2024']['genes_disease_annotation'], sep="\t", index=False)
# cCREs_primary.to_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_primary'], sep="\t", index=False)
# cCREs_organoid.to_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_organoid'], sep="\t", index=False)
# differentiall_expressed_genes.to_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['differentiall_expressed_genes'], sep="\t", index=False)



In [130]:
# read as tsv
emani_et_al_2024_reg_net_human_brains_identified_variants = pd.read_csv(config['files']['creating']['from_other_paper']['SC_reg_network_human_brains_emani_et_al_2024']['identified_variants'], sep="\t")
emani_et_al_2024_reg_net_human_brains_identified_variants # 1389

emani_et_al_2024_reg_net_human_brains_genes_disease_annotation = pd.read_csv(config['files']['creating']['from_other_paper']['SC_reg_network_human_brains_emani_et_al_2024']['genes_disease_annotation'], sep="\t")
emani_et_al_2024_reg_net_human_brains_genes_disease_annotation # 330

# fix job: load all data into dataframes => change the config to (.tsv) and write to same file (i want all files to be tsv to be consistent)
cCREs_primary = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_primary'], sep="\t")
cCREs_primary # 46370

cCREs_organoid = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_organoid'], sep="\t")
cCREs_organoid # 43902

differentiall_expressed_genes = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['differentiall_expressed_genes'], sep="\t")
differentiall_expressed_genes # 29637


,gene_name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
0,5S_rRNA,0,NaN,NaN,NaN,NaN,NaN
1,A1BG,"267,9148116","-4,010688725","0,293089477","-13,68417852","1,26E-42","7,87E-40"
2,A1CF,"4,854164066","4,139640408","1,81639756","2,279038741","0,022664762","0,055424689"
3,A2M,"526,6583454","0,081438689","0,351391922","0,231760276","0,816724212","0,879091484"
4,A2ML1,"13,41934408","0,359756358","0,878859147","0,409344728","0,682286696","0,782597011"
...,...,...,...,...,...,...,...
29632,ZYG11B,"2422,940843","0,479185889","0,121111077","3,956581851","7,60E-05","4,18E-04"
29633,ZYX,"1953,944391","0,320277085","0,139368658","2,298056752","0,021558559","0,053214005"
29634,ZYXP1,0,NaN,NaN,NaN,NaN,NaN
29635,ZZEF1,"2141,038036","0,844599904","0,119456968","7,07032766","1,55E-12","3,89E-11"


MPRA paper:
- use bedtools: 
- bedtools intersect -wb -a /home/kisa/coding/80K_MPRA/design_data/design_info/unique_ir_regions.bed -b /home/kisa/coding/80K_MPRA/cCREs_overlap/MPRA_neuro_paper/primary_regions_Deng_et_al_2024.bed > /home/kisa/coding/80K_MPRA/cCREs_overlap/MPRA_neuro_paper/in_reallife_tested_regions_and_primary_MPRA_dev_cortex_deng_2024_overlapping_regions_verbose.bed 
- 880 independent regions
- bedtools intersect -wa -a /home/kisa/coding/80K_MPRA/design_data/design_info/unique_ir_regions.bed -b /home/kisa/coding/80K_MPRA/cCREs_overlap/MPRA_neuro_paper/primary_regions_Deng_et_al_2024.bed > /home/kisa/coding/80K_MPRA/cCREs_overlap/MPRA_neuro_paper/in_reallife_tested_regions_and_primary_MPRA_dev_cortex_deng_2024_overlapping_regions.bed
- organoids: 
- bedtools intersect -wb -a /home/kisa/coding/80K_MPRA/design_data/design_info/unique_ir_regions.bed -b /home/kisa/coding/80K_MPRA/cCREs_overlap/MPRA_neuro_paper/Oranoids_regions_Deng_et_al_2024.bed > /home/kisa/coding/80K_MPRA/cCREs_overlap/MPRA_neuro_paper/in_reallife_tested_regions_and_organoid_MPRA_dev_cortex_deng_2024_overlapping_regions_verbose.bed 
- bedtools intersect -wa -a /home/kisa/coding/80K_MPRA/design_data/design_info/unique_ir_regions.bed -b /home/kisa/coding/80K_MPRA/cCREs_overlap/MPRA_neuro_paper/Oranoids_regions_Deng_et_al_2024.bed > /home/kisa/coding/80K_MPRA/cCREs_overlap/MPRA_neuro_paper/in_reallife_tested_regions_and_organoid_MPRA_dev_cortex_deng_2024_overlapping_regions.bed

In [132]:
# fast job: make MPRA data overlap:
cCREs_primary = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_primary'], sep="\t")
cCREs_primary # 46370
cCREs_primary['insert_name'].nunique() # 46370

cCREs_organoid = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_organoid'], sep="\t")
cCREs_organoid # 43902

# convert to bed like:
cCREs_primary_bed = cCREs_primary[['insert_chrom', 'insert_start', 'insert_end', 'insert_name']]
# cCREs_primary_bed.to_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_primary_bed'], sep="\t", index=False, header=False)

cCREs_organoid_bed = cCREs_organoid[['insert_chrom', 'insert_start', 'insert_end', 'insert_name']]
# cCREs_organoid_bed.to_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['cCREs_organoid_bed'], sep="\t", index=False, header=False)

In [133]:

# add in reality open regions: config files, creating, from_other_paper, MPRA_dev_cortex_deng_et_al_2024, primary_overlap_with_tested_bed
in_reality_MPRA_primary_region_overlap_bed = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['primary_overlap_with_tested_bed'], sep="\t", header=None)
in_reality_MPRA_primary_region_overlap_bed.columns = ['MPRA_primary_chr', 'MPRA_primary_start', 'MPRA_primary_end', 'MPRA_primary_name', 'MPRA_primary_score', 'MPRA_primary_strand']

# add region_id from header
in_reality_MPRA_primary_region_overlap_bed['MPRA_primary_region_id'] = in_reality_MPRA_primary_region_overlap_bed['MPRA_primary_name'].apply(hf.get_region_info)
in_reality_MPRA_primary_region_overlap_bed

# remove duplicated rows
print(in_reality_MPRA_primary_region_overlap_bed.shape[0]) # 880
in_reality_MPRA_primary_region_overlap_bed = in_reality_MPRA_primary_region_overlap_bed.drop_duplicates()
print(f'Number of unique regions with overlap in H1 neural progenitor DNase seq data from SCREEN: {in_reality_MPRA_primary_region_overlap_bed["MPRA_primary_region_id"].nunique()}') # 880

meta_data_file = meta_data_file.merge(in_reality_MPRA_primary_region_overlap_bed, left_on='tmp_region_id', right_on='MPRA_primary_region_id', how='left')
print(f'Dataframe size after left join: {meta_data_file.shape[0]} (expected: 73846 because only tested and missing regions for 84 sequences (ALT and REF in header but not in variant region map))')

# get the number of unique MPRA_primary_region_id
print(f'Unique number of matched screen ids: {meta_data_file["MPRA_primary_region_id"].nunique()}') # 1755 (8 in H1 dnase seq open tested regions but not in final design table)
meta_data_file['tmp_MPRA_primary_region_overlap'] = meta_data_file.apply(lambda row: column_not_na(row, 'MPRA_primary_start'), axis=1)

# drop all columns starting with 'MPRA_primary'
columns_to_drop = [col for col in meta_data_file.columns if col.startswith('MPRA_primary')]
meta_data_file.drop(columns=columns_to_drop, inplace=True)

# check remaining columns
meta_data_file.columns

880
Number of unique regions with overlap in H1 neural progenitor DNase seq data from SCREEN: 880
Dataframe size after left join: 73846 (expected: 73846 because only tested and missing regions for 84 sequences (ALT and REF in header but not in variant region map))
Unique number of matched screen ids: 880


Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_region_id', 'tmp_open_NGN2_WTC11_NP_screen_regions',
       'tmp_chr_start_end', 'tmp_open_NGN2_WTC11_NP',
       'tmp_open_h1_neuro_dnase_screen', 'tmp_MPRA_primary_region_overlap'],
      dtype='object')

In [134]:
meta_data_file['tmp_MPRA_primary_region_overlap'].value_counts() # 2626


False    71220
True      2626
Name: tmp_MPRA_primary_region_overlap, dtype: int64

In [135]:

# add in reality open regions: config files, creating, from_other_paper, MPRA_dev_cortex_deng_et_al_2024, organoid_overlap_with_tested_bed
in_reality_MPRA_organoid_region_overlap_bed = pd.read_csv(config['files']['creating']['from_other_paper']['MPRA_dev_cortex_deng_et_al_2024']['organoid_overlap_with_tested_bed'], sep="\t", header=None)
in_reality_MPRA_organoid_region_overlap_bed.columns = ['MPRA_organoid_chr', 'MPRA_organoid_start', 'MPRA_organoid_end', 'MPRA_organoid_name', 'MPRA_organoid_score', 'MPRA_organoid_strand']

# add region_id from header
in_reality_MPRA_organoid_region_overlap_bed['MPRA_organoid_region_id'] = in_reality_MPRA_organoid_region_overlap_bed['MPRA_organoid_name'].apply(hf.get_region_info)
in_reality_MPRA_organoid_region_overlap_bed

# remove duplicated rows
print(in_reality_MPRA_organoid_region_overlap_bed.shape[0]) # 862
in_reality_MPRA_organoid_region_overlap_bed = in_reality_MPRA_organoid_region_overlap_bed.drop_duplicates()
print(f'Number of unique regions with overlap in H1 neural progenitor DNase seq data from SCREEN: {in_reality_MPRA_organoid_region_overlap_bed["MPRA_organoid_region_id"].nunique()}') # 862

meta_data_file = meta_data_file.merge(in_reality_MPRA_organoid_region_overlap_bed, left_on='tmp_region_id', right_on='MPRA_organoid_region_id', how='left')
print(f'Dataframe size after left join: {meta_data_file.shape[0]} (expected: 73846 because only tested and missing regions for 84 sequences (ALT and REF in header but not in variant region map))')

# get the number of unique MPRA_organoid_region_id
print(f'Unique number of matched screen ids: {meta_data_file["MPRA_organoid_region_id"].nunique()}') # 1755 (8 in H1 dnase seq open tested regions but not in final design table)
meta_data_file['tmp_MPRA_organoid_region_overlap'] = meta_data_file.apply(lambda row: column_not_na(row, 'MPRA_organoid_start'), axis=1)

# drop all columns starting with 'MPRA_organoid'
columns_to_drop = [col for col in meta_data_file.columns if col.startswith('MPRA_organoid')]
meta_data_file.drop(columns=columns_to_drop, inplace=True)

# check remaining columns
meta_data_file.columns

862
Number of unique regions with overlap in H1 neural progenitor DNase seq data from SCREEN: 862
Dataframe size after left join: 73846 (expected: 73846 because only tested and missing regions for 84 sequences (ALT and REF in header but not in variant region map))
Unique number of matched screen ids: 862


Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_region_id', 'tmp_open_NGN2_WTC11_NP_screen_regions',
       'tmp_chr_start_end', 'tmp_open_NGN2_WTC11_NP',
       'tmp_open_h1_neuro_dnase_screen', 'tmp_MPRA_primary_region_overlap',
       'tmp_MPRA_organoid_region_overlap'],
      dtype='object')

In [136]:
meta_data_file['tmp_MPRA_organoid_region_overlap'].value_counts() # 2591

False    71255
True      2591
Name: tmp_MPRA_organoid_region_overlap, dtype: int64

investigate the overlaps
- number of unique regions investigated: 
- open in h1 
  - screen regions: tmp_open_h1_neuro_dnase_screen
  - tested regions
- open in wtc11 
  - screen regions: tmp_open_NGN2_WTC11_NP_screen_regions
  - tested regions: tmp_open_NGN2_WTC11_NP
- overlap with MPRA paper 
  - primary: tmp_MPRA_primary_region_overlap
  - organoid: tmp_MPRA_organoid_region_overlap
  - genes TODO: add columns
- overlap with single-cell paper (TODO: add columns)
  - overlap with disease associated genes 
  - overlap with variants 

##### Add gene information to metadata file

In [138]:
def get_gene_name(header, with_controls=True):
    """Returns the gene name: cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778533_fwd_tile1-1 => SKI"""
    if 'cardiac_neuro_cava_random' not in header:
        if not with_controls:
            print(header)
            raise ValueError("Function only defined for tested headers")
        return "Not given - control sequence"


    if 'ALT_' in header:
        return header.split(":ALT_")[1].split('|')[0]

    if 'REF_' in header:
        return header.split(":REF_")[1].split('|')[0]

    return header.split(':')[1].split('|')[0]


def get_gene_lookup_dict(gene_lists):
    """
    prepare a dict which has gene_name: (list of associated gene_sets)
    :gene_lists - list of files containing gene names and have useful names
    """
    gene_lookup_dict = defaultdict(list)
    for file in gene_lists:
        with open(os.path.join(gene_list_dir, file), 'r') as f:
            gene_list = f.read().splitlines()
            gene_list_name = file.split('.')[0]
            for gene in gene_list:
                gene_lookup_dict[gene].append(gene_list_name)
    return gene_lookup_dict

In [139]:
from collections import defaultdict

# get gene_list files:
gene_list_dir = '/home/kisa/coding/80K_MPRA/MPRA_design/resources/gene_lists'
gene_lists = os.listdir(gene_list_dir)

# prepare a dict which has gene_name: (tuple of associated gene_sets)
gene_lookup_dict = get_gene_lookup_dict(gene_lists=gene_lists)
# # get gene name from variant id
meta_data_file['tmp_gene_name'] = meta_data_file[col_name].apply(get_gene_name)
# add gene_set annotation

meta_data_file['tmp_gene_set'] = meta_data_file['tmp_gene_name'].apply(lambda gene: gene_lookup_dict[gene] if not 'control' in gene else ['control'])

meta_data_file['tmp_gene_set'].value_counts()

[neuro]                   35493
[cardiac]                 21131
[cava]                     8691
[random]                   4711
[neuro, cardiac]           2246
[neuro, cava]               858
[cava, cardiac]             513
[neuro, cava, cardiac]      203
Name: tmp_gene_set, dtype: int64

In [140]:
meta_data_file['tmp_is_neuro'] = meta_data_file['tmp_gene_set'].apply(lambda x: 'neuro' in x)
meta_data_file['tmp_is_cardiac'] = meta_data_file['tmp_gene_set'].apply(lambda x: 'cardiac' in x)
meta_data_file['tmp_is_cava'] = meta_data_file['tmp_gene_set'].apply(lambda x: 'cava' in x)
meta_data_file['tmp_is_random'] = meta_data_file['tmp_gene_set'].apply(lambda x: 'random' in x)


In [141]:
print(meta_data_file.shape[0])
meta_data_file.columns

73846


Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_region_id', 'tmp_open_NGN2_WTC11_NP_screen_regions',
       'tmp_chr_start_end', 'tmp_open_NGN2_WTC11_NP',
       'tmp_open_h1_neuro_dnase_screen', 'tmp_MPRA_primary_region_overlap',
       'tmp_MPRA_organoid_region_overlap', 'tmp_gene_name', 'tmp_gene_set',
       'tmp_is_neuro', 'tmp_is_cardiac', 'tmp_is_cava', 'tmp_is_random'],
      dtype='object')

In [142]:
meta_data_file.head()

,header,sequence,tmp_label,name,category,class,source,ref,variant_class,variant_pos,...,tmp_open_NGN2_WTC11_NP,tmp_open_h1_neuro_dnase_screen,tmp_MPRA_primary_region_overlap,tmp_MPRA_organoid_region_overlap,tmp_gene_name,tmp_gene_set,tmp_is_neuro,tmp_is_cardiac,tmp_is_cava,tmp_is_random
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[83],...,True,False,False,False,SKI,[neuro],True,False,False,False
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[181],...,False,False,False,False,SKI,[neuro],True,False,False,False
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCTGGGTGACCCGGAGAACACCAAGGCTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[43],...,False,False,False,False,SKI,[neuro],True,False,False,False
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCATGCGGTGGCCACAGCCTCGGGTGAGTTC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[116],...,False,False,False,False,SKI,[neuro],True,False,False,False
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGGACTCCGGTGCCTTCGCATTCCCGAGCTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[205],...,False,False,False,False,SKI,[neuro],True,False,False,False


In [143]:
# investigate number of uniquely defined regions
n_unique_regions_to_be_tested = meta_data_file.drop_duplicates(subset=['chr', 'start', 'end']).shape[0] # 27199

In [144]:
# make table of overlapping designed sequences (7891)
sequences_with_overlaps = meta_data_file.loc[meta_data_file['tmp_MPRA_organoid_region_overlap'] | meta_data_file['tmp_MPRA_primary_region_overlap'] | meta_data_file['tmp_open_h1_neuro_dnase_screen'] | meta_data_file['tmp_open_NGN2_WTC11_NP_screen_regions'] | meta_data_file['tmp_open_NGN2_WTC11_NP']]
n_sequences_with_overlaps = sequences_with_overlaps.shape[0] # 7891

# open chromatin focus wtc11

open_wtc11_tested_regions = meta_data_file.loc[meta_data_file['tmp_open_NGN2_WTC11_NP']]
n_open_wtc11_tested_regions = open_wtc11_tested_regions.shape[0] # 1044
print(f'Number of of overlap: n_open_wtc11_tested_regions: {n_open_wtc11_tested_regions} ({round(n_open_wtc11_tested_regions/ n_unique_regions_to_be_tested * 100, 3)}%)')
open_wtc11_tested_regions_screen = meta_data_file.loc[meta_data_file['tmp_open_NGN2_WTC11_NP_screen_regions']]
n_open_wtc11_tested_regions_screen = open_wtc11_tested_regions_screen.shape[0] # 990
print(f'Number of of overlap: n_open_wtc11_tested_regions_screen: {n_open_wtc11_tested_regions_screen} ({round(n_open_wtc11_tested_regions_screen/ n_unique_regions_to_be_tested * 100, 3)}%)')

# open in h1

open_h1_neuro_dnase_screen = meta_data_file.loc[meta_data_file['tmp_open_h1_neuro_dnase_screen']]
n_open_h1_neuro_dnase_screen = open_h1_neuro_dnase_screen.shape[0] # 4920
print(f'Number of of overlap: n_open_h1_neuro_dnase_screen: {n_open_h1_neuro_dnase_screen} ({round(n_open_h1_neuro_dnase_screen/ n_unique_regions_to_be_tested * 100, 3)}%)')

# MPRA primary and organoid
primary_neuro_cell_overlap = meta_data_file.loc[meta_data_file['tmp_MPRA_primary_region_overlap']]
n_primary_neuro_cell_overlap = primary_neuro_cell_overlap.shape[0] # 2626
print(f'Number of of overlap: n_primary_neuro_cell_overlap: {n_primary_neuro_cell_overlap} ({round(n_primary_neuro_cell_overlap/ n_unique_regions_to_be_tested * 100, 3)}%)')
organoid_neuro_cell_overlap = meta_data_file.loc[meta_data_file['tmp_MPRA_organoid_region_overlap']]
n_organoid_neuro_cell_overlap = organoid_neuro_cell_overlap.shape[0] # 2591
print(f'Number of of overlap: n_organoid_neuro_cell_overlap: {n_organoid_neuro_cell_overlap} ({round(n_organoid_neuro_cell_overlap/ n_unique_regions_to_be_tested * 100, 3)}%)')

Number of of overlap: n_open_wtc11_tested_regions: 1044 (3.838%)
Number of of overlap: n_open_wtc11_tested_regions_screen: 990 (3.64%)
Number of of overlap: n_open_h1_neuro_dnase_screen: 4920 (18.089%)
Number of of overlap: n_primary_neuro_cell_overlap: 2626 (9.655%)
Number of of overlap: n_organoid_neuro_cell_overlap: 2591 (9.526%)


open chromatin focus: 
- number of neuro, cardiac, cava random (and proportion)

In [145]:
# overall ratio
tested_regions_open_wtc11_ratio = n_open_wtc11_tested_regions / n_unique_regions_to_be_tested # 0.03838 (3.8%)
tested_regions_open_wtc11_screen_ratio = n_open_wtc11_tested_regions_screen / n_unique_regions_to_be_tested # 0.0364 (3.6%)
tested_regions_open_wtc11_screen_ratio

0.036398396999889705

In [146]:
n_open_wtc11_tested_regions
n_open_h1_neuro_dnase_screen
n_primary_neuro_cell_overlap
n_organoid_neuro_cell_overlap

2591

In [147]:
print('Proportion of neuro associated sequences with overlap to open chromatin regions: ', open_wtc11_tested_regions.loc[open_wtc11_tested_regions['tmp_is_neuro']].shape[0] / open_wtc11_tested_regions.shape[0], open_wtc11_tested_regions.shape[0]) # ~0.596

Proportion of neuro associated sequences with overlap to open chromatin regions:  0.5957854406130269 1044


##### Add gene information to metadata file

In [148]:
def get_gene_name(header, with_controls=True):
    """Returns the gene name: cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778533_fwd_tile1-1 => SKI"""
    if 'cardiac_neuro_cava_random' not in header:
        if not with_controls:
            print(header)
            raise ValueError("Function only defined for tested headers")
        return "Not given - control sequence"


    if 'ALT_' in header:
        return header.split(":ALT_")[1].split('|')[0]

    if 'REF_' in header:
        return header.split(":REF_")[1].split('|')[0]

    return header.split(':')[1].split('|')[0]


def get_gene_lookup_dict(gene_lists):
    """
    prepare a dict which has gene_name: (list of associated gene_sets)
    :gene_lists - list of files containing gene names and have useful names
    """
    gene_lookup_dict = defaultdict(list)
    for file in gene_lists:
        with open(os.path.join(gene_list_dir, file), 'r') as f:
            gene_list = f.read().splitlines()
            gene_list_name = file.split('.')[0]
            for gene in gene_list:
                gene_lookup_dict[gene].append(gene_list_name)
    return gene_lookup_dict

In [149]:
from collections import defaultdict

# get gene_list files:
gene_list_dir = '/home/kisa/coding/80K_MPRA/MPRA_design/resources/gene_lists'
gene_lists = os.listdir(gene_list_dir)

# prepare a dict which has gene_name: (tuple of associated gene_sets)
gene_lookup_dict = get_gene_lookup_dict(gene_lists=gene_lists)

In [150]:
# # get gene name from variant id
meta_data_file['tmp_gene_name'] = meta_data_file[col_name].apply(get_gene_name)
# add gene_set annotation

meta_data_file['tmp_gene_set'] = meta_data_file['tmp_gene_name'].apply(lambda gene: gene_lookup_dict[gene] if not 'control' in gene else ['control'])

meta_data_file['tmp_gene_set'].value_counts()

[neuro]                   35493
[cardiac]                 21131
[cava]                     8691
[random]                   4711
[neuro, cardiac]           2246
[neuro, cava]               858
[cava, cardiac]             513
[neuro, cava, cardiac]      203
Name: tmp_gene_set, dtype: int64

In [151]:
meta_data_file['tmp_is_neuro'] = meta_data_file['tmp_gene_set'].apply(lambda x: 'neuro' in x)
meta_data_file['tmp_is_cardiac'] = meta_data_file['tmp_gene_set'].apply(lambda x: 'cardiac' in x)
meta_data_file['tmp_is_cava'] = meta_data_file['tmp_gene_set'].apply(lambda x: 'cava' in x)
meta_data_file['tmp_is_random'] = meta_data_file['tmp_gene_set'].apply(lambda x: 'random' in x)


Store the current state of the metadata file as Datafreeze

In [152]:
meta_data_file
# make sure that the allele column is a list of values ['alt'] and ['ref', ...]

,header,sequence,tmp_label,name,category,class,source,ref,variant_class,variant_pos,...,tmp_open_NGN2_WTC11_NP,tmp_open_h1_neuro_dnase_screen,tmp_MPRA_primary_region_overlap,tmp_MPRA_organoid_region_overlap,tmp_gene_name,tmp_gene_set,tmp_is_neuro,tmp_is_cardiac,tmp_is_cava,tmp_is_random
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[83],...,True,False,False,False,SKI,[neuro],True,False,False,False
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[181],...,False,False,False,False,SKI,[neuro],True,False,False,False
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCTGGGTGACCCGGAGAACACCAAGGCTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[43],...,False,False,False,False,SKI,[neuro],True,False,False,False
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCATGCGGTGGCCACAGCCTCGGGTGAGTTC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[116],...,False,False,False,False,SKI,[neuro],True,False,False,False
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGGACTCCGGTGCCTTCGCATTCCCGAGCTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[205],...,False,False,False,False,SKI,[neuro],True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73841,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGACTCTGGGCTGCTCAGAGGCTGCCTTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,False,False,False,False,FLNA,[cardiac],False,True,False,False
73842,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGAGCCCTGGGGAACGCCATGAGCCCTCAGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,False,False,False,False,FLNA,[cardiac],False,True,False,False
73843,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGGACTTAAACCCCAGCCTCCCCCGTCCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,False,False,False,False,FLNA,[cardiac],False,True,False,False
73844,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGCCCATAATTTATTGATTTTTTAAAATTTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,False,False,False,False,FLNA,[cardiac],False,True,False,False


In [154]:
meta_data_file.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_region_id', 'tmp_open_NGN2_WTC11_NP_screen_regions',
       'tmp_chr_start_end', 'tmp_open_NGN2_WTC11_NP',
       'tmp_open_h1_neuro_dnase_screen', 'tmp_MPRA_primary_region_overlap',
       'tmp_MPRA_organoid_region_overlap', 'tmp_gene_name', 'tmp_gene_set',
       'tmp_is_neuro', 'tmp_is_cardiac', 'tmp_is_cava', 'tmp_is_random'],
      dtype='object')

In [155]:
meta_data_file.loc[meta_data_file[col_name].str.contains('REF_')][col_variant_pos]

46374    None
46375    None
46376    None
46377    None
46378    None
         ... 
64941    None
64942    None
64943    None
64944    None
64945    None
Name: variant_pos, Length: 18572, dtype: object

In [158]:
# meta_data_file.to_csv(config['files']['creating']['datafreeze_table'], sep="\t", index=False)
# write_data_with_json(meta_data_file, config['files']['creating']['datafreeze_table'], meta_data_file.columns)


,header,sequence,tmp_label,name,category,class,source,ref,variant_class,variant_pos,...,tmp_open_NGN2_WTC11_NP,tmp_open_h1_neuro_dnase_screen,tmp_MPRA_primary_region_overlap,tmp_MPRA_organoid_region_overlap,tmp_gene_name,tmp_gene_set,tmp_is_neuro,tmp_is_cardiac,tmp_is_cava,tmp_is_random
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,"[""SNV""]",[83],...,True,False,False,False,SKI,[neuro],True,False,False,False
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,"[""SNV""]",[181],...,False,False,False,False,SKI,[neuro],True,False,False,False
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCTGGGTGACCCGGAGAACACCAAGGCTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,"[""SNV""]",[43],...,False,False,False,False,SKI,[neuro],True,False,False,False
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCATGCGGTGGCCACAGCCTCGGGTGAGTTC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,"[""SNV""]",[116],...,False,False,False,False,SKI,[neuro],True,False,False,False
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGGACTCCGGTGCCTTCGCATTCCCGAGCTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,"[""SNV""]",[205],...,False,False,False,False,SKI,[neuro],True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73841,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGACTCTGGGCTGCTCAGAGGCTGCCTTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,False,False,False,False,FLNA,[cardiac],False,True,False,False
73842,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGAGCCCTGGGGAACGCCATGAGCCCTCAGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,False,False,False,False,FLNA,[cardiac],False,True,False,False
73843,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTTTGGGACTTAAACCCCAGCCTCCCCCGTCCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,False,False,False,False,FLNA,[cardiac],False,True,False,False
73844,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,AGGACCGGATCAACTAGCCCATAATTTATTGATTTTTTAAAATTTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,None,None,...,False,False,False,False,FLNA,[cardiac],False,True,False,False


In [ ]:
asdf breakpoint asdf

#### Add BCalm results: ref and variant 

In [ ]:
meta_data_file.columns

In [ ]:
variant_input_path = config['files']['creating']['bc_MPRAlm_bbmap_standard_mapq35_noLength']

variant_becalm_raw = pd.read_csv(variant_input_path, sep="\t")
variant_becalm_raw.columns = ['tmp_variant_BECALM_logFC', 'tmp_variant_BECALM_AveExpr', 'tmp_variant_BECALM_t', 'tmp_variant_BECALM_P.Value', 'tmp_variant_BECALM_adj.P.Val', 'tmp_variant_BECALM_B', 'tmp_variant_BECALM_variant_id']

# important columns:
becalm_variant_important_columns = ['tmp_variant_BECALM_variant_id', 'tmp_variant_BECALM_logFC', 'tmp_variant_BECALM_adj.P.Val']
bcMPRAlm_table = variant_becalm_raw[becalm_variant_important_columns]

metadata_file_alt_sequences = metadata_file.loc[metadata_file[col_allele].apply(hf.is_alternative)]

# join with variant information
metadata_file_alt_sequences_alt_seqs_variant_info = metadata_file_alt_sequences.merge(variant_region_map, left_on=col_name, right_on='tmp_variant_map_ALT_ID', how='left')
# join with becalm variant info
metadata_file_alt_seqs_variant_info_becalm = metadata_file_alt_sequences_alt_seqs_variant_info.merge(bcMPRAlm_table, left_on='tmp_variant_map_Variant_id', right_on='tmp_variant_BECALM_variant_id', how='left')

metadata_file_alt_seqs_variant_info_becalm['is_significant'] = metadata_file_alt_seqs_variant_info_becalm['tmp_variant_BECALM_adj.P.Val'] < 0.05

In [ ]:
# negative neuron np vs tested (left sided)
element_analysis_r_result = pd.read_csv(config['files']['creating']['element_analysis_np_negative_neuron_vs_tested'], sep="\t")
element_analysis_r_result
# # get gene name from variant id
element_analysis_r_result['gene_name'] = element_analysis_r_result['variant_id'].apply(get_gene_name_from_element)
# add gene_set annotation
element_analysis_r_result['gene_set'] = element_analysis_r_result['gene_name'].apply(lambda gene: gene_lookup_dict[gene])

# add gene info boolean columns
element_analysis_r_result['is_neuro'] = element_analysis_r_result['gene_set'].apply(lambda x: 'neuro' in x)
element_analysis_r_result['is_cardiac'] = element_analysis_r_result['gene_set'].apply(lambda x: 'cardiac' in x)
element_analysis_r_result['is_cava'] = element_analysis_r_result['gene_set'].apply(lambda x: 'cava' in x)
element_analysis_r_result['is_random'] = element_analysis_r_result['gene_set'].apply(lambda x: 'random' in x)

# add significant column
element_analysis_r_result['is_significant'] = element_analysis_r_result['adj.P.Val'] < 0.05

#### Control
- design: GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs9661525|STARR-seq-HepG2_fwd_tile1-1
- variant_map: GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2,rs9661525|STARR-seq-HepG2_fwd_tile1-1
- GC_Mendelian_variants:REF_chr8:11703860G*T|GATA4_headerDuplicate2_2_headerDuplicate1_2

- I can use the region file for: 
    - GC_Atrial_fib
    - GC_Cort_Chengyu
    - GC_GABA_Chengyu
    - GC_Glut_Chengyu
    - GC_Hon
    - GC_Kircher
    - GC_Liang
    - GC_Mohlke
    - GC_Selvarajan
    - GC_Vista

=> I can not use the region file for 
    - MK                                   2397 
    - C_positive_heart_AB                   909
    - C_negative_heart_MK                   243 (done)
    - C_negative_neuron_MK                  222 (done)
    - C_negative_neuron_NP                  217 (done)
    - GC_Mendelian_variants                 209 
      - bed file:  /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/oligo_design/GC_Mendelian_variants/design_variants.regions.bed.gz: bed file without label and without "REF_" pattern and > instead of *;
      - variant region map:/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/oligo_design/GC_Mendelian_variants/design_variants.variant_region_map.tsv.gz
    - C_SLEA                                200 (done)
    - C_positive_neuron_NP                   99 (done)
    - C_positive_heart_CAD                   97  
      - variant region map: /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/oligo_design/C_positive_heart_CAD/design_variants.region_map.tsv.gz
      - region bed: /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/oligo_design/C_positive_heart_CAD/design_variants.regions.bed.gz)
    - C_positive_heart_MK                    97 (done)
    - C_positive_neuron_MK                   96 (done)
    - C_positive_neuron_CD                   94 (done)
    - GC_DNase_positive_shuffeled            55 (done)
    - GC_DNase_positive                      41 (done)
    - GC_DNase_negative_blood_shuffeled      19 (done)
    - GC_DNase_negative_brain_shuffeled      16 (done)
    - GC_DNase_negative_brain                15 
    - GC_DNase_negative_blood                15


In [ ]:
# column names
col_name = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class' # SNP
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'

# control bed files
mendelian_variants_bed = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/oligo_design/GC_Mendelian_variants/design_variants.regions.bed.gz'
CAD_region_bed = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/oligo_design/C_positive_heart_CAD/design_variants.regions.bed.gz'


def get_start_end_strand_control(row):
    """
    Special cases to set chr, start, end and strand for control sequences from their header (because not in region bed)
    Case: C_positive_heart_AB:
            header: C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.1:chr1:2226527-2226797:Length::270
                => chr: 1, start: 2226527, end: 2226797, strand: + (assume all are +)

    Case: MK: (all MK sequences have "|chr" pattern)
            header: MK:rdhs_664198|chr8-10365156+10365425|reference
                => chr: 8, start: 10365156, end: 10365425, strand: +
            header: MK:ACAP2|chr3-195443140-195443409|reference
                => chr: 3, start: 195443140, end: 195443409, strand: -

    Case: C_negative_neuron_MK: (all C_negative_neuron_MK sequences have "_chr" pattern)
            header: C_negative_neuron_MK:tile_14444_chr15_67066278_67066547_reference__1.1385203581298
                => chr: 15, start: 67066278, end: 67066547, strand: . (no information given)

    Case: C_positive_neuron_MK: (all C_positive_neuron_MK sequences have "_chr" pattern")
            header: C_positive_neuron_MK:tile_35742_chr6_3247831_3248100_reference_0.892141141777512
                => chr: 6, start: 3247831, end: 3248100, strand: . (no information given)

    Case: C_negative_heart_MK: ( all C_negative_heart_MK sequences have "_chr" pattern)
            header: C_negative_heart_MK:tile_6903_chr11_9614045_9614314_reference__0.958461950470297
                => chr: 11, start: 9614045, end: 9614314, strand: . (no information given)

    Case: C_positive_heart_MK: (all C_positive_heart_MK sequences have "_chr" pattern)
            header: C_positive_heart_MK:tile_7939_chr11_65487592_65487861_reference_1.25449216981846
                => chr: 11, start: 65487592, end: 65487861, strand: . (no information given)

    Case: C_negative_neuron_NP: (all C_negative_neuron_NP sequences have "_chr" pattern)
            header: C_negative_neuron_NP:GW18_PFC_ABC_chr15_89400286_89400556_0.830617698776558
                => chr: 15, start: 89400286, end: 89400556, strand: . (no information given)
            additional condition: scrambled_control____2.28928058620308
                => category: scrambled

    Case: C_positive_neuron_NP: (all C_positive_neuron_NP sequences have "_chr" pattern)
            header: C_positive_neuron_NP:GW18_PFC_ABC_chr11_65487667_65487937_5.27702983385667
                => chr: 11, start: 65487667, end: 65487937, strand: . (no information given)

    Case: C_SLEA: (all C_SLEA sequences have ":chr" pattern and all have "|" as separator)
            header: C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:V_HNF4_Q6:AAGGTCCAG;155:V_HNF4_Q6:AAGGTCCAG
                => chr: 2, start: 210861483, end: 210861650, strand: . (no information given)
            additional condition: category: 'synthetic', ref: 'hg18', strand: '.'

    Case: C_positive_heart_CAD: (headers need to be split by "_")
            header: C_positive_heart_CAD:REF_rs604723
                => use region map

    Case: C_positive_neuron_CD: headers have "::chr" pattern and delimited by "-mean_ratio"
            header: C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A::chr4:155359187-155359457-mean_ratio2.42
                => chr: 4, start: 155359187, end: 155359457, strand: . (no information given)
            additional condition: C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_94-mean_ratio2.31

    Case:  GC_DNase_positive: header have ":chr" pattern delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_positive:chr9:88419918-88420187_active_count_112[.]

    Case: GC_DNase_positive_shuffeled: header have ":chr" pattern delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_positive_shuffeled:chr1:121484605-121484874_active_count_114
                => chr: 1, start: 121484605, end: 121484874, strand: . (no information given)
                + shuffled

    Case: GC_DNase_negative_blood_shuffeled: header have ":chr" pattern and delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_negative_blood_shuffeled:chr1:8213090-8213359_active_count_12_1
                => chr: 1, start: 8213090, end: 8213359, strand: . (no information given)
                + shuffled

    Case: GC_DNase_negative_brain_shuffeled: header have ":chr" pattern and delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_negative_brain_shuffeled:chr1:4768943-4769212_active_count_9_2
                => chr: 1, start: 4768943, end: 4769212, strand: . (no information given)
                + shuffled

    Case: GC_DNase_negative_brain: header have ":chr" pattern and delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_negative_brain:chr1:4768943-4769212_active_count_9_2[.]
                => chr: 1, start: 4768943, end: 4769212, strand: . (no information given)

    Case: GC_DNase_negative_blood: header have ":chr" pattern and delimited by "_active_count_" (Coordinates are based on GRCh37)
            header: GC_DNase_negative_blood:chr1:8213090-8213359_active_count_12_1[.]
                => chr: 1, start: 8213090, end: 8213359, strand: . (no information given)
    """
    name = row[col_name]
    print(name) # TODO: remove debug
    if name.startswith('C_positive_heart_AB'): # >C_positive_heart_AB:SKI-ENST00000378536.5:Oligo-Sub-Id:0.1:chr1:2226527-2226797:Length::270
        region_info = name.split(':chr')[1].split(':Length')[0] # 1:2226527-2226797
        row[col_chr] = region_info.split(':')[0]
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.' # TODO: make sure which strand they are on
        row[col_class] = 'element inactive control'
        row[col_info] = row[col_info] + '; strand information not restorable from header'

    elif name.startswith('MK'):
        row[col_class] = 'element inactive control' # for the tested sequences the MK scrambled can be used as negative control; the rest of MK is of unknown importance
        if 'scramble' in name:
            row[col_category] = 'scrambled'
            return row
        region_info = name.split('|chr')[1].split('|')[0] # 8-10365156+10365425 or 3-195443140-195443409
        row[col_chr] = region_info.split('-')[0]
        row[col_strand] = '+' if '+' in region_info else '-'
        genomic_range = region_info.split('-')[1]
        if row[col_strand] == '-': # if "-" strand then genomic region is already split
            genomic_range = '-'.join(region_info.split('-')[1:])
        row[col_start] = int(genomic_range.split(row[col_strand])[0])
        row[col_end] = int(genomic_range.split(row[col_strand])[1])

    elif name.startswith('C_negative_heart_MK') or name.startswith('C_negative_neuron_MK') or name.startswith('C_positive_heart_MK') or name.startswith('C_positive_neuron_MK'):
        row[col_class] = 'element inactive control'
        if 'positive_neuron' in name:
            row[col_class] = 'element active control'
        region_info = '_'.join(name.split('_chr')[1].split('_')[:3]) # 11_9614045_9614314
        print(region_info)
        row[col_chr] = region_info.split('_')[0]
        row[col_start] = int(region_info.split('_')[1])
        row[col_end] = int(region_info.split('_')[2])
        row[col_strand] = '.'

    elif name.startswith('C_negative_neuron_NP') or name.startswith('C_positive_neuron_NP'):
        row[col_class] = 'element inactive control'
        if 'scramble' in name:
            row[col_category] = 'scrambled'
            return row
        if 'positive_neuron' in name:
            row[col_class] = 'element active control'
        region_info = '_'.join(name.split('_chr')[1].split('_')[:3]) # 11_9614045_9614314
        print(region_info)
        row[col_chr] = region_info.split('_')[0]
        row[col_start] = int(region_info.split('_')[1])
        row[col_end] = int(region_info.split('_')[2])
        row[col_strand] = '.'

    elif name.startswith('C_SLEA'):
        row[col_class] = 'element inactive control'
        row[col_category] = 'synthetic'
        row[col_ref] = 'hg18'
        region_info = name.split(':chr')[1].split('|')[0] # 2:210861483-210861650
        row[col_chr] = region_info.split(':')[0]
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'

    elif name.startswith('C_positive_heart_CAD'):
        row[col_class] = 'element inactive control'
        print("TODO: use region map + bed file or add this information to the final variant map / bed file")

    elif name.startswith('C_positive_neuron_CD'):
        row[col_class] = 'element active control'
        if 'NA_NA_NA' in name:
            row[col_category] = 'scrambled'
            row[col_chr] = 'NA'
            row[col_start] = 'NA'
            row[col_end] = 'NA'
            row[col_strand] = 'NA'
            row[col_info] = row[col_info] + '; Information not restorable from header'
            return row
        region_info = name.split('::chr')[1].split('-mean_ratio')[0] # 4:155359187-155359457
        row[col_chr] = region_info.split(':')[0]
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'

    elif name.startswith('GC_DNase_positive:') or name.startswith('GC_DNase_negative_brain:') or name.startswith('GC_DNase_negative_blood:'):
        row[col_class] = 'element inactive control'
        region_info = name.split(':chr')[1].split('_active_count_')[0]
        row[col_chr] = region_info.split(':')[0]
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'
        # add info that coordinates of GRCh37 are used
        row[col_info] = row[col_info] + '; Coordinates are based on GRCh37'

    elif name.startswith('GC_DNase_positive_shuffeled:') or name.startswith('GC_DNase_negative_blood_shuffeled') or name.startswith('GC_DNase_negative_brain_shuffeled'):
        row[col_class] = 'element inactive control'
        region_info = name.split(':chr')[1].split('_active_count_')[0]
        row[col_category] = 'scrambled' # all sequences are scrambled
        row[col_chr] = region_info.split(':')[0]
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '.'
        # add info that coordinates of GRCh37 are used
        row[col_info] = row[col_info] + '; Coordinates are based on GRCh37'
    else:
        print(f'No special case for {name}')
    return row

In [ ]:
# control_meta_data_file
control_meta_data_file = meta_data_file.loc[~meta_data_file['name'].str.startswith('cardiac_neuro_cava_random')]
C_negative_heart_MK_test = control_meta_data_file.loc[control_meta_data_file['name'].str.startswith('C_negative_heart_MK')]
C_negative_neuron_NP_test = control_meta_data_file.loc[control_meta_data_file['name'].str.startswith('C_negative_neuron_NP')]
MK_test = control_meta_data_file.loc[control_meta_data_file['name'].str.startswith('MK')]
# C_negative_heart_MK_test.apply(get_start_end_strand_control, axis=1)
# C_negative_neuron_NP_test.apply(get_start_end_strand_control, axis=1)
MK_test.apply(get_start_end_strand_control, axis=1)

control_meta_data_file_with_coords = control_meta_data_file.apply(get_start_end_strand_control, axis=1)
control_meta_data_file_with_coords



nan
nan


,name,sequence,category,class,source,ref,chr,start,end,strand,...,tmp_open_NGN2_WTC11_NP,tmp_open_h1_neuro_dnase_screen,tmp_MPRA_primary_region_overlap,tmp_MPRA_organoid_region_overlap,tmp_gene_name,tmp_gene_set,tmp_is_neuro,tmp_is_cardiac,tmp_is_cava,tmp_is_random


In [ ]:
# write to file: files, creating, metadata_table_local_controls_august
control_meta_data_file_with_coords.columns
# drop columns starting with tmp_
columns_to_drop = [col for col in control_meta_data_file_with_coords.columns if col.startswith('tmp_')]
control_meta_data_file_with_coords.drop(columns=columns_to_drop, inplace=True)
print('shape: ', control_meta_data_file_with_coords.shape[0]) # 6275
control_meta_data_file_with_coords.head()
print('entries with coordinates: ', control_meta_data_file_with_coords[col_chr].notna().sum()) # 4229
# control_meta_data_file_with_coords.to_csv(config['files']['creating']['metadata_table_local_controls_august'], sep="\t", index=False)

shape:  0
entries with coordinates:  0


##### Variants: 
- Variant region map table: 

  | Number  |  Group |
  |---------|--------|
  |      49 |  C_positive_heart_CAD |
  |      23 |  GC_Atrial_fib |
  |     198 |  GC_Kircher    |
  |       8 |  GC_Liang      |
  |     174 |  GC_Mendelian_variants |
  |      20 |  GC_Mohlke     |
  |     198 |  GC_Selvarajan |
  |   46374 |  cardiac_neuro_cava_random |
- I can use variant region map for: 
  - C_positive_heart_CAD
  - GC_Atrial_fib
  - GC_Kircher
  - GC_Liang
  - GC_Mendelian_variants
  - GC_Mohlke from 20 variants in variant region map: 13 could be found (in design file: 20 ALT and 19 References)
  - GC_Selvarajan
  - cardiac_neuro_cava_random

  GC_Selvarajan           119
C_positive_heart_CAD     48
GC_Atrial_fib            17
GC_Mohlke                13
GC_Liang                  8

In [ ]:
# read variant region map
# load variant region map
variant_region_map = pd.read_csv(config['files']['final_design']['variant_table'], sep='\t', header=None)
variant_region_map.columns = ['ID', 'Region', 'REF_ID', 'ALT_ID']
# split (only controls)
control_variant_region_map = variant_region_map.loc[~variant_region_map['Variant'].str.startswith('cardiac_neuro_cava_random')]
control_variant_region_map
# sanity check if ALT_ID is unique:
print('Number of unique ids in ALT_ID: ', control_variant_region_map['ALT_ID'].nunique(), ' expected: ', control_variant_region_map.shape[0]) # 670



Number of unique ids in ALT_ID:  670  expected:  670


In [ ]:
control_alt_ids = control_variant_region_map['ALT_ID'].to_list()
control_ref_ids = list(control_variant_region_map['REF_ID'].unique())

len([ref for ref in control_ref_ids if 'Mohlke:REF_' in ref]) # 18 refs of mohlke in variant region map
mohlke_ref_variant_region_map = [ref for ref in control_ref_ids if 'Mohlke:REF_' in ref]

control_alt_ids
len([alt for alt in control_alt_ids if 'Mohlke:ALT_' in alt]) # 20 alts of mohlke in variant region map
mohlke_alt_variant_region_map = [alt for alt in control_alt_ids if 'Mohlke:ALT_' in alt]
# how many in the control metadata file:

control_meta_data_file[col_name].str.contains('Mohlke:ALT_').sum() # 17
control_meta_data_file[col_name].str.contains('Mohlke:REF_').sum() # 17
mohlke_ref_design = control_meta_data_file.loc[control_meta_data_file[col_name].str.contains('Mohlke:REF_')]['name'].to_list()
mohlke_alt_design = control_meta_data_file.loc[control_meta_data_file[col_name].str.contains('Mohlke:ALT_')]['name'].to_list()

missing_mohlke_refs = set(mohlke_ref_variant_region_map) - set(mohlke_ref_design)
missing_mohlke_alts = set(mohlke_alt_variant_region_map) - set(mohlke_alt_design)
missing_mohlke_refs
# {'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile1-3',
#  'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile2-3',
#  'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3',
#  'GC_Mohlke:REF_NC000003.12|186977424|G|T|MohlkeNonHepControl,NC000003.12|186977519|A|C|MohlkeNonHepControl_fwd_tile1-1',
#  'GC_Mohlke:REF_NC000010.11|100315721|G|A|MohlkeNonHepControl_fwd_tile1-1'}

missing_mohlke_alts
# {'GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile1-3_NC000001_11_230158967_C_A',
#  'GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile2-3_NC000001_11_230159168_C_T',
#  'GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_NC000001_11_230159168_C_T',
#  'GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_NC000001_11_230159329_CTTAAAGTGTTCAGCACTCCCCT_CT',
#  'GC_Mohlke:ALT_NC000003.12|186977424|G|T|MohlkeNonHepControl,NC000003.12|186977519|A|C|MohlkeNonHepControl_fwd_tile1-1_NC000003_12_186977424_G_T',
#  'GC_Mohlke:ALT_NC000003.12|186977424|G|T|MohlkeNonHepControl,NC000003.12|186977519|A|C|MohlkeNonHepControl_fwd_tile1-1_NC000003_12_186977519_A_C',
#  'GC_Mohlke:ALT_NC000010.11|100315721|G|A|MohlkeNonHepControl_fwd_tile1-1_NC000010_11_100315721_G_A'}

{'GC_Mohlke:ALT_NC000001.11|159752292|A|G|MohlkeHepControls_fwd_tile1-1_NC000001_11_159752292_A_G',
 'GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile1-3_NC000001_11_230158967_C_A',
 'GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile2-3_NC000001_11_230159168_C_T',
 'GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_NC000001_11_230159168_C_T',
 'GC_Mohlke:ALT_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_NC000001_11_230159329_CTTAAAGTGTTCAGCACTCCCCT_CT',
 'GC_Mohlke:ALT_NC000001.11|23016

In [ ]:
control_alt_ids = control_variant_region_map['ALT_ID'].to_list()
control_ref_ids = list(control_variant_region_map['REF_ID'].unique())

def get_control_allele(row):
    """
    Returns based on the header if it is a reference or alternative allele or a element (returns current value)
    assumes col_allele is already there (default: 'allele')
    """
    if row[col_name] in control_ref_ids:
        row[col_allele] = 'ref'
    elif row[col_name] in control_alt_ids:
        row[col_allele] = 'alt'
    return row

In [ ]:
control_meta_data_file.shape[0]

0

In [ ]:
# control_meta_data_file_test['allele'].value_counts()

In [ ]:
# define allele: ref and alt: if header in REF_ID then ref if header in ALT_ID then alt else leave NA


control_meta_data_file_test = control_meta_data_file.apply(get_control_allele, axis=1)
# Add region to the variants and chr, start, end, strand as for the variants
# split controls based on ref and alt
controls_refs = control_meta_data_file_test.loc[control_meta_data_file_test[col_allele] == 'ref']
controls_alts = control_meta_data_file_test.loc[control_meta_data_file_test[col_allele] == 'alt']
# get numbers of refs and alts:
print('Number of reference sequences: ', controls_refs.shape[0]) # 670
print('Number of alternative sequences: ', controls_alts.shape[0]) # 670
# add region with ALT_ID

# add region with REF_ID


# which information do we have about the variants? (=> variant position, SPDI)

Number of reference sequences:  0
Number of alternative sequences:  0


In [ ]:
controls_alts['tmp_label'].value_counts()

Series([], Name: tmp_label, dtype: int64)

In [ ]:
controls_refs['tmp_label'].value_counts()
controls_alts['tmp_label'].value_counts()

Series([], Name: tmp_label, dtype: int64)

In [ ]:
controls_refs

,name,sequence,category,class,source,ref,chr,start,end,strand,...,tmp_open_NGN2_WTC11_NP,tmp_open_h1_neuro_dnase_screen,tmp_MPRA_primary_region_overlap,tmp_MPRA_organoid_region_overlap,tmp_gene_name,tmp_gene_set,tmp_is_neuro,tmp_is_cardiac,tmp_is_cava,tmp_is_random


In [ ]:
controls_alts

,name,sequence,category,class,source,ref,chr,start,end,strand,...,tmp_open_NGN2_WTC11_NP,tmp_open_h1_neuro_dnase_screen,tmp_MPRA_primary_region_overlap,tmp_MPRA_organoid_region_overlap,tmp_gene_name,tmp_gene_set,tmp_is_neuro,tmp_is_cardiac,tmp_is_cava,tmp_is_random


In [ ]:
adfasdfef

NameError: name 'adfasdfef' is not defined

### Data for element analysis:
1. Use snakemake output of assigned barcodes and check the numbers
  - Filter for DNA and RNA counts >= 1 
2. Filter current metadata file: 
  - remove alt sequences
  - remove unused controls

In [86]:
# ! scp /data/cephfs-1/home/users/kisa11_c/unmirrored/projects/MPRA/IGVF_Y1_design/experiment/new_filtering_correct_umi_final_resequencing/results/experiments/standard_bwa/assigned_counts/assignmentFixDuplicates/standardConfig/NGN2_allreps_merged_barcode_assigned_counts.tsv.gz /home/kisa/coding/80K_MPRA/server_results/MPRAsnakeflow/

cp: cannot stat '/data/cephfs-1/home/users/kisa11_c/unmirrored/projects/MPRA/IGVF_Y1_design/experiment/new_filtering_correct_umi_final_resequencing/results/experiments/standard_bwa/assigned_counts/assignmentFixDuplicates/standardConfig/NGN2_allreps_merged_barcode_assigned_counts.tsv.gz': No such file or directory


In [7]:
snakemake_output = '/data/cephfs-1/home/users/kisa11_c/unmirrored/projects/MPRA/IGVF_Y1_design/experiment/final_resequencing/results/experiments/standard_bwa/assigned_counts/assignmentFixDuplicates/standardConfig/NGN2_allreps_merged_barcode_assigned_counts.tsv.gz'
snakemake_output = config['files']['creating']['mprasnakeflow_resequencing_default_bwa']

# this was computed locally
# snakemake_output = config['files']['creating']['mprasnakeflow_resequencing_default_bbmap_standard_mapq35_NoLength_notFiltered_RNA_DNA_count']
snakemake_output = config['files']['creating']['mprasnakeflow_resequencing_bbmap35_assigned_barcodes']
snakemake_output = config['files']['creating']['mprasnakeflow_resequencing_bbmap10_assigned_barcodes_unique_variants']

mpra_tested_assigned_barcodes_per_replicate = pd.read_csv(snakemake_output, sep='\t')


In [9]:
mpra_tested_assigned_barcodes_per_replicate.head()
mpra_tested_assigned_barcodes_per_replicate.head()['name'].to_list()


,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3
0,TCTACATGTCCCCAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,1.0,5.0,NaN,NaN,2.0,6.0
1,CGGCGGATGAGCCCT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,2.0,5.0,1.0,8.0,NaN,NaN
2,CGTTCTACCCGTTTA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,13.0,16.0,14.0,13.0,9.0,13.0
3,GGACAATGACCGCCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,4.0,20.0,6.0,9.0,4.0,17.0
4,TAGAACCCTAACTCA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,2.0,8.0,3.0,5.0


#### Remove alt sequences

In [49]:
variant_map.head()

,cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C,cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1,cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...


In [88]:
# takes 1min 40sec
# read variant map without header
variant_map = pd.read_csv(config['files']['final_design']['variant_table'], sep="\t", header=None)
variant_map.columns = ['ID', 'Region', 'REF_ID', 'ALT_ID']

variant_map_reference_sequences = variant_map['REF_ID'].to_list()
variant_map_alternative_sequences = variant_map['ALT_ID'].to_list()

def is_variant_ref_sequence(header):
    """Check if header is in the REF_ID of the used variant map"""
    # assuming REF_ID holds label information need to split header first
    return header in variant_map_reference_sequences


def is_variant_alt_sequence(header):
    """Check if header is in the ALT_ID of the used variant map"""
    # assuming ALT_ID holds label information need to split header first
    return header in variant_map_alternative_sequences

def is_reference_region_sequence(header):
    """Check if header defines a reference sequence in the experiment (if it is not describing a variant)"""
    return header not in variant_map_alternative_sequences

# make unique list of names
name_list = mpra_tested_assigned_barcodes_per_replicate['name'].unique().tolist() # 71663

# create a dataframe with the names
mpra_ref_alt_df = pd.DataFrame(name_list, columns=['name']) # 71663

# add column with is_alternative (true or false)
mpra_ref_alt_df['is_variant_alternative'] = mpra_ref_alt_df['name'].apply(is_variant_alt_sequence)
# add column with is_reference (ture or false)
mpra_ref_alt_df['is_variant_reference'] = mpra_ref_alt_df['name'].apply(is_variant_ref_sequence)
mpra_ref_alt_df['is_reference_region'] = mpra_ref_alt_df['name'].apply(is_reference_region_sequence)

# add tmp_label column
mpra_ref_alt_df['tmp_label'] = mpra_ref_alt_df['name'].apply(hf.get_label)
mpra_ref_alt_df

,name,is_variant_alternative,is_variant_reference,is_reference_region,tmp_label
0,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,False,False,True,C_SLEA
1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,False,False,True,C_SLEA
2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,False,False,True,C_SLEA
3,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,False,False,True,C_SLEA
4,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,False,False,True,C_SLEA
...,...,...,...,...,...
71610,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,False,False,True,cardiac_neuro_cava_random
71611,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,False,False,True,cardiac_neuro_cava_random
71612,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,False,False,True,cardiac_neuro_cava_random
71613,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,False,False,True,cardiac_neuro_cava_random


##### Investigate the control variants and a possible problem of the name matching between variant region map and the names at hand 
- checked by looking the not matchable headers (not reference / alternative) but contain REF or ALT in name 
  - found that the renamed variant region map has matching problems with control headers (, => ~ and * => >) (variant map => design file)
- for the bbmap experiment: /data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/experiment/bbmap_80K_test/80K_removed_brackets_design_no_duplicates_sequence_and_header.fa was used which has not been renamed for the controls but the variant region map has been renamed

- find the controls with the "*"
```bash
cat /data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_Y1_design/experiment/bbmap_80K_test/80K_removed_brackets_design_no_duplicates_sequence_and_header.fa | grep -v "cardiac_neuro_cava_random" |  grep ">" | \
awk '{ split($column, arr, ":")
    for (i=2; i<=length(arr); i++) {
        printf "%s%s", arr[i], (i<length(arr) ? ":" : "")
    }
    print ""
}' | grep "*" |less
```

In [89]:
# check how many controls are eighter reference or alternative
mpra_ref_alt_df_controls = mpra_ref_alt_df.loc[~mpra_ref_alt_df['name'].str.startswith('cardiac_neuro_cava_random')]
mpra_ref_alt_df_controls.groupby(['is_variant_reference', 'is_variant_alternative']).size()
# 188 variants within the control set (in variant region map are 670 variants)
mpra_ref_alt_df_controls_ref_alt = mpra_ref_alt_df_controls.loc[mpra_ref_alt_df_controls['is_variant_reference'] | mpra_ref_alt_df_controls['is_variant_alternative']]
# check the number of labels
mpra_ref_alt_df_controls_ref_alt['tmp_label'].value_counts()
# check if GC_Selvarajan contains "~" => surprisingly no GC_Selvarajan contains "~" (was already in the renaming script)
mpra_ref_alt_df_controls_ref_alt.loc[(mpra_ref_alt_df_controls_ref_alt['tmp_label'] == 'GC_Selvarajan') & mpra_ref_alt_df_controls_ref_alt['name'].str.contains('~')] # 0
# => surprisingly no GC_Selvarajan contains "~" (was already in the renaming script)

# investigate the set of names which are not reference and not alternative
controls_not_ref_alt = mpra_ref_alt_df_controls.loc[~mpra_ref_alt_df_controls['is_variant_reference'] & ~mpra_ref_alt_df_controls['is_variant_alternative']]

# find if some names contain ref or alt
controls_not_ref_alt.loc[controls_not_ref_alt['name'].str.contains('REF') | controls_not_ref_alt['name'].str.contains('ALT')]['name'].to_list() # 513
controls_not_ref_alt_but_contain_ref_alt = controls_not_ref_alt.loc[controls_not_ref_alt['name'].str.contains('REF') | controls_not_ref_alt['name'].str.contains('ALT')] # 513

# groups of names which are within the current version of the variant region map
# C_positive_heart_CAD
# GC_Atrial_fib
# GC_Kircher
# GC_Liang
# GC_Mendelian_variants
# GC_Mohlke
# GC_Selvarajan

# groups of names which are not considered reference or alternative but have ALT or REF in their name
controls_not_ref_alt_but_contain_ref_alt['tmp_label'].value_counts()

GC_Mendelian_variants    2
GC_Selvarajan            2
C_positive_heart_CAD     1
GC_Mohlke                1
Name: tmp_label, dtype: int64

In [90]:
# investigate the groups and if there is a problem in the matching of the headers
C_positive_heart_CAD = controls_not_ref_alt_but_contain_ref_alt.loc[controls_not_ref_alt_but_contain_ref_alt['tmp_label'] == 'C_positive_heart_CAD']
print(C_positive_heart_CAD)

GC_Mohlke = controls_not_ref_alt_but_contain_ref_alt.loc[controls_not_ref_alt_but_contain_ref_alt['tmp_label'] == 'GC_Mohlke']
GC_Mohlke['name'].to_list()

GC_Kircher = controls_not_ref_alt_but_contain_ref_alt.loc[controls_not_ref_alt_but_contain_ref_alt['tmp_label'] == 'GC_Kircher']
GC_Kircher['name'].to_list()

GC_Mendelian_variants = controls_not_ref_alt_but_contain_ref_alt.loc[controls_not_ref_alt_but_contain_ref_alt['tmp_label'] == 'GC_Mendelian_variants']
GC_Mendelian_variants['name'].to_list()

GC_Selvarajan = controls_not_ref_alt_but_contain_ref_alt.loc[controls_not_ref_alt_but_contain_ref_alt['tmp_label'] == 'GC_Selvarajan']
GC_Selvarajan['name'].to_list()

                                                name  is_variant_alternative  \
1376  C_positive_heart_CAD:ALT_rs67180937_rs67180937                   False   

      is_variant_reference  is_reference_region             tmp_label  
1376                 False                 True  C_positive_heart_CAD  


['GC_Selvarajan:ALT_rs2297787|STARR-seq-HepG2_fwd_tile1-1_rs2297787',
 'GC_Selvarajan:REF_rs754064|STARR-seq-HepG2_fwd_tile1-1']

In [ ]:
# change rename file script for all the control groups
# python /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/scripts/rename_file_2_design_style.py --input-file /home/kisa/coding/80K_MPRA/design_data/design_info/variant_region_map.tsv.gz --output-file /home/kisa/coding/80K_MPRA/design_data/design_info/renamed_variant_region_map.tsv.gz --file-type variant_map

##### Continue removing alt sequences

In [82]:
mpra_tested_assigned_barcodes_per_replicate.columns

Index(['Barcode', 'name', 'dna_count_1', 'rna_count_1', 'dna_count_2',
       'rna_count_2', 'dna_count_3', 'rna_count_3'],
      dtype='object')

In [94]:
mpra_ref_alt_df.columns

Index(['name', 'is_variant_alternative', 'is_variant_reference',
       'is_reference_region', 'tmp_label'],
      dtype='object')

In [96]:
# left join with the barcode association information
mpra_tested_assigned_barcodes_per_replicate_ref_alt = mpra_tested_assigned_barcodes_per_replicate.merge(mpra_ref_alt_df, on='name', how='left')

In [97]:
mpra_tested_assigned_barcodes_per_replicate_ref_alt.groupby(['is_reference_region'])['name'].size()
mpra_tested_assigned_barcodes_per_replicate_ref_alt.groupby(['is_variant_reference'])['name'].size()
mpra_tested_assigned_barcodes_per_replicate_ref_alt.groupby(['is_variant_alternative'])['name'].size()

is_variant_alternative
False    2671275
True     3551994
Name: name, dtype: int64

In [98]:
mpra_tested_assigned_barcodes_per_replicate_ref_alt.columns

Index(['Barcode', 'name', 'dna_count_1', 'rna_count_1', 'dna_count_2',
       'rna_count_2', 'dna_count_3', 'rna_count_3', 'is_variant_alternative',
       'is_variant_reference', 'is_reference_region', 'tmp_label'],
      dtype='object')

In [99]:
mpra_tested_assigned_barcodes_per_replicate_references = mpra_tested_assigned_barcodes_per_replicate_ref_alt.loc[mpra_tested_assigned_barcodes_per_replicate_ref_alt['is_reference_region']]
print(f"Number of reference sequences: {mpra_tested_assigned_barcodes_per_replicate_references['name'].nunique()}") # 17011 # with reference_region: 29780

Number of reference sequences: 29780


In [119]:
import pandas as pd
# Assuming your DataFrame is named 'mpra_tested_assigned_barcodes_per_replicate_ref_alt' and has a 'name' column
result = mpra_tested_assigned_barcodes_per_replicate_ref_alt.groupby('name').agg({
    'is_variant_reference': 'value_counts',
    'is_variant_alternative': 'value_counts',
    'is_reference_region': 'value_counts'
})

# Unstack the result for a more readable format
result = result.unstack(level=1, fill_value=0)

# Rename the columns for clarity
result.columns = [f'{col[0]}_{col[1]}' for col in result.columns]

# Add a column for the total count per name
result['total_count'] = mpra_tested_assigned_barcodes_per_replicate_ref_alt['name'].value_counts()


# focus on tested group
mpra_tested_assigned_barcodes_per_replicate_ref_alt_only_cardiac_neuro_cava_random = mpra_tested_assigned_barcodes_per_replicate_ref_alt.loc[mpra_tested_assigned_barcodes_per_replicate_ref_alt['name'].str.startswith('cardiac_neuro_cava_random')]

# Assuming your DataFrame is named 'mpra_tested_assigned_barcodes_per_replicate_ref_alt' and has a 'name' column
cardiac_neuro_cava_random_results = mpra_tested_assigned_barcodes_per_replicate_ref_alt_only_cardiac_neuro_cava_random.groupby('name').agg({
    'is_variant_reference': 'value_counts',
    'is_variant_alternative': 'value_counts',
    'is_reference_region': 'value_counts'
})

# Unstack the cardiac_neuro_cava_random_results for a more readable format
cardiac_neuro_cava_random_results = cardiac_neuro_cava_random_results.unstack(level=1, fill_value=0)

# Rename the columns for clarity
cardiac_neuro_cava_random_results.columns = [f'{col[0]}_{col[1]}' for col in cardiac_neuro_cava_random_results.columns]

# Add a column for the total count per name
cardiac_neuro_cava_random_results['total_count'] = mpra_tested_assigned_barcodes_per_replicate_ref_alt['name'].value_counts()


In [132]:
result
# result['is_reference_region_True'].notna().sum() # 29780
# result['is_variant_reference_True'].notna().sum() # 17011
result['is_variant_alternative_True'].notna().sum() # 41835

41835

In [134]:
# reference ratio assigned with barcodes:
17011/18882
# alternative ratio assigned with barcodes:
41835/47043

# number of designed names - unique variants in design
73940 - 46373 # only tested:
# 80141 - 47043 # all variants => # number of not references (not really useful)

27567

In [131]:
# tested_results:
cardiac_neuro_cava_random_results
cardiac_neuro_cava_random_results['is_reference_region_True'].notna().sum() # 25042
cardiac_neuro_cava_random_results['is_variant_reference_True'].notna().sum() # 17011
cardiac_neuro_cava_random_results['is_variant_alternative_True'].notna().sum() # 41265

41265

In [135]:
# variants: alternative tested:
41265/46373
# variants reference tested:
16726/18571
# # no alt seq
25042/27567

0.908404976965212

In [322]:
# # remove ALT sequences # takes 46 minutes for 6223425


# mpra_tested_assigned_barcodes_per_replicate_references = mpra_tested_assigned_barcodes_per_replicate.loc[~mpra_tested_assigned_barcodes_per_replicate['name'].apply(is_alt_sequence)]

# print(mpra_tested_assigned_barcodes_per_replicate.shape[0])
# # print number of sequences after apply
# print(mpra_tested_assigned_barcodes_per_replicate_references.shape[0]) # 80000

6223425
2701390


In [58]:
more_or_less_interesting_groups = ['cardiac_neuro_cava_random', 'GC_Atrial_fib', 'GC_Liang',
       'GC_Selvarajan', 'GC_Mohlke', 'GC_Kircher',
       'GC_Mendelian_variants', 'C_positive_heart_CAD', 'GC_Cort_Chengyu',
       'GC_GABA_Chengyu', 'GC_Glut_Chengyu', 'GC_Hon', 'GC_Vista',
       'GC_DNase_negative_blood', 'C_negative_heart_MK',
       'C_negative_neuron_MK', 'C_negative_neuron_NP',
       'C_positive_heart_MK', 'C_positive_neuron_CD',
       'C_positive_neuron_MK', 'C_positive_neuron_NP',
       'C_positive_heart_AB', 'C_SLEA', 'MK']
# remove DNAase groups (wrong coordinates)

In [61]:
def get_mk_scrambled_label(row):
    """
    Returns the label of the MK scrambled sequences
    """
    if (('MK:' in row[col_name]) and ('scramble_negative' in row[col_name])):
        return 'MK_scrambled'
    else:
        return row['tmp_label']


In [137]:
element_assigned_barcode_file.columns

Index(['Barcode', 'name', 'dna_count_1', 'rna_count_1', 'dna_count_2',
       'rna_count_2', 'dna_count_3', 'rna_count_3', 'is_variant_alternative',
       'is_variant_reference', 'is_reference_region', 'tmp_label'],
      dtype='object')

In [222]:
# filter based on tmp_label (more or less interesting groups)
element_assigned_barcode_file = mpra_tested_assigned_barcodes_per_replicate_references.loc[mpra_tested_assigned_barcodes_per_replicate_references['tmp_label'].isin(more_or_less_interesting_groups)]

# remove all alternative sequences
element_assigned_barcode_file = element_assigned_barcode_file.loc[element_assigned_barcode_file['is_reference_region']]

# add label for MK scrambled sequences
element_assigned_barcode_file['tmp_label'] = element_assigned_barcode_file.apply(get_mk_scrambled_label, axis=1)

# investigate the MK scrambled sequences
element_assigned_barcode_file.loc[element_assigned_barcode_file['tmp_label'] == 'MK_scrambled']

,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3,is_variant_alternative,is_variant_reference,is_reference_region,tmp_label
314200,AACGTATCCACGCGA,MK:tile_10128|chr12-111211746+111212015|scramb...,1.0,1.0,NaN,NaN,2.0,2.0,False,False,True,MK_scrambled
314201,GTAGAGTTCGTACTA,MK:tile_10128|chr12-111211746+111212015|scramb...,NaN,NaN,NaN,NaN,1.0,3.0,False,False,True,MK_scrambled
314202,AGATACATCCTTGCA,MK:tile_10128|chr12-111211746+111212015|scramb...,2.0,13.0,2.0,9.0,3.0,2.0,False,False,True,MK_scrambled
314203,AATCAAATATTAGAG,MK:tile_10128|chr12-111211746+111212015|scramb...,NaN,NaN,NaN,NaN,1.0,2.0,False,False,True,MK_scrambled
314204,AGTTGCCGAGCACCC,MK:tile_10128|chr12-111211746+111212015|scramb...,NaN,NaN,1.0,2.0,NaN,NaN,False,False,True,MK_scrambled
...,...,...,...,...,...,...,...,...,...,...,...,...
475413,GGTTGTTAACTTTTG,MK:tile_9944|chr12-98489717+98489986|scramble_...,3.0,20.0,4.0,20.0,2.0,23.0,False,False,True,MK_scrambled
475414,GCGGCGCCGAAACCG,MK:tile_9944|chr12-98489717+98489986|scramble_...,NaN,NaN,1.0,3.0,NaN,NaN,False,False,True,MK_scrambled
475415,GTCATTAATCAAGGC,MK:tile_9944|chr12-98489717+98489986|scramble_...,3.0,11.0,6.0,11.0,2.0,4.0,False,False,True,MK_scrambled
475416,CATTCATCCGAACTG,MK:tile_9944|chr12-98489717+98489986|scramble_...,1.0,8.0,1.0,3.0,2.0,7.0,False,False,True,MK_scrambled


In [225]:
# check the number of sequences in the element_assigned_barcode_file (and number when focused on cardiac_neuro_cava_random)
element_assigned_barcode_file.name.nunique() # 80000
element_assigned_barcode_file.loc[element_assigned_barcode_file['name'].str.startswith('cardiac_neuro_cava_random')].name.nunique() # 18882

25042

In [223]:
element_assigned_barcode_file.tmp_label.unique()

array(['C_SLEA', 'C_negative_heart_MK', 'C_negative_neuron_MK',
       'C_negative_neuron_NP', 'C_positive_heart_AB',
       'C_positive_heart_CAD', 'C_positive_heart_MK',
       'C_positive_neuron_CD', 'C_positive_neuron_MK',
       'C_positive_neuron_NP', 'GC_Atrial_fib', 'GC_Cort_Chengyu',
       'GC_DNase_negative_blood', 'GC_GABA_Chengyu', 'GC_Glut_Chengyu',
       'GC_Hon', 'GC_Kircher', 'GC_Liang', 'GC_Mendelian_variants',
       'GC_Mohlke', 'GC_Selvarajan', 'GC_Vista', 'MK', 'MK_scrambled',
       'cardiac_neuro_cava_random'], dtype=object)

In [226]:
# remove unnecessary columns
element_assigned_barcode_file.drop(columns=['is_variant_reference', 'is_variant_alternative', 'is_reference_region'], inplace=True)

# write output to file
# element_assigned_barcode_file.to_csv(config['files']['creating']['element_assigned_barcode_file'], sep='\t', index=False, compression='gzip')

# write label file:
label_file = element_assigned_barcode_file[['name', 'tmp_label']].drop_duplicates()
# label_file.to_csv(config['files']['creating']['element_analysis_label_file'], sep='\t', index=False)

In [227]:
element_assigned_barcode_file
element_assigned_barcode_file['name'].nunique() # 29641

29641

##### Filter barcode counts and dna rna counts: 

In [229]:
# Threshold value
dna_count_threshold = 1

# Select columns that match the pattern "DNA" (case-insensitive)
pattern = "dna"
matching_columns = element_assigned_barcode_file.filter(regex=pattern, axis=1).columns

# Apply condition to all matching columns
filtered_df = element_assigned_barcode_file.loc[(element_assigned_barcode_file[matching_columns] >= dna_count_threshold).all(axis=1)]

# Select columns that match the pattern "RNA" (case-insensitive)
pattern = "rna"
matching_columns = filtered_df.filter(regex=pattern, axis=1).columns

# Apply condition to all matching columns
filtered_df = filtered_df.loc[(filtered_df[matching_columns] >= dna_count_threshold).all(axis=1)]


# Print the result
print(filtered_df) # 2304249 => 1576190 | now: 1807313 #oligo 29182

filtered_df['name'].nunique() # 28573 => 28010 => 563 oligos removed

                 Barcode                                               name  \
0        GGCCTCTTCGGTCAG  C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...   
3        ATATCAAGACGCGAT  C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...   
4        TAAATATCATAAGAT  C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...   
5        TGTGTGCACGGTGAG  C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...   
7        CAGAGTAGCCATGAT  C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...   
...                  ...                                                ...   
6223263  GCGCTCGTGCGACCC  cardiac_neuro_cava_random:ZNF462|ENSG000001481...   
6223264  TTTCGACCGGATGCG  cardiac_neuro_cava_random:ZNF462|ENSG000001481...   
6223266  TGAGGCTTGTTGGGC  cardiac_neuro_cava_random:ZNF462|ENSG000001481...   
6223267  TATTGTCAAGCGGGA  cardiac_neuro_cava_random:ZNF462|ENSG000001481...   
6223268  AGTTAGGTTCGCGGA  cardiac_neuro_cava_random:ZNF462|ENSG000001481...   

         dna_count_1  rna_count_1  dna_count_2  rna

29182

In [230]:
bc_threshold = 10

In [231]:
# Group by 'name' and count the number of barcodes for each name
barcode_counts = filtered_df.groupby('name')['Barcode'].count().reset_index()

# Display the barcode counts
print("\nBarcode counts for each name:")
print(barcode_counts)

# Filter out names with fewer than 10 barcodes
names_to_keep = barcode_counts.loc[barcode_counts['Barcode'] >= bc_threshold].name.to_list()

# Filter the original DataFrame to keep only these names
filtered_element_assigned_barcode_file = filtered_df[filtered_df['name'].isin(names_to_keep)]


Barcode counts for each name:
                                                    name  Barcode
0      C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...      103
1      C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...       97
2      C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...       74
3      C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...       98
4      C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...       56
...                                                  ...      ...
29177  cardiac_neuro_cava_random:ZNF462|ENSG000001481...       12
29178  cardiac_neuro_cava_random:ZNF462|ENSG000001481...       73
29179  cardiac_neuro_cava_random:ZNF462|ENSG000001481...       22
29180  cardiac_neuro_cava_random:ZNF462|ENSG000001481...       22
29181  cardiac_neuro_cava_random:ZNF462|ENSG000001481...      108

[29182 rows x 2 columns]


In [232]:
filtered_element_assigned_barcode_file.shape[0] # 2304249 # bbmapq35 no_collisions: 1802259
filtered_element_assigned_barcode_file['name'].nunique() # 29641 # 28527

28527

In [234]:
filtered_element_assigned_barcode_file.tmp_label.unique()

array(['C_SLEA', 'C_negative_heart_MK', 'C_negative_neuron_MK',
       'C_negative_neuron_NP', 'C_positive_heart_AB',
       'C_positive_heart_CAD', 'C_positive_heart_MK',
       'C_positive_neuron_CD', 'C_positive_neuron_MK',
       'C_positive_neuron_NP', 'GC_Atrial_fib', 'GC_Cort_Chengyu',
       'GC_DNase_negative_blood', 'GC_GABA_Chengyu', 'GC_Glut_Chengyu',
       'GC_Hon', 'GC_Kircher', 'GC_Liang', 'GC_Mendelian_variants',
       'GC_Mohlke', 'GC_Selvarajan', 'GC_Vista', 'MK', 'MK_scrambled',
       'cardiac_neuro_cava_random'], dtype=object)

In [233]:
filtered_element_assigned_barcode_file.columns

Index(['Barcode', 'name', 'dna_count_1', 'rna_count_1', 'dna_count_2',
       'rna_count_2', 'dna_count_3', 'rna_count_3', 'tmp_label'],
      dtype='object')

In [235]:
# write filtered tsv of associated barcodes:
# write output to file
# filtered_element_assigned_barcode_file.to_csv(config['files']['creating']['filtered_element_assigned_barcode_file'], sep='\t', index=False, compression='gzip')

# write label file:
filtered_label_file = filtered_element_assigned_barcode_file[['name', 'tmp_label']].drop_duplicates()
# filtered_label_file.to_csv(config['files']['creating']['filtered_element_analysis_label_file'], sep='\t', index=False)

##### Only positive_neuron_NP and tested elements

In [236]:
testable_groups = ['cardiac_neuro_cava_random', 'C_positive_neuron_NP']
testable_assigned_barcodes_file = filtered_element_assigned_barcode_file.loc[filtered_element_assigned_barcode_file['tmp_label'].isin(testable_groups)]
print(f"Groups in element table: {testable_assigned_barcodes_file['tmp_label'].unique()}")

# print number of sequence and barcode combinations
print(f"Number of barcodes: {testable_assigned_barcodes_file.shape[0]}") # 1954771
print(f"Number of sequences: {testable_assigned_barcodes_file['name'].nunique()}") # 24310

# write to file
# testable_assigned_barcodes_file.to_csv(config['files']['creating']['element_assigned_barcode_file_positive_neuron_np_vs_tested'], sep='\t', index=False, compression='gzip')


testable_assigned_barcodes_file.groupby('tmp_label')['name'].nunique()
#old without filtering:
# 63
# 24247

# new with wrong filtering:
# Number of barcodes: 1523362
# Number of sequences: 24757
# tmp_label
# C_positive_neuron_NP            70
# cardiac_neuro_cava_random    24687


# # with correct filtering:
# Groups in element table: ['C_positive_neuron_NP' 'cardiac_neuro_cava_random']
# Number of barcodes: 1519201
# Number of sequences: 24217

# tmp_label
# C_positive_neuron_NP            68
# cardiac_neuro_cava_random    24149


Groups in element table: ['C_positive_neuron_NP' 'cardiac_neuro_cava_random']
Number of barcodes: 1519201
Number of sequences: 24217


tmp_label
C_positive_neuron_NP            68
cardiac_neuro_cava_random    24149
Name: name, dtype: int64

##### Only negative_neuron_NP and tested elements

In [208]:
filtered_element_assigned_barcode_file['tmp_label'].unique()

array(['C_SLEA', 'C_negative_heart_MK', 'C_negative_neuron_MK',
       'C_negative_neuron_NP', 'C_positive_heart_AB',
       'C_positive_heart_CAD', 'C_positive_heart_MK',
       'C_positive_neuron_CD', 'C_positive_neuron_MK',
       'C_positive_neuron_NP', 'GC_Atrial_fib', 'GC_Cort_Chengyu',
       'GC_DNase_negative_blood', 'GC_GABA_Chengyu', 'GC_Glut_Chengyu',
       'GC_Hon', 'GC_Kircher', 'GC_Liang', 'GC_Mendelian_variants',
       'GC_Mohlke', 'GC_Selvarajan', 'GC_Vista', 'MK', 'MK_scrambled',
       'cardiac_neuro_cava_random'], dtype=object)

In [207]:
testable_groups = ['cardiac_neuro_cava_random', 'C_negative_neuron_NP']
testable_assigned_barcodes_file = filtered_element_assigned_barcode_file.loc[filtered_element_assigned_barcode_file['tmp_label'].isin(testable_groups)]
print(f"Groups in element table: {testable_assigned_barcodes_file['tmp_label'].unique()}")

# print number of sequence and barcode combinations
print(f"Number of barcodes: {testable_assigned_barcodes_file.shape[0]}") # 1959311
print(f"Number of sequences: {testable_assigned_barcodes_file['name'].nunique()}") # 24368

# # write to file
# testable_assigned_barcodes_file.to_csv(config['files']['creating']['element_assigned_barcode_file_negative_neuron_np_vs_tested'], sep='\t', index=False, compression='gzip')


testable_assigned_barcodes_file.groupby('tmp_label')['name'].nunique()
# 121
# 24247

# new without filtering
# Groups in element table: ['C_negative_neuron_NP' 'cardiac_neuro_cava_random']
# Number of barcodes: 2251492
# Number of sequences: 25176

# tmp_label
# C_negative_neuron_NP           134
# cardiac_neuro_cava_random    25042

# new with first barcode then rna + dna filtering (wrong filtering)
# Number of barcodes: 1523363
# Number of sequences: 24811
# tmp_label
# C_negative_neuron_NP           124
# cardiac_neuro_cava_random    24687

# new with correct filtering:
# Number of barcodes: 1518991
# Number of sequences: 24240
# tmp_label
# C_negative_neuron_NP            91
# cardiac_neuro_cava_random    24149


Groups in element table: ['C_negative_neuron_NP' 'cardiac_neuro_cava_random']
Number of barcodes: 1518991
Number of sequences: 24240


tmp_label
C_negative_neuron_NP            91
cardiac_neuro_cava_random    24149
Name: name, dtype: int64

In [237]:
filtered_element_assigned_barcode_file.tmp_label.unique()

array(['C_SLEA', 'C_negative_heart_MK', 'C_negative_neuron_MK',
       'C_negative_neuron_NP', 'C_positive_heart_AB',
       'C_positive_heart_CAD', 'C_positive_heart_MK',
       'C_positive_neuron_CD', 'C_positive_neuron_MK',
       'C_positive_neuron_NP', 'GC_Atrial_fib', 'GC_Cort_Chengyu',
       'GC_DNase_negative_blood', 'GC_GABA_Chengyu', 'GC_Glut_Chengyu',
       'GC_Hon', 'GC_Kircher', 'GC_Liang', 'GC_Mendelian_variants',
       'GC_Mohlke', 'GC_Selvarajan', 'GC_Vista', 'MK', 'MK_scrambled',
       'cardiac_neuro_cava_random'], dtype=object)

In [242]:
testable_groups = ['cardiac_neuro_cava_random', 'C_negative_neuron_NP', 'C_negative_neuron_MK']
testable_assigned_barcodes_file = filtered_element_assigned_barcode_file.loc[filtered_element_assigned_barcode_file['tmp_label'].isin(testable_groups)]
print(f"Groups in element table: {testable_assigned_barcodes_file['tmp_label'].unique()}")

def add_control_label(label):
    """
    Change the label of the negative control groups to one label: C_combined_negative_neuron_MK_NP
    """
    if label != "cardiac_neuro_cava_random":
        return "C_combined_negative_neuron_MK_NP"
    return label

# print number of sequence and barcode combinations
print(f"Number of barcodes: {testable_assigned_barcodes_file.shape[0]}") # 1959311
print(f"Number of sequences: {testable_assigned_barcodes_file['name'].nunique()}") # 24368
testable_assigned_barcodes_file['tmp_label'] = testable_assigned_barcodes_file['tmp_label'].apply(add_control_label)

# # write to file
# testable_assigned_barcodes_file.to_csv(config['files']['creating']['element_assigned_barcode_file_negative_neuron_ctrls_vs_tested'], sep='\t', index=False, compression='gzip')
# write label file:
testable_assigned_barcodes_label_file = testable_assigned_barcodes_file[['name', 'tmp_label']].drop_duplicates()
# testable_assigned_barcodes_label_file.to_csv(config['files']['creating']['element_assigned_barcode_file_negative_neuron_ctrls_vs_tested_label_file'], sep='\t', index=False)


testable_assigned_barcodes_file.groupby('tmp_label')['name'].nunique()
testable_assigned_barcodes_label_file.groupby('tmp_label')['name'].nunique()

# Number of barcodes: 1526366
# Number of sequences: 24409
# tmp_label
# C_negative_neuron_MK           169
# C_negative_neuron_NP            91
# cardiac_neuro_cava_random    24149

Groups in element table: ['C_negative_neuron_MK' 'C_negative_neuron_NP' 'cardiac_neuro_cava_random']
Number of barcodes: 1526366
Number of sequences: 24409


/tmp/ipykernel_131506/1294392145.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  testable_assigned_barcodes_file['tmp_label'] = testable_assigned_barcodes_file['tmp_label'].apply(add_control_label)


tmp_label
C_combined_negative_neuron_MK_NP      260
cardiac_neuro_cava_random           24149
Name: name, dtype: int64

In [170]:
# check differences to the filtering done in R: /home/kisa/coding/80K_MPRA/element_analysis_output/negative_neuron_NP_vs_tested_element_assigned_barcodes_no_alt_filtered.tsv

# load file
element_assigned_barcodes_file_no_alt_filtered = pd.read_csv('/home/kisa/coding/80K_MPRA/element_analysis_output/negative_neuron_NP_vs_tested_element_assigned_barcodes_no_alt_filtered.tsv', sep='\t')
element_assigned_barcodes_file_no_alt_filtered

,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3,tmp_label
0,CAAGTTTGCCTGACG,C_negative_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,5,3,6,5,7,3,C_negative_neuron_NP
1,ACCCCCTAGACCCTC,C_negative_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,5,3,3,1,1,3,C_negative_neuron_NP
2,CATATACGAAATTAA,C_negative_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,2,3,6,1,4,4,C_negative_neuron_NP
3,CCACTTGCTATCCGT,C_negative_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,3,3,4,2,1,2,C_negative_neuron_NP
4,AGAGTCCAAGCACAC,C_negative_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,3,1,2,3,2,1,C_negative_neuron_NP
...,...,...,...,...,...,...,...,...,...
1518986,GCGCTCGTGCGACCC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,4,20,7,12,5,11,cardiac_neuro_cava_random
1518987,TTTCGACCGGATGCG,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,3,11,3,10,3,9,cardiac_neuro_cava_random
1518988,TGAGGCTTGTTGGGC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,3,9,3,4,2,7,cardiac_neuro_cava_random
1518989,TATTGTCAAGCGGGA,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,1,16,2,12,3,7,cardiac_neuro_cava_random


In [172]:
testable_assigned_barcodes_file.head()

,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3,tmp_label
42546,CGTATGTCTCGTATG,C_negative_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,2.0,2.0,5.0,4.0,2.0,2.0,C_negative_neuron_NP
42550,GTCTCATATCGCGGA,C_negative_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,3.0,2.0,1.0,4.0,1.0,3.0,C_negative_neuron_NP
42552,AGTCCGGTTGTTTAA,C_negative_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,1.0,2.0,1.0,1.0,2.0,1.0,C_negative_neuron_NP
42559,GGAGATTGCGTTTAG,C_negative_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,6.0,1.0,1.0,2.0,1.0,2.0,C_negative_neuron_NP
42577,TGCTTGAAGTGGGCA,C_negative_neuron_NP:Fetal_Cerebrum_Cicero_GW1...,3.0,4.0,3.0,4.0,4.0,5.0,C_negative_neuron_NP


In [173]:
# make set of the name column
r_filtered_name_set = set(element_assigned_barcodes_file_no_alt_filtered['name'].to_list())

# check which names are in the testable_assigned_barcodes_file
python_filtered_name_set = set(testable_assigned_barcodes_file['name'].to_list())

In [183]:
# print the size of the sets and their differences with an upset plot
print(f"Number of names in R filtered file: {len(r_filtered_name_set)}")
print(f"Number of names in Python filtered file: {len(python_filtered_name_set)}")

# check the difference
print(f"Number of names in R filtered file but not in Python filtered file: {len(r_filtered_name_set - python_filtered_name_set)}")
print(f"Number of names in Python filtered file but not in R filtered file: {len(python_filtered_name_set - r_filtered_name_set)}")

# get the names of the 571 names which are only in the filtering of python and show their data
names_only_in_python = python_filtered_name_set - r_filtered_name_set
filtered_by_R = testable_assigned_barcodes_file.loc[testable_assigned_barcodes_file['name'].isin(names_only_in_python)]

filtered_by_R.groupby('name').size().min()

Number of names in R filtered file: 24240
Number of names in Python filtered file: 24811
Number of names in R filtered file but not in Python filtered file: 0
Number of names in Python filtered file but not in R filtered file: 571


1

##### Only MK_scrambled and tested elements

In [158]:
testable_groups = ['cardiac_neuro_cava_random', 'MK_scrambled']
testable_assigned_barcodes_file = element_assigned_barcode_file.loc[element_assigned_barcode_file['tmp_label'].isin(testable_groups)]

In [159]:
testable_assigned_barcodes_file['tmp_label'].unique()

array(['MK_scrambled', 'cardiac_neuro_cava_random'], dtype=object)

In [160]:
testable_assigned_barcodes_file.shape[0] # 19xx now: 2290567


2290567

In [161]:
testable_assigned_barcodes_file['name'].nunique() # old: 24721 now: 25532

25532

In [162]:
# group by tmp_label and count the number of unique name
testable_assigned_barcodes_file.groupby('tmp_label')['name'].nunique()
# MK_scrambled                   474
# cardiac_neuro_cava_random    24247

# tmp_label
# MK_scrambled                   490
# cardiac_neuro_cava_random    25042

tmp_label
MK_scrambled                   490
cardiac_neuro_cava_random    25042
Name: name, dtype: int64

In [163]:
# write output to file
# testable_assigned_barcodes_file.to_csv(config['files']['creating']['element_assigned_barcode_file_scrambled_vs_tested'], sep='\t', index=False, compression='gzip')


In [ ]:
asdfasdfasf

NameError: name 'asdfasdfasf' is not defined

: 

##### not used here anymore: put it further up Collision handling: 
- read the data: `/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/experiment/correct_umi_final_resequencing/design_check_collisions.err`
- if 'negative_neuron_NP' in row: remove the elements which are not this element; if 'REF' in row remove the others, elif 'reference' in row: take it elif take first elmenent: 

In [ ]:
# list of headers to be removed:
# 1,2: collision with reference
# 3: collision with C_positive_neuron_NP
# 4, 5, 6: same group (GC_Mendelian_variants): remove second
# 7: collision with reference (MK)
# 8: same group (GC_Mendelian_variants): remove second
# 9, 10: same group (MK): remove second
# 11, 12: same group (GC_Mendelian_variants), both references: remove second
# 13: same group (MK), both references: remove second
# 14: collision with C_negaive_neuron_NP
# 15: MK variants: removed second
# 16: GC_Mendelian_variants: both reference: removed second
# 17: MK: both references: removed second
# 18: collision with G_postive_neuron_NP: removed MK reference
# 19, 20: same group (GC_Mendelian_variants): remove second
# 21, 22, 23: same group (MK): remove second
# 24: both references (MK): removed second
# 25: GC_Selvarajan collision with cardiac_neuro_cava_random: removed GC_Selvarajan
# 26, 27: same group (GC_Mendelian_variants): remove second
# 28, 29: collision of C_positive_neuron_MK of 1 reference with 2 variants: removed variants
# 30: collision MK reference with variant: removed variant
# 31: collision MK variant and reference: removed variant
# 32: collision GC_Mendelian_variants: removed second
# 33: collision GC_Mendelian_variants: removed second
# 34: collision MK: removed second
# 35: collision GC_Mendelian_variants: removed second
# 35: collision GC_Mendelian_variants: removed second
# 36, 37: collision: MK removed variant
# 38, 39, 40: collision: GC_Mendelian_variants removed second
# 41: collision C_negative_neuron_MK: GC_Vista removed
# 42, 43: collision: GC_Mendelian_variants removed second
# 44: collision: MK removed second
# 45: collision: MK removed variant
# 46: collision: MK removed second
# 47: collision: GC_Mendelian_variants removed second
# 48: collision: MK removed variant
# 49, 50: collision: GC_Mendelian_variants removed second
# 51: collision MK reference and C_positive_neuron_MK: removed MK reference
# 52: collision MK: removed second
# 53: collision GC_salvarajan and cardiac_neuro_cava_random: removed GC_salvarajan
# 54, 55: collision: GC_Mendelian_variants removed second
# 56: collision: MK newcore vs tile: removed tile
# 57: collision MK: removed second
# 58: collision C_positive_neuron_NP vs MK newcore reference: removed MK newcore reference
# 59: collisoin MK: removed second
# 60: collision GC_Mendelian_variants: removed second
# 61: important: collision: Forward collision 59 for sequences:     cardiac_neuro_cava_random:REF_CELF2|ENSG00000048740.19|EH38E1447739_fwd_tile1-1 C_negative_neuron_NP:Fetal_Cerebrum_Cicero_chr10_11131297_11131567_2.37359878574768: removed C_negative_neuron_NP
# 62, 63: collision (MK): reference with two variants
# 64: collision (GC_Mendelian_variants): removed second
# UNTIL: Forward collision 62 for sequences:
# 65, 66: collision (GC_Mendelian_variants): removed second
# 67: collision (vista vs cardiac_neuro_cava_random): removed vista
# 68: collision (GC_Mendelian_variants) two references: removed second
# 69: collision (MK) two references: removed second
# 70: collision (MK) two variants vs reference: removed variants
# 71: collision (GC_Mendelian_variants): removed second
# 72: collision (MK): removed second
# 73: collision (GC_Mendelian_variants): removed second

removable_header_bc_of_collisions = ['MK:tile_19098|chr18-25465452+25465722|G-A-1', 'MK:tile_19098|chr18-25465452+25465722|A-G-268', 'MK:tile_37449|chr6-98703408+98703677|reference', 'GC_Mendelian_variants:ALT_chr7:156791255G*C|SHH_chr7:156791274T*TTAAGGAAGTGATT|SHH', 'GC_Mendelian_variants:ALT_chr7:156791581A*G|SHH_chr7:156791579C*T|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791542A*C|SHH', 'MK:tile_47638|chr16-52435608+52435878|A-T-1', 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791581A*G|SHH', 'MK:tile_47629|chr14-103542886+103543156|C-G-33', 'MK:tile_47742|chr5-87944920+87945190|G-C-165', 'GC_Mendelian_variants:REF_chr7:156791472C*G|SHH',
 'GC_Mendelian_variants:REF_chr7:156791579C*T|SHH', 'MK:tile_47629|chr14-103542886+103543156|reference', 'GC_Vista:fb;fm_mm1912_vistaElementControl|chr7:42140681-42140930', 'MK:tile_47627|chr14-103542806+103543076|T-A-248', 'GC_Mendelian_variants:REF_chr10:23219434A*G|PTF1A', 'MK:tile_47626|chr14-103542805+103543075|reference',
 'MK:tile_37403|chr6-98416848+98417117|reference', 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791459T*C|SHH', 'GC_Mendelian_variants:ALT_chr7:156791255G*C|SHH_chr7:156791255G*C|SHH', 'MK:tile_47744|chr5-87945000+87945270|T-A-62', 'MK:tile_47743|chr5-87944921+87945191|T-A-141', 'MK:tile_47626|chr14-103542805+103543075|C-G-114',
 'MK:tile_47745|chr5-87945001+87945271|reference', 'GC_Selvarajan:ALT_rs499966|STARR-seq-HepG2_fwd_tile1-1_rs499966', 'GC_Mendelian_variants:ALT_chr7:156791474G*A|SHH_chr7:156791571T*A|SHH', 'GC_Mendelian_variants:ALT_chr10:23219436A*G|PTF1A_chr10:23219434A*G|PTF1A',
 'C_positive_neuron_MK:tile_34824_chr5_141041068_141041337_A_T_1_0.562204129450572', 'C_positive_neuron_MK:tile_34824_chr5_141041068_141041337_G_C_269_0.550310056665328', 'MK:tile_47785|chr6-164344756+164345026|G-C-269', 'MK:tile_47725|chr4-125519552+125519822|T-A-269', 'GC_Mendelian_variants:ALT_chr10:23219434A*G|PTF1A_chr10:23219517A*C|PTF1A',
 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791547A*G|SHH', 'MK:tile_47629|chr14-103542886+103543156|T-A-168', 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791472C*G|SHH', 'GC_Mendelian_variants:ALT_chr10:23219434A*G|PTF1A_chr10:23219376A*C|PTF1A', 'MK:tile_47562|chr1-52663056+52663326|C-G-1', 'MK:tile_47791|chr6-170438656+170438926|G-C-1',
 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791480G*A|SHH', 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791472C*T|SHH', 'GC_Mendelian_variants:ALT_chr7:156791474G*A|SHH_chr7:156791542A*C|SHH', 'GC_Vista:fb_hs262_vistaElementControl|chr5:77645014-77645263', 'GC_Mendelian_variants:ALT_chr7:156791474G*A|SHH_chr7:156791581A*G|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791255G*C|SHH_chr7:156791257G*A|SHH', 'MK:tile_47744|chr5-87945000+87945270|G-C-265', 'MK:tile_47747|chr5-87945240+87945510|T-A-1', 'MK:tile_47627|chr14-103542806+103543076|C-G-129', 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791474G*A|SHH', 'MK:tile_47598|chr12-102961789+102962059|A-T-1',
 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791472C*T|SHH', 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791579C*T|SHH', 'MK:rdhs_334645|chr2-6772346+6772615|reference', 'MK:tile_47744|chr5-87945000+87945270|T-A-241', 'GC_Selvarajan:REF_rs499966|STARR-seq-HepG2_fwd_tile1-1', 'GC_Mendelian_variants:ALT_chr7:156791474G*A|SHH_chr7:156791413A*C|SHH',
 'GC_Mendelian_variants:ALT_chr7:156791581A*G|SHH_chr7:156791472C*G|SHH', 'MK:tile_44725|chr8-127646130+127646399|reference', 'MK:tile_47629|chr14-103542886+103543156|C-G-49', 'MK:newcore_405894|chrX-71182006+71182275|reference', 'GC_Mendelian_variants:ALT_chr7:156791474G*A|SHH_chr7:156791547A*G|SHH', 'C_negative_neuron_NP:Fetal_Cerebrum_Cicero_chr10_11131297_11131567_2.37359878574768',
 'MK:tile_985|chr1-33363965+33364235|C-T-268', 'MK:tile_985|chr1-33363965+33364235|T-C-1', 'GC_Mendelian_variants:ALT_chr10:23219436A*G|PTF1A_chr10:23219436A*G|PTF1A', 'GC_Mendelian_variants:ALT_chr7:156791579C*T|SHH_chr7:156791571T*A|SHH', 'GC_Mendelian_variants:ALT_chr7:156791472C*G|SHH_chr7:156791474G*A|SHH', 'GC_Vista:fb;mb__vistaElementControl|chr14:78308172-78308421',
 'GC_Mendelian_variants:REF_chr7:156791255G*C|SHH', 'MK:tile_47743|chr5-87944921+87945191|reference', 'MK:tile_10173|chr12-113932852+113933122|C-T-268', 'MK:tile_10173|chr12-113932852+113933122|G-A-1', 'GC_Mendelian_variants:ALT_chr10:23219434A*G|PTF1A_chr10:23219508A*G|PTF1A', 'MK:tile_47744|chr5-87945000+87945270|G-C-85', 'GC_Mendelian_variants:ALT_chr7:156791581A*G|SHH_chr7:156791480G*A|SHH'
]

In [ ]:
print(len(removable_header_bc_of_collisions))
print(len(set(removable_header_bc_of_collisions))) # all unique => sanity check

74
74


In [ ]:
# read design file:
# design_file_with_collisions = hf.fasta_to_dataframe('/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/removed_brackets_design_no_duplicates_sequence_and_header.fa')
# # remove these headers:
# design_file_without_collisions = design_file_with_collisions.loc[~design_file_with_collisions['header'].isin(removable_header_bc_of_collisions)]
# design_file_of_collisions = design_file_with_collisions.loc[design_file_with_collisions['header'].isin(removable_header_bc_of_collisions)]

# # write both files:
# hf.write_fasta(design_file_without_collisions, '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header_with_adapter_no_brackets_no_collisions.fa')
# hf.write_fasta(design_file_of_collisions, '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header_with_adapter_no_brackets_collisions.fa')

True

In [ ]:
design_file_without_collisions

,header,sequence
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AAGAATACAAGTAACTGATGAATGAAGGGGGCATCTTGTGTCCCCA...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,TTGGGTATGCTGCCCCCCAGCTGGCGGGGCACCGGGGACAGGCACA...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,ACGAGCAAGGGAATGAGAGAGAGTGGGTTAGAGAGTGAGTGAGCCA...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,CGTGGACACGCGTGATTGACCCTTTAACTGTATCCTTAACCACCGC...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,CCGGAGAGTCTCAGCTCCCGCAGCCCTAACAAACGACCACAGACCT...
...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,CTTAATCAAATAACCCATTAATTCTATATATCTACCTAATATTAAT...
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,CATCGGCCCTGGTGAAGCGTCCGTCCAGACGGGCCTGCCTAGCCTC...
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,TAAATATTCAGCGATACATTCCTATTCTTTTTCAGAAGTAGTTATT...
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,TGAAGCCCCTGATTCTGTTAGAATAAGGTTACTGAGTCGTGTATAC...


In [ ]:
design_file_of_collisions

,header,sequence
74049,GC_Selvarajan:REF_rs499966|STARR-seq-HepG2_fwd...,AGAACCGTCTCCATGTGGAGAAGATCCTTTCTCCCTAATTAGCCTT...
74231,GC_Selvarajan:ALT_rs499966|STARR-seq-HepG2_fwd...,AGAACCGTCTCCATGTGGAGAAGATCCTTTCTCCCTAATTAGCCTT...
74605,GC_Mendelian_variants:REF_chr10:23219434A*G|PTF1A,CACTTAAAGAGTCACTGTTACTTTGAGGTTTTATCTGTAAGATTCG...
74623,GC_Mendelian_variants:REF_chr7:156791255G*C|SHH,GAGATATGGCTTCATTTTCTGTAATAAACACTAAGATCAAAACATG...
74628,GC_Mendelian_variants:REF_chr7:156791472C*G|SHH,GCAAAAATAATGAAAGAATGCAATGAAAGCTCGTGGAGACAGAGGC...
...,...,...
79515,MK:tile_47747|chr5-87945240+87945510|T-A-1,ATATCAAAACTGATTACTAGCTGAGCTTTTGTGTCCATAAAATAAA...
79571,MK:tile_47785|chr6-164344756+164345026|G-C-269,AGTTTGGGCTCAGAGGCCTGACAGCATCTATCTGAACATCTAATAT...
79577,MK:tile_47791|chr6-170438656+170438926|G-C-1,CTTACCCTGGCAACGAAGCCCTTGATTGGCGGCGGCTCCACAAACA...
79630,MK:tile_985|chr1-33363965+33364235|C-T-268,TCTTAGGCTCTGTTCTCCGCCCCTTACTCCCCTCCCCCACCTGCTC...


### old code for renaming regions file

In [ ]:
def rename_regions(region_name):
    """
    Rename regions to be consistent with the naming in the final design fasta
    """
    if region_name.startswith('cardiac_neuro_cava_random'):
        return region_name.replace(',', '~')
    else: return region_name

# read bed file like csv and rename the name column ("," => "~") (only for tested regions)
# region_bed = pd.read_csv(config['files']['final_design']['region_bed'], sep='\t', header=None)
region_bed = pd.read_csv('/home/kisa/coding/80K_MPRA/design_data/design_info/renamed_regions.bed', sep='\t', header=None)
region_bed.columns = ['region_chr', 'region_start', 'region_end', 'region_name', 'region_score', 'region_strand']

# region_bed['region_design_name'] = region_bed['region_name'].apply(lambda x: x.replace(',', '~'))

# test if it was working:
# get metadata file
tested_region_bed = region_bed.loc[region_bed['region_name'].str.contains('cardiac_neuro_cava_random')]
tested_region_bed.columns
# tested_region_bed.shape[0] # 28390

Index(['region_chr', 'region_start', 'region_end', 'region_name',
       'region_score', 'region_strand'],
      dtype='object')

In [ ]:
# meta_data_file = pd.read_csv(config['files']['creating']['metadata_table'], sep="\t")

# # only tested sequences
# tested_meta_data_file = meta_data_file.loc[meta_data_file['name'].str.startswith('cardiac_neuro_cava_random')]

# # tested_meta_data_file # 73940
# tested_meta_data_file['variant_pattern'] = tested_meta_data_file['name'].apply(get_chrom_pos_ref_alt_pattern)

# # remove the alternative sequences
# tested_element_meta_data_file = tested_meta_data_file.loc[tested_meta_data_file['variant_pattern'] == 'NA']
# # remove the reference sequences
# tested_element_meta_data_file = tested_element_meta_data_file.loc[~tested_element_meta_data_file['name'].str.contains(':REF_')]
# tested_element_meta_data_file.shape[0] # 8900



# investigate if all element headers can be matched by the region bed (expectation: yes all can be matched)
check_new_region_header = tested_element_meta_data_file.merge(tested_region_bed, left_on='name', right_on='region_name', how='left')
check_new_region_header.columns
unmergable_fasta_bed = check_new_region_header.loc[check_new_region_header['region_start'].isna()] # 0
unmergable_fasta_bed.shape[0] # 0 => all could be matched so new header works better

0

### Summary of the format requirements:
- name - a unique-within-file identifier (free form, starts with alphabetical characters) one unique string per tested sequence
- sequence - (string consisting of A, C, G, or T, no "N")
- category - [variant, element, synthetic, scrambled]
- class - [test, variant positive control, variant negative control, element active control, element inactive control] source - (free form, e.g. "Cardio FG 2022")
- ref - reference sequence ["GRCh38", NA allowed]
- chr - chromosome [NA allowed]
- start - 0-based position of the left-most position of sequence with respect to the reference [NA allowed]
- end - 1-based position of the right-most position of sequence with respect to the reference [NA allowed]
- strand - strand of sequence in reference [+,-,.,NA]
- variant_class - array [SNP, indel, NA]
- variant_pos - 0-based position of the start of the variant(s) in the sequence - array [NA allowed] Note: for indels use normalized representation from SPDI Note: acceptable values range from 0 to length(sequence)-1
- SPDI - 0-based, validated SPDI representation of the variant(s) - array [NA allowed] Note: these look like "​​NC_000001.11:25253603:G:A"  Note: they should be validated using one of the two options, see Mike or Jon for details, see https://github.com/mikelove/igvf_spdi_demo 
- allele - the coding with respect to the reference which is hg38 for IGVF - array [ref, alt, NA]
- info - any additional comment (free form)


### Investigate current state of Metadata file:
- Problems: 
  - From header no genomic region known
- Focus on tested sequences:
  - Check category

- Open questions:
  - SPDI: representation of variants
  - Controls:
    - Elements are what is not in the variant region map?
    - What is the reference sequence? 
    - seq_chr, seq_start, seq_end, seq_strand
    - Variants:
      - We find them with the variant region map, but where are these variants? 
      - Do we have indels?

      

In [ ]:
# current stat metadata file 05.07.2024
meta_data_file = pd.read_csv(config['files']['creating']['metadata_table'], sep="\t")
print(meta_data_file.columns)
meta_data_file.head()

tested_meta_data_file = meta_data_file.loc[meta_data_file['name'].str.startswith('cardiac_neuro_cava_random')]

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'chr', 'start', 'end', 'strand', 'variant_class',
       'variant_pos', 'SPDI', 'allele', 'info', 'tmp_matching_header'],
      dtype='object')


#### Hard facts: 
- shape, current number of variants, elements
- variant region map: 
  - ref: 18883
  - alt: 47044

In [10]:
import ast
# 03.09.2024
# read datafreeze:
datafreeze = pd.read_csv(config['files']['creating']['datafreeze_table'], sep='\t', low_memory=False)


# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

# list columns col_variant_class, col_variant_pos, col_SPDI, col_allele,
list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]
# Apply the safe_eval function to the specified columns
for col in list_columns:
    datafreeze[col] = datafreeze[col].apply(safe_eval)

In [11]:
datafreeze.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_region_id', 'tmp_open_NGN2_WTC11_NP_screen_regions',
       'tmp_chr_start_end', 'tmp_open_NGN2_WTC11_NP',
       'tmp_open_h1_neuro_dnase_screen', 'tmp_MPRA_primary_region_overlap',
       'tmp_MPRA_organoid_region_overlap', 'tmp_gene_name', 'tmp_gene_set',
       'tmp_is_neuro', 'tmp_is_cardiac', 'tmp_is_cava', 'tmp_is_random'],
      dtype='object')

In [12]:
datafreeze['tmp_label'].unique()


array(['cardiac_neuro_cava_random'], dtype=object)

In [61]:
# get number of variants:
all_80K_variants = datafreeze.loc[datafreeze['allele'].apply(hf.is_alternative)] # 46374
all_80K_variants.shape[0] # 46374

# get chrom_pos_ref_alt pattern
all_80K_variants['tmp_chrom_pos_ref_alt'] = all_80K_variants['header'].apply(hf.get_chrom_pos_ref_alt_pattern)
all_80K_variants = all_80K_variants[['tmp_chrom_pos_ref_alt', 'header']]
# split in columns: chrom, pos, ref, alt
all_80K_variants[['chrom', 'pos', 'ref', 'alt']] = all_80K_variants['tmp_chrom_pos_ref_alt'].str.split('-', expand=True)

# write to file
output_path = os.path.join('/data/cephfs-1/home/users/kisa11_c/unmirrored/projects/MPRA/IGVF_Y1_design/projects/gnomadDB', 'all_80K_variants_for_gnomadDB.tsv')
gnomad_cols = ['header', 'chrom', 'pos', 'ref', 'alt']
all_80K_variants[gnomad_cols]
# all_80K_variants[gnomad_cols].to_csv(output_path, sep='\t', index=False)


/tmp/ipykernel_406854/3215843360.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_80K_variants['tmp_chrom_pos_ref_alt'] = all_80K_variants['header'].apply(hf.get_chrom_pos_ref_alt_pattern)


In [62]:
# number of regions and number of variants per gene list
datafreeze.columns
# datafreeze['tmp_gene_set'].value_counts()
# datafreeze['tmp_region_id'].value_counts()

# groupby gene set and count region id
region_number_per_gene_set = datafreeze.groupby('tmp_gene_set')['tmp_region_id'].nunique().reset_index()
region_number_per_gene_set
# region_number_per_gene_set is a table for each combination of gene_sets (cardiac, neuro, cava, random) and the number of regions associated to this gene set
# iterate over the gene set and get the overall number of regions associated to 'cardiac', 'neuro', 'cava' and 'random'
gene_sets = ['cardiac', 'neuro', 'cava', 'random']
region_number_per_gene_set_dict = {}
for gene_set in gene_sets:
    region_number_per_gene_set_dict[gene_set] = region_number_per_gene_set.loc[region_number_per_gene_set['tmp_gene_set'].apply(lambda gene_set_list: True if gene_set in gene_set_list else False)]['tmp_region_id'].sum()

# make to dataframe again
region_number_per_gene_set_df = pd.DataFrame(region_number_per_gene_set_dict.items(), columns=['gene_set', 'number_of_regions'])

In [63]:
region_number_per_gene_set_df


,gene_set,number_of_regions
0,cardiac,8639
1,neuro,17246
2,cava,1957
3,random,717


In [64]:
# number of genes per gene set
# groupby gene set and count gene name
gene_number_per_gene_set = datafreeze.groupby('tmp_gene_set')['tmp_gene_name'].nunique().reset_index()
gene_number_per_gene_set

# iterate over the gene set and get the overall number of genes associated to 'cardiac', 'neuro', 'cava' and 'random'
gene_sets = ['cardiac', 'neuro', 'cava', 'random']
gene_number_per_gene_set_dict = {}
for gene_set in gene_sets:
    gene_number_per_gene_set_dict[gene_set] = gene_number_per_gene_set.loc[gene_number_per_gene_set['tmp_gene_set'].apply(lambda gene_set_list: True if gene_set in gene_set_list else False)]['tmp_gene_name'].sum()

# make to dataframe again
gene_number_per_gene_set_df = pd.DataFrame(gene_number_per_gene_set_dict.items(), columns=['gene_set', 'number_of_genes'])

In [65]:
gene_number_per_gene_set_df

,gene_set,number_of_genes
0,cardiac,164
1,neuro,306
2,cava,48
3,random,25


- Investigate the variant numbers of common vs ultra-rare and singleton
- all: 46375 (expected: 35000 (<0.01 AF) + )
- common
    | gene_set | number_of_variants_common |
    |----------|---------------------------|
    | cardiac  | 7568                      |
    | neuro    | 13108                     |
    | cava     | 1576                      |
    | random   | 600                       |
- rare (<= 0.01 AF) (expected: 10000 for cardiac, neuro, cava, and 5000 random)
    |gene_set | number_of_variants_rare |
    |---------|--------------------------|
    |cardiac  | 7818                     |
    |neuro    | 8387                     |
    |cava     | 6732                     |
    |random   | 3394                     |
- singleton: 12137
- 

In [69]:
# read in the af and allele count data

all_80K_variants = datafreeze.loc[datafreeze['allele'].apply(hf.is_alternative)] # 46374

gnomad_data_all_variants = pd.read_csv(config['files']['creating']['gnomad_data_all_variants'], sep='\t')

In [80]:
gnomad_data_all_variants.head()
gnomad_data_all_variants.columns
gnomad_data_all_variants_interesting = gnomad_data_all_variants[['header', 'AF', 'AC']]

# add to the all_80K_variants the allele frequency and allele count
all_80K_variants_with_af = all_80K_variants.merge(gnomad_data_all_variants_interesting, on='header', how='left')

all_80K_variants_with_af['AF'].isna().sum() # 0

# make is common column
all_80K_variants_with_af['is_common'] = all_80K_variants_with_af['AF'].apply(lambda x: True if x > 0.01 else False)
all_80K_variants_with_af['is_rare'] = all_80K_variants_with_af['AF'].apply(lambda x: True if x <= 0.01 else False)
all_80K_variants_with_af['is_singleton'] = all_80K_variants_with_af['AC'].apply(lambda x: True if x == 1 else False)
# group by is common and count the number of variants
variant_count_common_rare = all_80K_variants_with_af.groupby(['tmp_gene_set','is_common'])['header'].count().reset_index()

common_variant_count = variant_count_common_rare.loc[variant_count_common_rare['is_common'] == True]

common_variant_count
# iterate over the gene set and get the overall number of genes associated to 'cardiac', 'neuro', 'cava' and 'random'
variant_number_per_gene_set_dict_common = {}
for gene_set in gene_sets:
    variant_number_per_gene_set_dict_common[gene_set] = common_variant_count.loc[common_variant_count['tmp_gene_set'].apply(lambda gene_set_list: True if gene_set in gene_set_list else False)]['header'].sum()

# make to dataframe again
variant_number_per_gene_set_df_common = pd.DataFrame(variant_number_per_gene_set_dict_common.items(), columns=['gene_set', 'number_of_variants_common'])
variant_number_per_gene_set_df_common

# | gene_set | number_of_variants_common |
# |----------|---------------------------|
# | cardiac  | 7568                      |
# | neuro    | 13108                     |
# | cava     | 1576                      |
# | random   | 600                       |

,gene_set,number_of_variants_common
0,cardiac,7568
1,neuro,13108
2,cava,1576
3,random,600


In [83]:
common_variant_count['header'].sum() # 22001


22001

In [81]:
# number of variants with ac == 1
all_80K_variants_with_af.loc[all_80K_variants_with_af['AC'] == 1] # 12137

,header,sequence,tmp_label,name,category,class,source,ref,variant_class,variant_pos,...,tmp_MPRA_organoid_region_overlap,tmp_gene_name,tmp_gene_set,tmp_is_neuro,tmp_is_cardiac,tmp_is_cava,tmp_is_random,AF,AC,is_common
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCTGGGTGACCCGGAGAACACCAAGGCTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[43],...,False,SKI,['neuro'],True,False,False,False,0.000007,1.0,False
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCATGCGGTGGCCACAGCCTCGGGTGAGTTC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[116],...,False,SKI,['neuro'],True,False,False,False,0.000007,1.0,False
6,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCCACTTGTCAGGAAGCCTGACCCCCAAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[71],...,False,SKI,['neuro'],True,False,False,False,0.000007,1.0,False
9,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCCACTTGTCAGGAAGCCTGACCCCCAAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[185],...,False,SKI,['neuro'],True,False,False,False,0.000007,1.0,False
11,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCCTGGCCCACGAGCCCCAGGCCACGGCCTC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[173],...,False,SKI,['neuro'],True,False,False,False,0.000007,1.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46365,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTTTGCTGAGTAGTATCCGTTGTATGAATGCAC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[148],...,False,G6PD,['cava'],False,False,True,False,0.000009,1.0,False
46368,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTTTGCTGAGTAGTATCCGTTGTATGAATGCAC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[75],...,False,G6PD,['cava'],False,False,True,False,0.000009,1.0,False
46369,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[64],...,False,G6PD,['cava'],False,False,True,False,0.000009,1.0,False
46370,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,[SNV],[149],...,False,G6PD,['cava'],False,False,True,False,0.000009,1.0,False


In [77]:
rare_variant_count = variant_count_common_rare.loc[variant_count_common_rare['is_common'] == False]
# iterate over the gene set and get the overall number of genes associated to 'cardiac', 'neuro', 'cava' and 'random'
variant_number_per_gene_set_dict_rare = {}
for gene_set in gene_sets:
    variant_number_per_gene_set_dict_rare[gene_set] = rare_variant_count.loc[rare_variant_count['tmp_gene_set'].apply(lambda gene_set_list: True if gene_set in gene_set_list else False)]['header'].sum()

# make to dataframe again
variant_number_per_gene_set_df_rare = pd.DataFrame(variant_number_per_gene_set_dict_rare.items(), columns=['gene_set', 'number_of_variants_rare'])
variant_number_per_gene_set_df_rare

# make markdown table
# |gene_set | number_of_variants_rare |
# |---------|--------------------------|
# |cardiac  | 7818                     |
# |neuro    | 8387                     |
# |cava     | 6732                     |
# |random   | 3394                     |


,gene_set,number_of_variants_rare
0,cardiac,7818
1,neuro,8387
2,cava,6732
3,random,3394


In [84]:
rare_variant_count['header'].sum() # 24373

24373

In [71]:
all_80K_variants_with_af.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_region_id', 'tmp_open_NGN2_WTC11_NP_screen_regions',
       'tmp_chr_start_end', 'tmp_open_NGN2_WTC11_NP',
       'tmp_open_h1_neuro_dnase_screen', 'tmp_MPRA_primary_region_overlap',
       'tmp_MPRA_organoid_region_overlap', 'tmp_gene_name', 'tmp_gene_set',
       'tmp_is_neuro', 'tmp_is_cardiac', 'tmp_is_cava', 'tmp_is_random', 'AF',
       'AC', 'is_common'],
      dtype='object')

##### Problem: 
- variant region map is not deduplicated
1. Read variant region map: 
2. Match with alt id to header (variant region map.merge(matchable header))


In [ ]:
variant_region_map = pd.read_csv(config['files']['final_design']['final_design_variant_region_map'], sep="\t")
variant_region_map

all_variants = variant_region_map.merge(tested_meta_data_file, left_on='ALT_ID', right_on='tmp_matching_header', how='left')
all_matchable_variants = all_variants.loc[~all_variants['name'].isna()]
all_matchable_variants
# all_matchable_variants['REF_ID'].nunique()

,Variant,Region,REF_ID,ALT_ID,header,sequence,tmp_label,name,category,class,...,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_matching_header
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,...,NaN,NaN,NaN,NaN,SNP,NaN,NaN,alt,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,...,NaN,NaN,NaN,NaN,SNP,NaN,NaN,alt,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCTGGGTGACCCGGAGAACACCAAGGCTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,...,NaN,NaN,NaN,NaN,SNP,NaN,NaN,alt,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCATGCGGTGGCCACAGCCTCGGGTGAGTTC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,...,NaN,NaN,NaN,NaN,SNP,NaN,NaN,alt,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGGACTCCGGTGCCTTCGCATTCCCGAGCTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,...,NaN,NaN,NaN,NaN,SNP,NaN,NaN,alt,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46369,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTGGAGCTCTGCCTCACCCCACCTGGCCCCAAT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,...,NaN,NaN,NaN,NaN,SNP,NaN,NaN,alt,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...
46370,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,AGGACCGGATCAACTATGTCTGAATTCACCTCCAAATAATGGGAAA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,variant,test,...,NaN,NaN,NaN,NaN,SNP,NaN,NaN,alt,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...
46371,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,cardiac_neuro

In [ ]:
# by removing variants not in final design
18572 + 46374

64946

In [ ]:
meta_data_file.shape

(80215, 18)

In [ ]:
meta_data_file['category'].value_counts()
# 64946 variants => how many entries in variant region map

variant      64946
element      14402
scrambled      667
synthetic      200
Name: category, dtype: int64

#### Focus on tested seqeuences: 
- verified the number of variants with variant region map by removing duplicates
  - unique REF_IDs: 18572
  - unique ALT_IDs: 46374 (matching to the variant region map) (46458 if "ALT_" pattern)
- read in blat results and investigate if all references are there

In [ ]:
tested_meta_data_file['category'].value_counts()

variant    64946
element     8994
Name: category, dtype: int64

In [ ]:
# read blat results in
blat_aligned_tested_elements = pd.read_csv(config['files']['creating']['blat_tested_element_alignment_local'], sep="\t")
blat_aligned_tested_elements # 27482 (expected number of reference or element sequences in tested group)
blat_aligned_tested_elements.columns = ['blat_name', 'blat_match', 'blat_strand', 'blat_style_chr', 'blat_start', 'blat_end']
# preprocess: change chromosome names
blat_aligned_tested_elements['blat_chr'] = blat_aligned_tested_elements['blat_style_chr'].apply(set_modified_chromosome)
blat_aligned_tested_elements.drop(columns=['blat_match', 'blat_style_chr'], inplace=True)
blat_aligned_tested_elements.head()
blat_aligned_tested_elements.dtypes

blat_name      object
blat_strand    object
blat_start      int64
blat_end        int64
blat_chr       object
dtype: object

In [ ]:
# # all matchable (since sum is 0)
# blat_aligned_tested_elements.merge(tested_meta_data_file, left_on='Q_name', right_on='header', how='left')['sequence'].isna().sum()


# check how many elements you would expect:
# 1 only tested
# 2 remove ALT_
# how many do we have? 27482

# match blat information to the meta data file
# 1. add REF_ID column ref id to tested_meta_data_file (left join from variant region map)
tested_meta_data_file_with_REF = tested_meta_data_file.merge(variant_region_map[['REF_ID', 'ALT_ID']], left_on='tmp_matching_header', right_on='ALT_ID', how='left')

# all alt were matched
tested_meta_data_file_with_REF.loc[~tested_meta_data_file_with_REF['REF_ID'].isna()].shape[0] # shape == 46374 => worked, all alt were matched

# 2. add match_blat_header (if REF_ID given -> REF_ID; else -> header )

def get_blat_header(row):
    if pd.isna(row['REF_ID']):
        return row['header']
    else:
        return row['REF_ID']

tested_meta_data_file_with_REF['match_blat_header'] = tested_meta_data_file_with_REF.apply(get_blat_header, axis=1)

# make headers matchable
tested_meta_data_file_with_REF['match_blat_header'] = tested_meta_data_file_with_REF['match_blat_header'].str.replace(',','~')
tested_meta_data_file_with_REF['match_blat_header'] = tested_meta_data_file_with_REF['match_blat_header'].str.replace('>','*')

# if na: a header is initially na
tested_meta_data_file_with_REF['match_blat_header'].isna().sum() # 0 => all matched

# 3. left join blat aligned tested
blat_aligned_tested_elements_with_meta = tested_meta_data_file_with_REF.merge(blat_aligned_tested_elements, left_on='match_blat_header', right_on='blat_name', how='left')

# 4. check if all rows have T_start
blat_aligned_tested_elements_with_meta['blat_start'].isna().sum() # 84 => not all matched

# 5. Handle the missing headers first
matchable_by_blat = blat_aligned_tested_elements_with_meta.loc[~blat_aligned_tested_elements_with_meta['blat_start'].isna()]['match_blat_header'].to_list()
def get_region_pattern(header):
    """
    For missing sequences which are alternatives without reference: get region pattern
    cardiac_neuro_cava_random:ALT_PRDM16|ENSG00000142611.17|EH38E2779927_fwd_tile1-1_PRDM16|ENSG00000142611.17|EH38E2779927|1-3314074-G-C => PRDM16|ENSG00000142611.17|EH38E2779927
    cardiac_neuro_cava_random:ALT_G6PD|ENSG00000160211.20|EH38E3949687_rev_tile1-1_G6PD|ENSG00000160211.20|EH38E3949687|X-154491628-G-C => G6PD|ENSG00000160211.20|EH38E3949687
    """
    if header in matchable_by_blat:
        return "blat_matchable"
    if '~' in header:
        #'cardiac_neuro_cava_random:ALT_DEAF1|ENSG00000177030.19|EH38E2937977~SLC25A22|ENSG00000177542.11|EH38E2937977_rev_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937977|11-744461-A-C~SLC25A22|ENSG00000177542.11|EH38E2937977|11-744461-A-C',
        # remove "~" and call function again
        new_header = get_region_pattern(header.split('~')[0] + '_suffix')
        return new_header
    if ',' in header:
        # remove "," and call function again
        new_header = get_region_pattern(header.split(',')[0] + '_suffix')
        return new_header
    if 'REF_' in header:
        return header.split(':REF_')[1].split('_')[0]
    elif 'ALT_' in header:
        return header.split(':ALT_')[1].split('_')[0]
    else:
        return header.split(':')[1].split('_')[0]

# invent gene_enhancer_id
blat_aligned_tested_elements_with_meta['gene_enhancer_id'] = blat_aligned_tested_elements_with_meta['match_blat_header'].apply(get_region_pattern)

# read new tested_regions.bed in and left join to the (find unix command to generate the file in 07_quality_control/notebooks/state_of_data.ipynb (07.07.2024))
modified_tested_region_bed = pd.read_csv(config['files']['creating']['modified_tested_region_bed'], sep=" ", header=None)
modified_tested_region_bed.columns = ['tested_region_chr', 'tested_region_start', 'tested_region_end', 'gene_enhancer_id', 'tested_region_score', 'tested_region_strand']
modified_tested_region_bed

# # left join with tested region bed and
blat_aligned_tested_elements_with_meta = blat_aligned_tested_elements_with_meta.merge(modified_tested_region_bed[['tested_region_chr', 'tested_region_start', 'tested_region_end', 'gene_enhancer_id', 'tested_region_strand']], on='gene_enhancer_id', how='left')
blat_aligned_tested_elements_with_meta.loc[~blat_aligned_tested_elements_with_meta['tested_region_start'].isna()].shape[0]

# 6. set chr, start, end and strand
# def set_chr_start_end_strand(row):
#     """
#     For blat matchable sequences and missing sequences set chr, start, end amd strand

#     """
#     if row['gene_enhancer_id'] == 'blat_matchable':
#         row['chr'] = row['blat_chr']
#         row['start'] = row['blat_start']
#         row['end'] = row['blat_end']
#         row['strand'] = row['blat_strand']
#         return row
#     if pd.isna(row['tested_region_chr']):
#         # error: unexpected case
#         raise ValueError('Unexpected case: tested_region_chr is na')
#     row['chr'] = row['tested_region_chr']
#     row['start'] = row['tested_region_start']
#     row['end'] = row['tested_region_end']
#     row['strand'] = row['tested_region_strand']
#     return row

blat_aligned_tested_elements_with_meta.columns
blat_aligned_tested_elements_with_meta = blat_aligned_tested_elements_with_meta.apply(set_chr_start_end_strand, axis=1)

# 7. remove unused columns
# remove all columns with prefix "blat_" or "tested_region_"
blat_aligned_tested_elements_with_meta.drop(columns=[col for col in blat_aligned_tested_elements_with_meta.columns if 'blat_' in col or 'tested_region_' in col], inplace=True)

# 8. Make start and end to int values
blat_aligned_tested_elements_with_meta['start'] = blat_aligned_tested_elements_with_meta['start'].astype('Int64')
blat_aligned_tested_elements_with_meta['end'] = blat_aligned_tested_elements_with_meta['end'].astype('Int64')

def get_chrom_pos_ref_alt_pattern(header):
    """
    In the 80K MPRA design each variant has a chr-pos-ref-alt pattern in the header
    This function is able to match it and returns the result
    """
    pattern = r'([A-Z]|[0-9]+)-[0-9]+-[A-Z]-[A-Z]'
    matches = re.search(pattern, header)
    if matches:
        return matches.group()
    else: return "NA"

# 9. compute the variant position for each variant:
def get_variant_position(row, header_col='name', with_adapter=False):
    """Returns the 0-based variant position if it is a variant (start is 0-based as well)"""
    adapter_count = 0
    if with_adapter:
        adapter_count = 15
    if row['category'] != 'variant' or ':REF_' in row[header_col]:
        row['variant_position'] = np.nan
    else: # compute variant position based on chr-pos-ref-alt pattern
        chrom_pos_ref_alt = get_chrom_pos_ref_alt_pattern(row[header_col])
        chrom, pos, ref, alt = chrom_pos_ref_alt.split('-')
        row['variant_pos'] = int(pos) - row['start'] - 1 # 0-based + adapter
        if row['strand'] == "-":
            revert_variant_pos = 270 - row['variant_pos']
            row['variant_pos'] = revert_variant_pos
        row['variant_pos'] = row['variant_pos'] + adapter_count
    return row

# 1. if row['category'] != 'variant': variant_position = np.nan
# 2. else: extract genomic variant position based on chr-pos-ref-alt and compute variant_position = row['variant_position'] - row[start] (+ 1) (check if computed number is correct for 2 examples)
# 3. return row
blat_aligned_tested_elements_with_meta_test = blat_aligned_tested_elements_with_meta.apply(lambda row: get_variant_position(row, header_col='name', with_adapter=False), axis=1)

# blat_aligned_tested_elements_with_meta # 73940 rows × 21 columns

In [ ]:
# # add spdi with script call:
# # skript expects csv and given column of the user string and adds SPDI column to the dataframe

# python make_spdi_list.py --input-file=test_data/user_variant_pattern_100.csv --output-file=test_data/user_variant_pattern_100_spdi.csv --column-separator='\t' --string-separator="_" --column-name="variant_string" --indices="0,1,3,4" --spdi-batch-processing-output="test_data/spdi_for_batch_processing.txt" --spdi-batch-output="test_data/spdi_batch_output.txt"

# /home/kisa/coding/80K_MPRA/igvf_spdi_demo/make_spdi_list.py

##### Code to add SPDI

In [233]:
# dict of chr number to refseq chromosome number
chrom_2_refseq = {"1": "NC_000001.11",
    "2": "NC_000002.12",
    "3": "NC_000003.12",
    "4": "NC_000004.12",
    "5": "NC_000005.10",
    "6": "NC_000006.12",
    "7": "NC_000007.14",
    "8": "NC_000008.11",
    "9": "NC_000009.12",
    "10": "NC_000010.11",
    "11": "NC_000011.10",
    "12": "NC_000012.12",
    "13": "NC_000013.11",
    "14": "NC_000014.9",
    "15": "NC_000015.10",
    "16": "NC_000016.10",
    "17": "NC_000017.11",
    "18": "NC_000018.10",
    "19": "NC_000019.10",
    "20": "NC_000020.11",
    "21": "NC_000021.9",
    "22": "NC_000022.11",
    "X": "NC_000023.11",
    "Y": "NC_000024.10"}

# get args with click: input file, seperator, ids of chr, pos, ref, alt in string

def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) - 1 # (input: 1-based => 0-based)
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'

def get_spdi(row, header_col='name'):
    """
    Returns the SPDI identifier for the given variant (tested only)
    Assumption: allele need to be set beforehands
    """
    # TODO: add case for controls
    if not (hf.is_alternative(row['allele'])):
        row['SPDI'] = np.nan
        return row
    # identify the variant chrom-pos-ref-alt pattern
    chrom_pos_ref_alt = hf.get_chrom_pos_ref_alt_pattern(row[header_col])
    if chrom_pos_ref_alt == "NA":
        raise ValueError('Variant pattern could not be found')
    # create the SPDI identifier
    row['SPDI'] = create_speedy_chromosomes(chrom_pos_ref_alt, seperator='-', indices=[0,1,2,3])
    return row

# blat_aligned_tested_elements_with_meta_test_spdi = blat_aligned_tested_elements_with_meta_test.apply(lambda row: get_spdi(row, header_col='name'), axis=1)


# blat_aligned_tested_elements_with_meta_test_spdi[['SPDI']].to_csv('SPDI_')

In [ ]:
chrom_2_refseq = {"1": "NC_000001.11",
                "2": "NC_000002.12",
                "3": "NC_000003.12",
                "4": "NC_000004.12",
                "5": "NC_000005.10",
                "6": "NC_000006.12",
                "7": "NC_000007.14",
                "8": "NC_000008.11",
                "9": "NC_000009.12",
                "10": "NC_000010.11",
                "11": "NC_000011.10",
                "12": "NC_000012.12",
                "13": "NC_000013.11",
                "14": "NC_000014.9",
                "15": "NC_000015.10",
                "16": "NC_000016.10",
                "17": "NC_000017.11",
                "18": "NC_000018.10",
                "19": "NC_000019.10",
                "20": "NC_000020.11",
                "21": "NC_000021.9",
                "22": "NC_000022.11",
                "X": "NC_000023.11",
                "Y": "NC_000024.10"}

In [ ]:
def make_love_example_variant_pattern(variant_pattern):
    """
    Gets variant pattern chrom-pos-ref-alt and creates: 1_25253604_hg38_G_A as test data
    """

    if variant_pattern == "NA":
        return "NA"
    chrom, pos, ref, alt = variant_pattern.split('-')
    return f'{chrom}_{pos}_hg38_{ref}_{alt}'


In [ ]:
blat_aligned_tested_elements_with_meta_test_spdi['variant_pattern'] = blat_aligned_tested_elements_with_meta_test_spdi['name'].apply(get_chrom_pos_ref_alt_pattern)
blat_aligned_tested_elements_with_meta_test_spdi['variant_string'] = blat_aligned_tested_elements_with_meta_test_spdi['variant_pattern'].apply(make_love_example_variant_pattern)
blat_aligned_tested_elements_with_meta_test_spdi['variant_name'] = ['variant_' + str(i+1) for i in range(len(blat_aligned_tested_elements_with_meta_test_spdi))]


In [ ]:
# # write SPDI column to file and test with spdi_batch.py
# blat_aligned_tested_elements_with_meta_test_spdi.loc[~blat_aligned_tested_elements_with_meta_test_spdi['SPDI'].isna()][['SPDI']].to_csv('tested_variants_spdi.csv', header=False, index=None)
# # blat_aligned_tested_elements_with_meta_test_spdi.loc[~blat_aligned_tested_elements_with_meta_test_spdi['SPDI'].isna()][['variant_pattern']].to_csv('tested_variant_pattern.csv', header=False, index=None)
# blat_aligned_tested_elements_with_meta_test_spdi.loc[~blat_aligned_tested_elements_with_meta_test_spdi['SPDI'].isna()][['variant_string','variant_name']].to_csv('tested_variant_pattern.csv', index=None, sep="\t")

In [ ]:
# head -n 100 /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/tested_variant_pattern.csv > /home/kisa/coding/80K_MPRA/igvf_spdi_demo/test_data/user_variant_pattern_100.csv

#### Check variant position
- open question: is the given variant position 0-based or 1-based?
  - checked using [genome viewer](https://www.ncbi.nlm.nih.gov/gdv/browser/genome/?id=GCF_000001405.40) (3:12148321) fwd: AC"C"CC (which fits but the viewer is one based => 0 based would be 12148320)

##### Minus strand

In [ ]:
blat_aligned_tested_elements_with_meta_test.loc[blat_aligned_tested_elements_with_meta_test['name'] == 'cardiac_neuro_cava_random:ALT_SMAD2|ENSG00000175387.16|EH38E1914457_rev_tile1-1_SMAD2|ENSG00000175387.16|EH38E1914457|18-47937202-G-T'][['name', 'start', 'end', 'strand', 'variant_pos']]


,name,start,end,strand,variant_pos
49885,cardiac_neuro_cava_random:ALT_SMAD2|ENSG000001...,47937088,47937358,-,172.0


##### Plus strand

In [ ]:
blat_aligned_tested_elements_with_meta_test.loc[blat_aligned_tested_elements_with_meta_test['name'] == 'cardiac_neuro_cava_random:ALT_SYN2|ENSG00000157152.17|EH38E2178908_fwd_tile1-1_SYN2|ENSG00000157152.17|EH38E2178908|3-12148321-C-T'][['name', 'start', 'end', 'strand', 'variant_pos']]

,name,start,end,strand,variant_pos
57840,cardiac_neuro_cava_random:ALT_SYN2|ENSG0000015...,12148219,12148489,+,116.0


In [ ]:
blat_aligned_tested_elements_with_meta_test.loc[blat_aligned_tested_elements_with_meta_test['name'] == 'cardiac_neuro_cava_random:ALT_SYN2|ENSG00000157152.17|EH38E2178908_fwd_tile1-1_SYN2|ENSG00000157152.17|EH38E2178908|3-12148321-C-T']['sequence'].to_list()

['AGGACCGGATCAACTGATCTGGGATCTGGTGAGGGGTCACACTCTCACCCATTCACAATCTGAGCACATTGCCTTGGGCCCTGCATGAGGAGAACAGCAGGGCTCCCTGGCCAGACTCCAAGTTCTCCCTTCAGTTTCACAAAGCAGCTGCCACATGAATGACGTTTTACATATTGTTCTCTGCTAGTTCATGGGAGTGGACAAATTTGGTGACTGTGAGGGGAGGGTCCCTTTCAGCTGATGACTTTGGGTTGTGTTTCTCAATATCTTTACTATGACTGCTGTCATTGCGTGAACCGA']

In [ ]:
blat_aligned_tested_elements_with_meta.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'chr', 'start', 'end', 'strand', 'variant_class',
       'variant_pos', 'SPDI', 'allele', 'info', 'tmp_matching_header',
       'REF_ID', 'ALT_ID', 'gene_enhancer_id'],
      dtype='object')

In [ ]:
# sanity check:
# columns
blat_aligned_tested_elements_with_meta.columns
# chr, start, end strand na? all zero
# print(blat_aligned_tested_elements_with_meta['chr'].isna().sum())
# print(blat_aligned_tested_elements_with_meta['start'].isna().sum())
# print(blat_aligned_tested_elements_with_meta['end'].isna().sum())
# print(blat_aligned_tested_elements_with_meta['strand'].isna().sum())
# sample 3 sequences and check if the coordinates are correct
np.random.seed(42)
sampled_examples = blat_aligned_tested_elements_with_meta.sample(n=3)

0
0
0
0


In [ ]:
sampled_examples[['name', 'chr', 'start', 'end', 'strand']]
# first sequences fine according to genome browswer: https://genome-euro.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr14%3A68520699%2D68520800&hgsid=343076591_9AcUCsUilw31AjifKDKNQOSBOl0e
# second sequence fine: look at the right end and then the other strand to compare with the sequence from left to right https://genome-euro.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr18%3A47937308%2D47937358&hgsid=343076591_9AcUCsUilw31AjifKDKNQOSBOl0e
# third sequence fine (samtools expects 1-based): samtools faidx GCF_000001405.26_GRCh38_genomic.fna NC_000003.12:12148219-12148489 (in folder: /data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26)


,name,chr,start,end,strand
14313,cardiac_neuro_cava_random:REF_RAD51B|ENSG00000...,chr14,68520699.0,68520969.0,+
49885,cardiac_neuro_cava_random:ALT_SMAD2|ENSG000001...,chr18,47937088.0,47937358.0,-
57840,cardiac_neuro_cava_random:ALT_SYN2|ENSG0000015...,chr3,12148219.0,12148489.0,+


In [ ]:
sampled_examples['name'].to_list()
# 'cardiac_neuro_cava_random:REF_RAD51B|ENSG00000182185.19|EH38E3101463_fwd_tile1-1',
# 'cardiac_neuro_cava_random:ALT_SMAD2|ENSG00000175387.16|EH38E1914457_rev_tile1-1_SMAD2|ENSG00000175387.16|EH38E1914457|18-47937202-G-T',
# 'cardiac_neuro_cava_random:ALT_SYN2|ENSG00000157152.17|EH38E2178908_fwd_tile1-1_SYN2|ENSG00000157152.17|EH38E2178908|3-12148321-C-T'

['cardiac_neuro_cava_random:REF_RAD51B|ENSG00000182185.19|EH38E3101463_fwd_tile1-1',
 'cardiac_neuro_cava_random:ALT_SMAD2|ENSG00000175387.16|EH38E1914457_rev_tile1-1_SMAD2|ENSG00000175387.16|EH38E1914457|18-47937202-G-T',
 'cardiac_neuro_cava_random:ALT_SYN2|ENSG00000157152.17|EH38E2178908_fwd_tile1-1_SYN2|ENSG00000157152.17|EH38E2178908|3-12148321-C-T']

In [ ]:
sampled_examples['sequence'].to_list()

['AGGACCGGATCAACTTACATAGTGAGGACCTACTGTGTGCCAGTTAGGGTGTGGGGAAGAACAGGAGACAAACCACCTGGTCTCATTCTCATGAATTTTACAGTCTATTGGGAAACAGGTATTACACAAGTCAATATGAAAATAAGTAAGTGATTATACATTGTCCTAAGGGCTGTGAAAGAAAAGTACAAGGTACAAAGAGAAACGTCAATAGTCCTGAAAGGTTGACTGCATCAAGAACATGTCCATGGAGACCAGGCTGATGGAGTTTTTGTTTGGATTGTTCATTGCGTGAACCGA',
 'AGGACCGGATCAACTATTACCCTATGTCAGCCTTAATTTACATGTCTTCTCAGATTTTCTGAGCCTTACCATCTCTGGGACCCTTAACCTCTTCTTCTGTTTCAGCTGCTACCAGTAGAGACTTAGTTAAGTTCCTTGTGGTTTAACTGCCTGAAGCCAACCAGCCACATAACCACAGTTTCCTCCCACCGCTTCTGAACTCCAGGGCATGGGACCTGGCTTTCTGAGTGACTGTTAAAGAAAATGGATAACCCAACTGGGAAGTGAGAAAATTTTAACTCCCTGCATTGCGTGAACCGA',
 'AGGACCGGATCAACTGATCTGGGATCTGGTGAGGGGTCACACTCTCACCCATTCACAATCTGAGCACATTGCCTTGGGCCCTGCATGAGGAGAACAGCAGGGCTCCCTGGCCAGACTCCAAGTTCTCCCTTCAGTTTCACAAAGCAGCTGCCACATGAATGACGTTTTACATATTGTTCTCTGCTAGTTCATGGGAGTGGACAAATTTGGTGACTGTGAGGGGAGGGTCCCTTTCAGCTGATGACTTTGGGTTGTGTTTCTCAATATCTTTACTATGACTGCTGTCATTGCGTGAACCGA']

In [ ]:
AGGACCGGATCAACT

In [ ]:
AGGACCGGATCAACTTACATAGTGAGGACCTACTGTGTGCCAGTTAGGGTGTGGGGAAGAACAGGAGACAAACCACCTGGTCTCATTCTCATGAATTTTACAGTCTATTGGGAAACAGGTATTACACAAGTCAATATGAAAATAAGTAAGTGATTATACATTGTCCTAAGGGCTGTGAAAGAAAAGTACAAGGTACAAAGAGAAACGTCAATAGTCCTGAAAGGTTGACTGCATCAAGAACATGTCCATGGAGACCAGGCTGATGGAGTTTTTGTTTGGATTGTTCATTGCGTGAACCGA

In [ ]:

def set_blat_start_end_strand(row):
    """
    Incorporate the genomic start end and strand information from the row
    1. check if blat_start is not na
    2. Set strand, start and end
    3. return row
    """
    if pd.isna(row['blat_start']):
        return row
    row['chr'] = row['blat_chr']
    row['start'] = row['blat_start']
    row['end'] = row['blat_end']
    row['strand'] = row['blat_strand']
    return row

In [ ]:
blat_aligned_tested_elements_with_meta[['name', 'chr', 'start', 'end', 'strand']].dtypes

name       object
chr        object
start     float64
end       float64
strand     object
dtype: object

In [ ]:
blat_aligned_tested_elements_with_meta.columns


Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'chr', 'start', 'end', 'strand', 'variant_class',
       'variant_pos', 'SPDI', 'allele', 'info', 'tmp_matching_header',
       'REF_ID', 'ALT_ID', 'match_blat_header', 'blat_name', 'blat_strand',
       'blat_start', 'blat_end', 'blat_chr', 'tmp_modified_chr'],
      dtype='object')

#### Sanity check if the genomic corrdinates of the regions (input to MPRAOligoDesign) are still valid or got changed
- Example: `SKI|ENSG00000157933.11|EH38E2778471` region
- From `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/input/regions_5K.bed`: `chr1	2179468	2179817	SKI|ENSG00000157933.11|EH38E2778471	.	+`
- From blat mapping: `NC_000001.11 2179508 217
9777`
- samtools expects first position to be 1 based / display in UCSC is also 1 based
  - `samtools faidx GCF_000001405.26_GRCh38_genomic.fna NC_000001.11:2179508-217
9777`
  - => results of blat is correct and start is 0-based

- Adapter
  - at the beginning: `AGGACCGGATCAACT`
  - at the end: `CATTGCGTGAACCGA`

- Trying to use the region bed for the missing cases: for the tested sequences: <gene>|<gene id>|<enhancer id> is unique => for each sequence we can create this pattern and get the genomic coordinates
  -  

In [ ]:
# blat_aligned_tested_elements_with_meta

In [ ]:
# Identifying the regions from blat
# remember escaping "|":
blat_aligned_tested_elements_with_meta.loc[blat_aligned_tested_elements_with_meta['match_blat_header'].str.contains('SKI\|ENSG00000157933.11\|EH38E2778471_fwd_tile1-1')]
blat_aligned_tested_elements_with_meta.loc[blat_aligned_tested_elements_with_meta['match_blat_header'].str.contains('SKI\|ENSG00000157933.11\|EH38E2778471_fwd_tile1-1')]['match_blat_header'].to_list()
# blat_aligned_tested_elements_with_meta.loc[blat_aligned_tested_elements_with_meta['REF_ID'] == 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1']

['cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1']

In [ ]:
print(blat_aligned_tested_elements_with_meta.loc[blat_aligned_tested_elements_with_meta['match_blat_header'] == 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1']['sequence'].to_list())
blat_aligned_tested_elements_with_meta.loc[blat_aligned_tested_elements_with_meta['match_blat_header'] == 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1']

['AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGTGGTGAGTTAACAGCTGAGCCACCACCTGCTGCCCAACCCCACGTGTCCCACATGGCCGGGCGGTGCTGCGCTAACTCATCTCCCCCTGGATGGAAACGTTTGCGTGGTGACAGCCGATTCTCTTGAGAGTCATTTGCTGCCCATGTTGCTGGGGAGATTCTGCCTCAGGGCCAGGAGTGGTTTGCTCCTCCCACCCCGGGCCCAGGGCTGCTGGTGGGAGGCCCCAGGGAGGAGCAAGGCATTGCGTGAACCGA', 'AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGTGGTGAGTTAACAGCTGAGCCACCACCTGCTGCCCAACCCCACGTGTCCCACACGGCCGGGCGGTGCTGCGCTAACTCATCTCCCCCTGGATGGAAACGTTTGCGTGGTGACAGCCGATTCTCTTGAGAGTCATTTGCTGCCCATGTTGCTGGGGAGATTCTGCCTCAGGGCCAGGAGTGGTTTGCTCCTCCCACCCCGGGCCCAGGGCTGCTGGTGGGAGGCCCCAGGGAGGAGCAAGGCATTGCGTGAACCGA']


,header,sequence,tmp_label,name,category,class,source,ref,chr,start,...,tmp_matching_header,REF_ID,ALT_ID,match_blat_header,Q_name,match,strand_y,T_name,T_start,T_end
8900,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,NaN,NaN,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,270.0,+,NC_000001.11,2179507.0,2179777.0
27482,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,270.0,+,NC_000001.11,2179507.0,2179777.0


In [ ]:
blat_aligned_tested_elements_with_meta_test.loc[blat_aligned_tested_elements_with_meta_test['header'].str.contains('SKI\|ENSG00000157933.11\|EH38E2778471_fwd_tile1-1')]['header'].to_list()
# 'cardiac_neuro_cava_random:REF_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1',
#  'cardiac_neuro_cava_random:ALT_SKI|ENSG00000157933.11|EH38E2778471_fwd_tile1-1_SKI|ENSG00000157933.11|EH38E2778471|1-2179591-T-C'
blat_aligned_tested_elements_with_meta_test.loc[blat_aligned_tested_elements_with_meta_test['header'].str.contains('SKI\|ENSG00000157933.11\|EH38E2778471_fwd_tile1-1')][['name', 'variant_pos', 'strand', 'start', 'end']]

,name,variant_pos,strand,start,end
8900,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,NaN,+,2179507,2179777
27482,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,98.0,+,2179507,2179777


In [ ]:
# 1-2179591-T-C
2179591 - 2179507

84

#### Problem: I find 84 ALT_ (header) sequences and for these there is no corresponding entry in the variant region map 
- use the region map to create a tested_region.map (invent unique header (<gene>|<gene_id>|<enhancer_id>)) and map to not matchable sequences

In [ ]:
# investigate the list of headers which is not matchable
blat_aligned_tested_elements_with_meta.loc[blat_aligned_tested_elements_with_meta['T_start'].isna()]['match_blat_header'].to_list()

['cardiac_neuro_cava_random:ALT_PRDM16|ENSG00000142611.17|EH38E2779927_fwd_tile1-1_PRDM16|ENSG00000142611.17|EH38E2779927|1-3314074-G-C',
 'cardiac_neuro_cava_random:ALT_RERE|ENSG00000142599.20|EH38E2783640_rev_tile1-1_RERE|ENSG00000142599.20|EH38E2783640|1-8313520-T-G',
 'cardiac_neuro_cava_random:ALT_CASZ1|ENSG00000130940.15|EH38E2785710_rev_tile1-1_CASZ1|ENSG00000130940.15|EH38E2785710|1-10695590-C-G',
 'cardiac_neuro_cava_random:ALT_SDHB|ENSG00000117118.10|EH38E1322856_rev_tile1-1_SDHB|ENSG00000117118.10|EH38E1322856|1-17067208-A-T',
 'cardiac_neuro_cava_random:ALT_ECE1|ENSG00000117298.16|EH38E1325981_rev_tile1-1_ECE1|ENSG00000117298.16|EH38E1325981|1-21280872-A-G',
 'cardiac_neuro_cava_random:ALT_AHDC1|ENSG00000126705.15|EH38E2797860_rev_tile1-1_AHDC1|ENSG00000126705.15|EH38E2797860|1-27540408-A-C',
 'cardiac_neuro_cava_random:ALT_FHL3|ENSG00000183386.10|EH38E2804002_rev_tile1-1_FHL3|ENSG00000183386.10|EH38E2804002|1-38011158-A-C',
 'cardiac_neuro_cava_random:ALT_DOCK7|ENSG0000011

In [ ]:
matchable_by_blat = blat_aligned_tested_elements_with_meta.loc[~blat_aligned_tested_elements_with_meta['T_start'].isna()]['match_blat_header'].to_list()
def get_region_pattern(header):
    """
    For missing sequences which are alternatives without reference: get region pattern
    cardiac_neuro_cava_random:ALT_PRDM16|ENSG00000142611.17|EH38E2779927_fwd_tile1-1_PRDM16|ENSG00000142611.17|EH38E2779927|1-3314074-G-C => PRDM16|ENSG00000142611.17|EH38E2779927
    cardiac_neuro_cava_random:ALT_G6PD|ENSG00000160211.20|EH38E3949687_rev_tile1-1_G6PD|ENSG00000160211.20|EH38E3949687|X-154491628-G-C => G6PD|ENSG00000160211.20|EH38E3949687
    """
    if header in matchable_by_blat:
        return "blat_matchable"
    if '~' in header:
        #'cardiac_neuro_cava_random:ALT_DEAF1|ENSG00000177030.19|EH38E2937977~SLC25A22|ENSG00000177542.11|EH38E2937977_rev_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937977|11-744461-A-C~SLC25A22|ENSG00000177542.11|EH38E2937977|11-744461-A-C',
        # remove "~" and call function again
        new_header = get_region_pattern(header.split('~')[0] + '_suffix')
        return new_header
    if ',' in header:
        # remove "," and call function again
        new_header = get_region_pattern(header.split(',')[0] + '_suffix')
        return new_header
    if 'REF_' in header:
        return header.split(':REF_')[1].split('_')[0]
    elif 'ALT_' in header:
        return header.split(':ALT_')[1].split('_')[0]
    else:
        return header.split(':')[1].split('_')[0]

In [ ]:
# invent gene_enhancer_id
blat_aligned_tested_elements_with_meta['gene_enhancer_id'] = blat_aligned_tested_elements_with_meta['match_blat_header'].apply(get_region_pattern)


##### Create tested_region.bed from final region bed

In [ ]:
# # modify region bed identifiers and create a tested regions bed file:
# #!/bin/bash
# # Process the BED file and update the fourth column with the extracted pattern
# cat /home/kisa/coding/80K_MPRA/design_data/design_info/regions.bed | grep "cardiac_neuro_cava_random" | awk '{
#     if ($4 ~ /~/) {
#         # Split the fourth field based on the colon
#         split($4, parts, ":")

#         # Split the second part based on the first underscore to handle the gene part
#         split(parts[2], subparts, "~")
#     } else if ($4 ~ /,/) {
#         # Split the fourth field based on the colon
#         split($4, parts, ":")

#         # Split the second part based on the first underscore to handle the gene part
#         split(parts[2], subparts, ",")
#     } else {
#         # Split the fourth field based on the colon
#         split($4, parts, ":")

#         # Split the second part based on the first underscore to handle the gene part
#         split(parts[2], subparts, "_")
#     }

#     # Replace the fourth field with the new gene_info
#     $4 = subparts[1]

#     # Print the modified line
#     print

# }' > /home/kisa/coding/80K_MPRA/design_data/design_info/tested_regions.bed

In [ ]:
# read new tested_regions.bed in and left join to the
modified_tested_region_bed = pd.read_csv(config['files']['creating']['modified_tested_region_bed'], sep=" ", header=None)
modified_tested_region_bed.columns = ['tested_region_chr', 'tested_region_start', 'tested_region_end', 'gene_enhancer_id', 'tested_region_score', 'tested_region_strand']
modified_tested_region_bed

# # left join
blat_aligned_tested_elements_with_meta_tested_region = blat_aligned_tested_elements_with_meta.merge(modified_tested_region_bed[['tested_region_chr', 'tested_region_start', 'tested_region_end', 'gene_enhancer_id', 'tested_region_strand']], on='gene_enhancer_id', how='left')
blat_aligned_tested_elements_with_meta_tested_region.loc[~blat_aligned_tested_elements_with_meta_tested_region['tested_region_start'].isna()].shape[0]

KeyError: 'gene_enhancer_id'

#### Add genomic coordinates of missing sequences to metadata file:
- If enhaner gene id != blat_matchable => 'start' = tested_region_start and 'end' = tested_region_end

In [ ]:
def add_start_end_missed_by_blat(row):
    """
    For the 84 sequences which were not matched by blat, add start and end from the modified region file.

    """
    if row['gene_enhancer_id'] == 'blat_matchable':
        return row
    row['chr'] = row['tested_region_chr']
    row['start'] = row['tested_region_start']
    row['end'] = row['tested_region_end']
    row['strand'] = row['tested_region_strand']
    return row

In [ ]:
blat_aligned_tested_elements_with_meta_tested_region = blat_aligned_tested_elements_with_meta_tested_region.apply(add_start_end_missed_by_blat, axis=1)

In [ ]:
print(blat_aligned_tested_elements_with_meta_tested_region.columns)
blat_aligned_tested_elements_with_meta_tested_region['chr'].value_counts()

Index(['ALT_ID', 'Q_name', 'REF_ID', 'SPDI', 'T_end', 'T_name', 'T_start',
       'allele', 'category', 'chr', 'class', 'end', 'gene_enhancer_id',
       'header', 'info', 'match', 'match_blat_header', 'name', 'ref',
       'sequence', 'source', 'start', 'strand', 'strand_x', 'strand_y',
       'tested_region_chr', 'tested_region_end', 'tested_region_start',
       'tested_region_strand', 'tmp_label', 'tmp_matching_header',
       'variant_class', 'variant_pos'],
      dtype='object')


chr1     13
chr2     11
chr19     7
chr16     6
chr20     6
chr17     6
chrX      5
chr11     4
chr6      4
chr9      4
chr7      3
chr4      3
chr18     3
chr5      2
chr8      2
chr10     1
chr12     1
chr15     1
chr22     1
chr3      1
Name: chr, dtype: int64

##### Santiy checking if it worked with: 


In [ ]:
blat_aligned_tested_elements_with_meta.loc[blat_aligned_tested_elements_with_meta['T_start'].isna()]['match_blat_header'].to_list()

['cardiac_neuro_cava_random:ALT_PRDM16|ENSG00000142611.17|EH38E2779927_fwd_tile1-1_PRDM16|ENSG00000142611.17|EH38E2779927|1-3314074-G-C',
 'cardiac_neuro_cava_random:ALT_RERE|ENSG00000142599.20|EH38E2783640_rev_tile1-1_RERE|ENSG00000142599.20|EH38E2783640|1-8313520-T-G',
 'cardiac_neuro_cava_random:ALT_CASZ1|ENSG00000130940.15|EH38E2785710_rev_tile1-1_CASZ1|ENSG00000130940.15|EH38E2785710|1-10695590-C-G',
 'cardiac_neuro_cava_random:ALT_SDHB|ENSG00000117118.10|EH38E1322856_rev_tile1-1_SDHB|ENSG00000117118.10|EH38E1322856|1-17067208-A-T',
 'cardiac_neuro_cava_random:ALT_ECE1|ENSG00000117298.16|EH38E1325981_rev_tile1-1_ECE1|ENSG00000117298.16|EH38E1325981|1-21280872-A-G',
 'cardiac_neuro_cava_random:ALT_AHDC1|ENSG00000126705.15|EH38E2797860_rev_tile1-1_AHDC1|ENSG00000126705.15|EH38E2797860|1-27540408-A-C',
 'cardiac_neuro_cava_random:ALT_FHL3|ENSG00000183386.10|EH38E2804002_rev_tile1-1_FHL3|ENSG00000183386.10|EH38E2804002|1-38011158-A-C',
 'cardiac_neuro_cava_random:ALT_DOCK7|ENSG0000011